## Final frozen-test RQ1 profiling


In [5]:
# ======================================================================
# PURPOSE
# -------
# Final TEST evaluation for:
#
# RQ1:
# "Where and to what extent does search-space growth occur during
#  RoG's relation-constrained graph retrieval?"
#
# ALSO performs POST-FREEZE oracle analysis:
#   - doomed-prefix work
#   - oracle-prunable intermediate branches
#   - oracle-avoidable downstream edges
#   - final-hop-protected doomed candidates
#
# IMPORTANT
# ---------
# AFP is ALREADY FROZEN.
#
# This cell:
#   - does NOT tune anything
#   - does NOT train anything
#   - does NOT change planner/scorer/selector
#   - uses TEST gold only for POST-HOC reachability/oracle analysis
#
# REQUIRED INPUT
# --------------
# Exact RoG TEST planning outputs with schema:
#
#   id
#   question
#   q_entity
#   a_entity
#   graph
#   predicted_paths
#
# The cell auto-detects already-materialized TEST planning rows/files.
#
# It WILL NOT invent or regenerate planner outputs using a new
# implementation.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import hashlib
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE PREREQUISITES
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "FINAL_AFP_CONFIG_FROZEN",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing required frozen-development objects:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA
), (
    "AFP freeze SHA mismatch. "
    "STOP before TEST evaluation."
)


print("Cell 16 freeze gate: PASSED")
print("AFP freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. OUTPUT DIRECTORY
# ======================================================================

ROOT = Path(
    "/kaggle/working/step3_rq2_dev_v1"
)

FINAL_TEST_ROOT = (
    ROOT
    / "15_final_frozen_test"
)

RQ1_TEST_DIR = (
    FINAL_TEST_ROOT
    / "rq1_profiling"
)

RQ1_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 3. HASH HELPERS
# ======================================================================

def sha256_file_16(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def rows_sha256_16(rows):

    canonical = json.dumps(
        rows,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )

    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()


# ======================================================================
# 4. TEST PLANNING SCHEMA
# ======================================================================

REQUIRED_PLAN_FIELDS_16 = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
}


def planning_rows_valid_16(rows):

    if not isinstance(rows, list):

        return False

    if len(rows) == 0:

        return False

    sample_indices = sorted(
        set(
            [
                0,
                len(rows) // 2,
                len(rows) - 1,
            ]
        )
    )

    for idx in sample_indices:

        rec = rows[idx]

        if not isinstance(rec, dict):

            return False

        if not REQUIRED_PLAN_FIELDS_16.issubset(
            rec.keys()
        ):

            return False

    return True


# ======================================================================
# 5. LOAD JSONL
# ======================================================================

def read_jsonl_16(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            rows.append(
                json.loads(line)
            )

    return rows


# ======================================================================
# 6. DISCOVER ALREADY-MATERIALIZED TEST PLANNING OUTPUT
# ======================================================================

def discover_test_planning_16(
    dataset_name
):

    dataset_name = dataset_name.lower()

    # --------------------------------------------------------------
    # First: surviving globals.
    # --------------------------------------------------------------

    candidate_global_names = [
        f"{dataset_name}_test_plan_rows",
        f"{dataset_name}_test_planning_rows",
        f"{dataset_name}_planning_test_rows",
        f"{dataset_name}_test_plans",
    ]


    valid_globals = []


    for name in candidate_global_names:

        if name not in globals():

            continue

        value = globals()[name]

        if planning_rows_valid_16(
            value
        ):

            valid_globals.append(
                (
                    name,
                    value
                )
            )


    if len(valid_globals) == 1:

        name, rows = valid_globals[0]

        print(
            f"{dataset_name.upper()} TEST planning "
            f"loaded from global: {name}"
        )

        return (
            rows,
            None,
            rows_sha256_16(rows)
        )


    if len(valid_globals) > 1:

        hashes = {
            rows_sha256_16(rows)
            for _, rows in valid_globals
        }

        assert len(hashes) == 1, (
            f"{dataset_name}: multiple non-identical "
            "TEST planning globals found."
        )

        name, rows = valid_globals[0]

        print(
            f"{dataset_name.upper()}: multiple identical "
            f"globals found; using {name}"
        )

        return (
            rows,
            None,
            rows_sha256_16(rows)
        )


    # --------------------------------------------------------------
    # Second: frozen planning JSONL artifacts under /kaggle/working.
    # --------------------------------------------------------------

    candidates = []


    for path in Path(
        "/kaggle/working"
    ).rglob("*.jsonl"):

        lower = str(
            path
        ).lower()

        if dataset_name not in lower:

            continue

        if "test" not in lower:

            continue


        try:

            rows = read_jsonl_16(
                path
            )

        except Exception:

            continue


        if not planning_rows_valid_16(
            rows
        ):

            continue


        candidates.append(
            {
                "path":
                    path,

                "rows":
                    rows,

                "sha256":
                    sha256_file_16(
                        path
                    ),
            }
        )


    assert len(candidates) >= 1, (
        f"\n{dataset_name.upper()} frozen TEST planning "
        "artifact was not found.\n\n"
        "STOP HERE.\n"
        "Do NOT generate test relation plans with a new or "
        "approximate planner implementation.\n\n"
        "Materialize TEST plans using the EXACT frozen RoG "
        "planner-generation code/configuration used for validation, "
        "then rerun Cell 16."
    )


    if len(candidates) > 1:

        content_hashes = {
            rows_sha256_16(
                row["rows"]
            )
            for row in candidates
        }

        assert len(
            content_hashes
        ) == 1, (
            f"{dataset_name}: multiple non-identical TEST "
            "planning artifacts found. STOP."
        )


    chosen = sorted(
        candidates,
        key=lambda x:
            str(
                x["path"]
            )
    )[0]


    print(
        f"{dataset_name.upper()} TEST planning:"
    )

    print(
        " ",
        chosen[
            "path"
        ]
    )

    print(
        " SHA256:",
        chosen[
            "sha256"
        ]
    )


    return (
        chosen[
            "rows"
        ],
        chosen[
            "path"
        ],
        chosen[
            "sha256"
        ]
    )


(
    webqsp_test_plan_rows,
    WEBQSP_TEST_PLAN_PATH,
    WEBQSP_TEST_PLAN_SHA
) = discover_test_planning_16(
    "webqsp"
)


(
    cwq_test_plan_rows,
    CWQ_TEST_PLAN_PATH,
    CWQ_TEST_PLAN_SHA
) = discover_test_planning_16(
    "cwq"
)


# ======================================================================
# 7. TEST / VALIDATION DISJOINTNESS GATE
# ======================================================================

def id_set_16(rows):

    return {
        str(
            rec["id"]
        )
        for rec in rows
    }


assert not (
    id_set_16(
        webqsp_test_plan_rows
    )
    &
    id_set_16(
        webqsp_val_plan_rows
    )
), (
    "WebQSP TEST IDs overlap validation IDs."
)


assert not (
    id_set_16(
        cwq_test_plan_rows
    )
    &
    id_set_16(
        cwq_val_plan_rows
    )
), (
    "CWQ TEST IDs overlap validation IDs."
)


print(
    "\nTEST / validation ID-disjointness: PASSED"
)

print(
    "WebQSP TEST questions:",
    len(
        webqsp_test_plan_rows
    )
)

print(
    "CWQ TEST questions:",
    len(
        cwq_test_plan_rows
    )
)


# ======================================================================
# 8. EXACT GRAPH HELPERS
# ======================================================================

def matching_neighbors_16(
    adjacency,
    entity,
    relation
):

    return [
        neighbor
        for neighbor, edge_relation
        in adjacency.get(
            entity,
            {}
        ).items()
        if edge_relation == relation
    ]


def degree_16(
    adjacency,
    entity
):

    return len(
        adjacency.get(
            entity,
            {}
        )
    )


# ======================================================================
# 9. QUESTION-LOCAL ORACLE FUNCTIONS
# ======================================================================
#
# POST-FREEZE only.
#
# suffix_reachable(entity, suffix):
#   whether exact remaining relation sequence can reach gold.
#
# downstream_cost(entity, suffix):
#   exact future edge examinations RoG would perform from ONE prefix.
#
# Repeated prefixes are not merged because RoG traversal preserves
# path-prefix multiplicity.
# ======================================================================

def make_oracle_functions_16(
    adjacency,
    gold_answers
):

    gold_set = {
        str(x)
        for x in gold_answers
    }


    reachable_cache = {}

    cost_cache = {}


    def suffix_reachable(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suffix
        )

        key = (
            entity,
            suffix
        )

        if key in reachable_cache:

            return reachable_cache[
                key
            ]


        if len(
            suffix
        ) == 0:

            result = (
                entity
                in
                gold_set
            )

            reachable_cache[
                key
            ] = result

            return result


        relation = suffix[0]


        result = any(
            suffix_reachable(
                neighbor,
                suffix[
                    1:
                ]
            )

            for neighbor
            in matching_neighbors_16(
                adjacency,
                entity,
                relation
            )
        )


        reachable_cache[
            key
        ] = bool(
            result
        )


        return bool(
            result
        )


    def downstream_cost(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suffix
        )

        key = (
            entity,
            suffix
        )

        if key in cost_cache:

            return cost_cache[
                key
            ]


        if len(
            suffix
        ) == 0:

            cost_cache[
                key
            ] = 0

            return 0


        relation = suffix[0]


        total = degree_16(
            adjacency,
            entity
        )


        for neighbor in matching_neighbors_16(
            adjacency,
            entity,
            relation
        ):

            total += downstream_cost(
                neighbor,
                suffix[
                    1:
                ]
            )


        total = int(
            total
        )


        cost_cache[
            key
        ] = total


        return total


    return (
        suffix_reachable,
        downstream_cost
    )


# ======================================================================
# 10. EXACT RQ1 PROFILER
# ======================================================================

def profile_rq1_dataset_16(
    dataset_name,
    planning_rows
):

    hop_rows = []
    plan_rows = []
    question_rows = []


    oracle_totals = {
        "total_edges":
            0,

        "doomed_prefix_edges":
            0,

        "candidate_branches":
            0,

        "intermediate_candidate_branches":
            0,

        "oracle_prunable_intermediate_branches":
            0,

        "doomed_final_protected_branches":
            0,

        "oracle_avoidable_downstream_edges":
            0,
    }


    for q_index, rec in enumerate(
        tqdm(
            planning_rows,
            desc=f"{dataset_name} final TEST RQ1"
        )
    ):

        qid = str(
            rec["id"]
        )


        graph = rec[
            "graph"
        ]


        adjacency = (
            build_exact_rog_adjacency(
                graph
            )
        )


        topics = (
            as_entity_list(
                rec["q_entity"]
            )
        )


        gold = (
            as_entity_list(
                rec["a_entity"]
            )
        )


        plans = (
            normalize_relation_plans(
                rec[
                    "predicted_paths"
                ]
            )
        )


        (
            suffix_reachable,
            downstream_cost
        ) = make_oracle_functions_16(
            adjacency,
            gold
        )


        question_edges = 0
        question_candidates = 0
        question_active_prefixes = 0

        question_reachable = False

        q_unique_expanded_entities = set()

        q_peak_frontier = len(
            topics
        )

        q_has_intermediate_active = False
        q_has_decision = False


        for plan_index, plan in enumerate(
            plans
        ):

            if len(
                plan
            ) == 0:

                plan_rows.append(
                    {
                        "dataset":
                            dataset_name,

                        "question_id":
                            qid,

                        "question_index":
                            q_index,

                        "plan_index":
                            plan_index,

                        "plan_length":
                            0,

                        "empty_plan":
                            True,

                        "active_hop_rows":
                            0,

                        "active_prefixes":
                            0,

                        "edges_examined":
                            0,

                        "candidate_branches":
                            0,

                        "retrieved_paths":
                            0,

                        "peak_frontier":
                            len(
                                topics
                            ),

                        "reachable":
                            False,

                        "eligible_intermediate":
                            False,

                        "has_decision":
                            False,

                        "oracle_prunable_intermediate":
                            0,

                        "oracle_avoidable_downstream_edges":
                            0,
                    }
                )

                continue


            active = [
                (
                    str(
                        entity
                    ),
                )
                for entity in topics
            ]


            plan_active_rows = 0
            plan_active_prefixes = 0
            plan_edges = 0
            plan_candidates = 0

            plan_peak = len(
                active
            )

            plan_eligible = False
            plan_decision = False

            plan_oracle_prunable = 0
            plan_oracle_avoidable = 0


            L = len(
                plan
            )


            for hop, target_relation in enumerate(
                plan
            ):

                if not active:

                    break


                plan_active_rows += 1

                plan_active_prefixes += len(
                    active
                )


                q_peak_frontier = max(
                    q_peak_frontier,
                    len(
                        active
                    )
                )


                plan_peak = max(
                    plan_peak,
                    len(
                        active
                    )
                )


                # ------------------------------------------------------
                # Current active prefixes
                # ------------------------------------------------------

                current_unique_entities = {
                    prefix[-1]
                    for prefix in active
                }


                q_unique_expanded_entities.update(
                    current_unique_entities
                )


                edges_this_hop = sum(
                    degree_16(
                        adjacency,
                        prefix[-1]
                    )
                    for prefix in active
                )


                plan_edges += edges_this_hop

                question_edges += edges_this_hop

                oracle_totals[
                    "total_edges"
                ] += edges_this_hop


                # ------------------------------------------------------
                # Doomed-prefix current work
                # ------------------------------------------------------

                remaining_from_active = (
                    plan[
                        hop:
                    ]
                )


                doomed_prefix_edges = 0


                for prefix in active:

                    endpoint = prefix[
                        -1
                    ]


                    if not suffix_reachable(
                        endpoint,
                        remaining_from_active
                    ):

                        doomed_prefix_edges += (
                            degree_16(
                                adjacency,
                                endpoint
                            )
                        )


                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges


                # ------------------------------------------------------
                # Exact relation-constrained candidates
                # ------------------------------------------------------

                candidates = []


                for parent_index, prefix in enumerate(
                    active
                ):

                    endpoint = prefix[
                        -1
                    ]


                    for neighbor, relation in (
                        adjacency.get(
                            endpoint,
                            {}
                        ).items()
                    ):

                        if relation != target_relation:

                            continue


                        candidates.append(
                            prefix
                            +
                            (
                                neighbor,
                            )
                        )


                candidate_count = len(
                    candidates
                )


                plan_candidates += candidate_count

                question_candidates += (
                    candidate_count
                )


                oracle_totals[
                    "candidate_branches"
                ] += candidate_count


                unique_candidate_entities = {
                    prefix[-1]
                    for prefix in candidates
                }


                duplicate_ratio = (
                    1.0
                    -
                    len(
                        unique_candidate_entities
                    )
                    /
                    candidate_count

                    if candidate_count > 0

                    else 0.0
                )


                rho = (
                    candidate_count
                    /
                    len(
                        active
                    )
                )


                is_intermediate = (
                    hop
                    <
                    L - 1
                )


                if is_intermediate:

                    q_has_intermediate_active = True
                    plan_eligible = True

                    oracle_totals[
                        "intermediate_candidate_branches"
                    ] += candidate_count


                decision = (
                    is_intermediate
                    and
                    candidate_count > 1
                )


                if decision:

                    q_has_decision = True
                    plan_decision = True


                # ------------------------------------------------------
                # Oracle analysis AFTER current candidate generation.
                #
                # Current-hop edge cost has already been paid.
                # ------------------------------------------------------

                suffix_after_candidate = (
                    plan[
                        hop + 1:
                    ]
                )


                oracle_prunable_this_hop = 0
                oracle_avoidable_this_hop = 0
                downstream_work_all_candidates = 0


                if is_intermediate:

                    for candidate in candidates:

                        endpoint = candidate[
                            -1
                        ]


                        future_cost = (
                            downstream_cost(
                                endpoint,
                                suffix_after_candidate
                            )
                        )


                        downstream_work_all_candidates += (
                            future_cost
                        )


                        feasible = (
                            suffix_reachable(
                                endpoint,
                                suffix_after_candidate
                            )
                        )


                        if not feasible:

                            oracle_prunable_this_hop += 1

                            oracle_avoidable_this_hop += (
                                future_cost
                            )


                    oracle_totals[
                        "oracle_prunable_intermediate_branches"
                    ] += oracle_prunable_this_hop


                    oracle_totals[
                        "oracle_avoidable_downstream_edges"
                    ] += oracle_avoidable_this_hop


                    plan_oracle_prunable += (
                        oracle_prunable_this_hop
                    )


                    plan_oracle_avoidable += (
                        oracle_avoidable_this_hop
                    )


                else:

                    # Final candidate branches are protected.
                    doomed_final = sum(
                        1
                        for candidate
                        in candidates
                        if not suffix_reachable(
                            candidate[-1],
                            ()
                        )
                    )


                    oracle_totals[
                        "doomed_final_protected_branches"
                    ] += doomed_final


                hop_rows.append(
                    {
                        "dataset":
                            dataset_name,

                        "question_id":
                            qid,

                        "question_index":
                            q_index,

                        "plan_index":
                            plan_index,

                        "plan_length":
                            L,

                        "hop":
                            hop,

                        "is_intermediate":
                            is_intermediate,

                        "active_prefixes":
                            len(
                                active
                            ),

                        "unique_active_entities":
                            len(
                                current_unique_entities
                            ),

                        "edges_examined":
                            int(
                                edges_this_hop
                            ),

                        "candidate_branches":
                            int(
                                candidate_count
                            ),

                        "unique_candidate_entities":
                            len(
                                unique_candidate_entities
                            ),

                        "duplicate_endpoint_ratio":
                            float(
                                duplicate_ratio
                            ),

                        "rho":
                            float(
                                rho
                            ),

                        "decision_hop":
                            bool(
                                decision
                            ),

                        "oracle_prunable_candidates":
                            int(
                                oracle_prunable_this_hop
                            ),

                        "oracle_avoidable_downstream_edges":
                            int(
                                oracle_avoidable_this_hop
                            ),

                        "downstream_work_all_candidates":
                            int(
                                downstream_work_all_candidates
                            ),
                    }
                )


                active = candidates


                q_peak_frontier = max(
                    q_peak_frontier,
                    len(
                        active
                    )
                )


                plan_peak = max(
                    plan_peak,
                    len(
                        active
                    )
                )


            answer_set = {
                str(x)
                for x in gold
            }


            plan_reachable = any(
                prefix[-1]
                in
                answer_set

                for prefix in active
            )


            if plan_reachable:

                question_reachable = True


            plan_rows.append(
                {
                    "dataset":
                        dataset_name,

                    "question_id":
                        qid,

                    "question_index":
                        q_index,

                    "plan_index":
                        plan_index,

                    "plan_length":
                        len(
                            plan
                        ),

                    "empty_plan":
                        False,

                    "active_hop_rows":
                        int(
                            plan_active_rows
                        ),

                    "active_prefixes":
                        int(
                            plan_active_prefixes
                        ),

                    "edges_examined":
                        int(
                            plan_edges
                        ),

                    "candidate_branches":
                        int(
                            plan_candidates
                        ),

                    "retrieved_paths":
                        int(
                            len(
                                active
                            )
                        ),

                    "peak_frontier":
                        int(
                            plan_peak
                        ),

                    "reachable":
                        bool(
                            plan_reachable
                        ),

                    "eligible_intermediate":
                        bool(
                            plan_eligible
                        ),

                    "has_decision":
                        bool(
                            plan_decision
                        ),

                    "oracle_prunable_intermediate":
                        int(
                            plan_oracle_prunable
                        ),

                    "oracle_avoidable_downstream_edges":
                        int(
                            plan_oracle_avoidable
                        ),
                }
            )


            question_active_prefixes += (
                plan_active_prefixes
            )


        question_rows.append(
            {
                "dataset":
                    dataset_name,

                "question_id":
                    qid,

                "question_index":
                    q_index,

                "predicted_plan_count":
                    len(
                        plans
                    ),

                "edges_examined":
                    int(
                        question_edges
                    ),

                "candidate_branches":
                    int(
                        question_candidates
                    ),

                "active_prefixes":
                    int(
                        question_active_prefixes
                    ),

                "unique_expanded_entities":
                    int(
                        len(
                            q_unique_expanded_entities
                        )
                    ),

                "peak_frontier":
                    int(
                        q_peak_frontier
                    ),

                "reachable":
                    bool(
                        question_reachable
                    ),

                "eligible_intermediate":
                    bool(
                        q_has_intermediate_active
                    ),

                "has_decision":
                    bool(
                        q_has_decision
                    ),
            }
        )


    hop_df = pd.DataFrame(
        hop_rows
    )


    plan_df = pd.DataFrame(
        plan_rows
    )


    question_df = pd.DataFrame(
        question_rows
    )


    return (
        hop_df,
        plan_df,
        question_df,
        oracle_totals
    )


# ======================================================================
# 11. VALIDATION ORACLE SOFTWARE GATE
# ======================================================================
#
# Before exposing TEST oracle results, reproduce the previously verified
# validation oracle totals.
# ======================================================================

EXPECTED_VAL_ORACLE_16 = {
    "webqsp": {
        "total_edges":
            341526,

        "candidate_branches":
            7983,

        "doomed_prefix_edges":
            159039,

        "intermediate_candidate_branches":
            1707,

        "oracle_prunable_intermediate_branches":
            1340,

        "doomed_final_protected_branches":
            4882,

        "oracle_avoidable_downstream_edges":
            11717,
    },

    "cwq": {
        "total_edges":
            5257272,

        "candidate_branches":
            247161,

        "doomed_prefix_edges":
            3197488,

        "intermediate_candidate_branches":
            41576,

        "oracle_prunable_intermediate_branches":
            34431,

        "doomed_final_protected_branches":
            195499,

        "oracle_avoidable_downstream_edges":
            1176294,
    },
}


def oracle_validation_gate_16(
    dataset_name,
    validation_rows
):

    (
        _hop,
        _plan,
        _question,
        oracle
    ) = profile_rq1_dataset_16(
        f"{dataset_name}_validation_gate",
        validation_rows
    )


    expected = (
        EXPECTED_VAL_ORACLE_16[
            dataset_name
        ]
    )


    for key, expected_value in (
        expected.items()
    ):

        actual = int(
            oracle[
                key
            ]
        )


        assert actual == int(
            expected_value
        ), (
            f"{dataset_name} oracle gate mismatch "
            f"for {key}: {actual} != {expected_value}"
        )


    print(
        f"{dataset_name.upper()} validation oracle "
        "software gate: PASSED"
    )


oracle_validation_gate_16(
    "webqsp",
    webqsp_val_plan_rows
)


oracle_validation_gate_16(
    "cwq",
    cwq_val_plan_rows
)


print(
    "\nRQ1 oracle implementation fidelity: PASSED"
)


# ======================================================================
# 12. FINAL TEST PROFILING
# ======================================================================

(
    webqsp_test_hop_df,
    webqsp_test_plan_df,
    webqsp_test_question_df,
    webqsp_test_oracle
) = profile_rq1_dataset_16(
    "webqsp",
    webqsp_test_plan_rows
)


(
    cwq_test_hop_df,
    cwq_test_plan_df,
    cwq_test_question_df,
    cwq_test_oracle
) = profile_rq1_dataset_16(
    "cwq",
    cwq_test_plan_rows
)


# ======================================================================
# 13. DESCRIPTIVE STATISTICS HELPERS
# ======================================================================

def qstats_16(
    series
):

    series = pd.Series(
        series
    ).dropna()


    if len(
        series
    ) == 0:

        return {
            "n": 0
        }


    return {
        "n":
            int(
                len(
                    series
                )
            ),

        "mean":
            float(
                series.mean()
            ),

        "median":
            float(
                series.median()
            ),

        "p75":
            float(
                series.quantile(
                    0.75
                )
            ),

        "p90":
            float(
                series.quantile(
                    0.90
                )
            ),

        "p95":
            float(
                series.quantile(
                    0.95
                )
            ),

        "max":
            float(
                series.max()
            ),
    }


# ======================================================================
# 14. RQ1 FINAL SUMMARY
# ======================================================================

def build_rq1_summary_16(
    dataset_name,
    hop_df,
    plan_df,
    question_df,
    oracle
):

    active = hop_df.copy()


    intermediate = (
        hop_df[
            hop_df[
                "is_intermediate"
            ]
        ].copy()
    )


    decision = (
        intermediate[
            intermediate[
                "decision_hop"
            ]
        ].copy()
    )


    total_plans = len(
        plan_df
    )


    nonempty_plans = int(
        (
            ~plan_df[
                "empty_plan"
            ]
        ).sum()
    )


    empty_plans = (
        total_plans
        -
        nonempty_plans
    )


    reachable_plans = int(
        plan_df[
            "reachable"
        ].sum()
    )


    reachable_questions = int(
        question_df[
            "reachable"
        ].sum()
    )


    eligible_plans = int(
        plan_df[
            "eligible_intermediate"
        ].sum()
    )


    decision_plans = int(
        plan_df[
            "has_decision"
        ].sum()
    )


    eligible_questions = int(
        question_df[
            "eligible_intermediate"
        ].sum()
    )


    decision_questions = int(
        question_df[
            "has_decision"
        ].sum()
    )


    intermediate_active_hops = len(
        intermediate
    )


    decision_hops = len(
        decision
    )


    total_edges = int(
        oracle[
            "total_edges"
        ]
    )


    summary = {
        "dataset":
            dataset_name,

        "questions":
            int(
                len(
                    question_df
                )
            ),

        "total_predicted_plans":
            int(
                total_plans
            ),

        "nonempty_plans":
            int(
                nonempty_plans
            ),

        "empty_plans":
            int(
                empty_plans
            ),

        "active_hop_rows":
            int(
                len(
                    active
                )
            ),

        "active_prefixes_total":
            int(
                active[
                    "active_prefixes"
                ].sum()
            ),

        "edges_examined_total":
            int(
                active[
                    "edges_examined"
                ].sum()
            ),

        "candidate_branches_total":
            int(
                active[
                    "candidate_branches"
                ].sum()
            ),

        "reachable_plans":
            reachable_plans,

        "reachable_plan_rate":
            (
                reachable_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "reachable_questions":
            reachable_questions,

        "reachable_question_rate":
            (
                reachable_questions
                /
                len(
                    question_df
                )
            ),

        "branching_rate_candidate_gt1_all":
            float(
                (
                    active[
                        "candidate_branches"
                    ]
                    >
                    1
                ).mean()
            ),

        "branching_rate_candidate_gt1_intermediate":
            (
                float(
                    (
                        intermediate[
                            "candidate_branches"
                        ]
                        >
                        1
                    ).mean()
                )
                if
                len(
                    intermediate
                ) > 0
                else 0.0
            ),

        "rho_gt1_rate":
            float(
                (
                    active[
                        "rho"
                    ]
                    >
                    1
                ).mean()
            ),

        "rho_eq1_rate":
            float(
                np.isclose(
                    active[
                        "rho"
                    ],
                    1.0
                ).mean()
            ),

        "rho_lt1_rate":
            float(
                (
                    active[
                        "rho"
                    ]
                    <
                    1
                ).mean()
            ),

        "eligible_plans":
            eligible_plans,

        "eligible_plan_rate":
            (
                eligible_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "decision_plans":
            decision_plans,

        "decision_plan_rate":
            (
                decision_plans
                /
                nonempty_plans
                if nonempty_plans > 0
                else 0.0
            ),

        "eligible_questions":
            eligible_questions,

        "eligible_question_rate":
            (
                eligible_questions
                /
                len(
                    question_df
                )
            ),

        "decision_questions":
            decision_questions,

        "decision_question_rate":
            (
                decision_questions
                /
                len(
                    question_df
                )
            ),

        "intermediate_active_hops":
            int(
                intermediate_active_hops
            ),

        "decision_hops":
            int(
                decision_hops
            ),

        "decision_rate_among_intermediate_active":
            (
                decision_hops
                /
                intermediate_active_hops
                if intermediate_active_hops > 0
                else 0.0
            ),

        "duplicate_endpoint_hop_rate":
            float(
                (
                    active[
                        "duplicate_endpoint_ratio"
                    ]
                    >
                    0
                ).mean()
            ),

        "active_prefix_stats_all":
            qstats_16(
                active[
                    "active_prefixes"
                ]
            ),

        "edge_stats_all":
            qstats_16(
                active[
                    "edges_examined"
                ]
            ),

        "candidate_stats_all":
            qstats_16(
                active[
                    "candidate_branches"
                ]
            ),

        "unique_frontier_stats_all":
            qstats_16(
                active[
                    "unique_candidate_entities"
                ]
            ),

        "active_prefix_stats_intermediate":
            qstats_16(
                intermediate[
                    "active_prefixes"
                ]
            ),

        "edge_stats_intermediate":
            qstats_16(
                intermediate[
                    "edges_examined"
                ]
            ),

        "candidate_stats_intermediate":
            qstats_16(
                intermediate[
                    "candidate_branches"
                ]
            ),

        "question_edge_stats":
            qstats_16(
                question_df[
                    "edges_examined"
                ]
            ),

        "question_peak_frontier_stats":
            qstats_16(
                question_df[
                    "peak_frontier"
                ]
            ),

        "decision_downstream_work_stats":
            qstats_16(
                decision[
                    "downstream_work_all_candidates"
                ]
            ),

        "oracle": {
            **{
                key:
                    int(
                        value
                    )
                for key, value
                in oracle.items()
            },

            "doomed_prefix_edge_fraction":
                (
                    oracle[
                        "doomed_prefix_edges"
                    ]
                    /
                    total_edges
                    if total_edges > 0
                    else 0.0
                ),

            "oracle_avoidable_downstream_edge_fraction":
                (
                    oracle[
                        "oracle_avoidable_downstream_edges"
                    ]
                    /
                    total_edges
                    if total_edges > 0
                    else 0.0
                ),

            "oracle_prunable_intermediate_fraction":
                (
                    oracle[
                        "oracle_prunable_intermediate_branches"
                    ]
                    /
                    oracle[
                        "intermediate_candidate_branches"
                    ]
                    if
                    oracle[
                        "intermediate_candidate_branches"
                    ] > 0
                    else 0.0
                ),
        },
    }


    return summary


WEBQSP_FINAL_RQ1_SUMMARY = (
    build_rq1_summary_16(
        "webqsp",
        webqsp_test_hop_df,
        webqsp_test_plan_df,
        webqsp_test_question_df,
        webqsp_test_oracle
    )
)


CWQ_FINAL_RQ1_SUMMARY = (
    build_rq1_summary_16(
        "cwq",
        cwq_test_hop_df,
        cwq_test_plan_df,
        cwq_test_question_df,
        cwq_test_oracle
    )
)


FINAL_RQ1_TEST_REFERENCE = {
    "webqsp":
        WEBQSP_FINAL_RQ1_SUMMARY,

    "cwq":
        CWQ_FINAL_RQ1_SUMMARY,
}


# ======================================================================
# 15. PRINT CORE RQ1 RESULTS
# ======================================================================

def print_rq1_summary_16(
    summary
):

    print(
        "\n"
        + "=" * 118
    )

    print(
        summary[
            "dataset"
        ].upper(),
        "FINAL FROZEN-TEST RQ1"
    )

    print(
        "=" * 118
    )


    print(
        "questions:",
        summary[
            "questions"
        ]
    )

    print(
        "plans:",
        summary[
            "total_predicted_plans"
        ]
    )

    print(
        "active hop rows:",
        summary[
            "active_hop_rows"
        ]
    )

    print(
        "active prefixes:",
        summary[
            "active_prefixes_total"
        ]
    )

    print(
        "edges examined:",
        summary[
            "edges_examined_total"
        ]
    )

    print(
        "candidate branches:",
        summary[
            "candidate_branches_total"
        ]
    )

    print(
        "RoG reachable questions:",
        summary[
            "reachable_questions"
        ],
        f"({summary['reachable_question_rate']:.4%})"
    )

    print(
        "decision hops:",
        summary[
            "decision_hops"
        ],
        f"({summary['decision_rate_among_intermediate_active']:.4%} "
        "of intermediate-active hops)"
    )


    oracle = summary[
        "oracle"
    ]


    print(
        "\nPOST-FREEZE ORACLE"
    )

    print(
        "doomed-prefix edges:",
        oracle[
            "doomed_prefix_edges"
        ],
        f"({oracle['doomed_prefix_edge_fraction']:.4%})"
    )

    print(
        "oracle-prunable intermediate branches:",
        oracle[
            "oracle_prunable_intermediate_branches"
        ],
        "/",
        oracle[
            "intermediate_candidate_branches"
        ],
        f"({oracle['oracle_prunable_intermediate_fraction']:.4%})"
    )

    print(
        "oracle-avoidable downstream edges:",
        oracle[
            "oracle_avoidable_downstream_edges"
        ],
        f"({oracle['oracle_avoidable_downstream_edge_fraction']:.4%})"
    )

    print(
        "doomed final-hop protected branches:",
        oracle[
            "doomed_final_protected_branches"
        ]
    )


print_rq1_summary_16(
    WEBQSP_FINAL_RQ1_SUMMARY
)


print_rq1_summary_16(
    CWQ_FINAL_RQ1_SUMMARY
)


# ======================================================================
# 16. SAVE RAW TEST ARTIFACTS
# ======================================================================

WEBQSP_RQ1_HOP_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_hops.csv"
)

WEBQSP_RQ1_PLAN_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_plans.csv"
)

WEBQSP_RQ1_Q_CSV = (
    RQ1_TEST_DIR
    / "webqsp_test_rq1_questions.csv"
)


CWQ_RQ1_HOP_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_hops.csv"
)

CWQ_RQ1_PLAN_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_plans.csv"
)

CWQ_RQ1_Q_CSV = (
    RQ1_TEST_DIR
    / "cwq_test_rq1_questions.csv"
)


webqsp_test_hop_df.to_csv(
    WEBQSP_RQ1_HOP_CSV,
    index=False
)

webqsp_test_plan_df.to_csv(
    WEBQSP_RQ1_PLAN_CSV,
    index=False
)

webqsp_test_question_df.to_csv(
    WEBQSP_RQ1_Q_CSV,
    index=False
)


cwq_test_hop_df.to_csv(
    CWQ_RQ1_HOP_CSV,
    index=False
)

cwq_test_plan_df.to_csv(
    CWQ_RQ1_PLAN_CSV,
    index=False
)

cwq_test_question_df.to_csv(
    CWQ_RQ1_Q_CSV,
    index=False
)


# ======================================================================
# 17. SAVE RQ1 SUMMARY JSON
# ======================================================================

RQ1_SUMMARY_JSON = (
    RQ1_TEST_DIR
    / "final_frozen_test_rq1_summary.json"
)


with open(
    RQ1_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "freeze_sha256":
                FINAL_AFP_FREEZE_SHA256,

            "webqsp_test_planning_sha256":
                WEBQSP_TEST_PLAN_SHA,

            "cwq_test_planning_sha256":
                CWQ_TEST_PLAN_SHA,

            "webqsp":
                WEBQSP_FINAL_RQ1_SUMMARY,

            "cwq":
                CWQ_FINAL_RQ1_SUMMARY,

            "oracle_status":
                "post_freeze_test_analysis",

            "test_used_for_development":
                False,
        },
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16_COMPLETE = True

FINAL_RQ1_TEST_COMPLETE = True


print(
    "\n"
    + "=" * 120
)

print(
    "=== CELL 16: FINAL FROZEN-TEST RQ1 PROFILING COMPLETE ==="
)

print(
    "=" * 120
)

print(
    "AFP changed after freeze: NO"
)

print(
    "TEST used for tuning: NO"
)

print(
    "POST-FREEZE oracle analysis: COMPLETE"
)

print(
    "RQ1 final test artifacts:",
    RQ1_TEST_DIR
)

print(
    "\nNEXT: Cell 17 — final frozen-test RQ2 controlled comparison."
)

Cell 16 freeze gate: PASSED
AFP freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116
WEBQSP TEST planning loaded from global: webqsp_test_plan_rows
CWQ TEST planning loaded from global: cwq_test_plan_rows

TEST / validation ID-disjointness: PASSED
WebQSP TEST questions: 1628
CWQ TEST questions: 3531


webqsp_validation_gate final TEST RQ1:   0%|          | 0/246 [00:00<?, ?it/s]

WEBQSP validation oracle software gate: PASSED


cwq_validation_gate final TEST RQ1:   0%|          | 0/3519 [00:00<?, ?it/s]

AssertionError: cwq oracle gate mismatch for oracle_avoidable_downstream_edges: 1341283 != 1176294

In [6]:
# ======================================================================
# CELL 16 ORACLE-MISMATCH DIAGNOSTIC
# RUN AS A NEW CELL AFTER THE CWQ ORACLE GATE FAILURE
# ======================================================================
#
# PURPOSE:
#   Recover evidence about the previously verified oracle implementation
#   and compare it with the currently live Cell-16 implementation.
#
# THIS CELL:
#   - does NOT touch TEST traversal
#   - does NOT tune anything
#   - does NOT modify AFP
#   - does NOT overwrite artifacts
# ======================================================================

from pathlib import Path
import ast
import hashlib
import inspect
import json
import re


print("=" * 100)
print("CELL 16 ORACLE-MISMATCH DIAGNOSTIC")
print("=" * 100)


# ----------------------------------------------------------------------
# 1. Freeze gate
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Known previously verified RQ1 oracle references
# ----------------------------------------------------------------------

KNOWN_WEB = {
    "total_edges": 341526,
    "doomed_prefix_edges": 159039,
    "oracle_avoidable_downstream_edges": 11717,
    "candidate_branches": 7983,
    "intermediate_candidates": 1707,
    "prunable_intermediate_candidates": 1340,
    "doomed_final_protected": 4882,
}

KNOWN_CWQ = {
    "total_edges": 5257272,
    "doomed_prefix_edges": 3197488,
    "oracle_avoidable_downstream_edges": 1176294,
    "candidate_branches": 247161,
    "intermediate_candidates": 41576,
    "prunable_intermediate_candidates": 34431,
    "doomed_final_protected": 195499,
}

print("\nPreviously verified CWQ oracle reference:")
for k, v in KNOWN_CWQ.items():
    print(f"  {k}: {v}")


# ----------------------------------------------------------------------
# 3. Inspect CURRENT live oracle functions
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("A. CURRENT LIVE ORACLE-RELATED FUNCTIONS")
print("=" * 100)

oracle_function_names = []

for name, obj in sorted(globals().items()):

    if (
        callable(obj)
        and
        (
            "oracle" in name.lower()
            or "suffix" in name.lower()
            or "avoidable" in name.lower()
        )
    ):
        oracle_function_names.append(name)


print("Found functions:")
for name in oracle_function_names:
    print(" ", name)


def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


for name in oracle_function_names:

    obj = globals()[name]

    print("\n" + "-" * 100)
    print("FUNCTION:", name)

    try:
        src = inspect.getsource(obj)

        print("SHA256:", sha_text(src))
        print(src)

    except Exception as e:
        print("Could not inspect source:", repr(e))


# ----------------------------------------------------------------------
# 4. Specifically inspect oracle_validation_gate_16
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("B. oracle_validation_gate_16")
print("=" * 100)

assert "oracle_validation_gate_16" in globals()

try:
    gate_src = inspect.getsource(
        oracle_validation_gate_16
    )

    print(
        "oracle_validation_gate_16 SHA:",
        sha_text(gate_src)
    )

    print(gate_src)

except Exception as e:
    print(
        "inspect.getsource failed:",
        repr(e)
    )


# ----------------------------------------------------------------------
# 5. Search persisted notebook for OLD oracle implementation/evidence
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("C. PERSISTED NOTEBOOK ORACLE SEARCH")
print("=" * 100)

assert "persisted_nb_16ar" in globals()


search_terms = [
    "1176294",
    "3197488",
    "34431",
    "195499",
    "oracle_avoidable_downstream_edges",
    "doomed_prefix",
    "doomed_final",
    "suffix_dp",
]


notebook_hits = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    matched = [
        term
        for term in search_terms
        if term in source
    ]

    if matched:

        notebook_hits.append(
            (
                cell_idx,
                matched,
                source
            )
        )


print(
    "Relevant persisted notebook cells:",
    [x[0] for x in notebook_hits]
)


for cell_idx, matched, source in notebook_hits:

    print("\n" + "-" * 100)
    print(
        f"CELL {cell_idx} | matched:",
        matched
    )
    print("-" * 100)

    # Avoid dumping gigantic cells unnecessarily.
    lines = source.splitlines()

    interesting_indices = set()

    for i, line in enumerate(lines):

        if any(
            term in line
            for term in search_terms
        ):
            for j in range(
                max(0, i - 8),
                min(len(lines), i + 15)
            ):
                interesting_indices.add(j)


    for i in sorted(interesting_indices):
        print(
            f"{i + 1:04d}: {lines[i]}"
        )


# ----------------------------------------------------------------------
# 6. Search saved RQ1 DEVELOPMENT artifacts for verified oracle values
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("D. SAVED RQ1 DEVELOPMENT ARTIFACT SEARCH")
print("=" * 100)

RQ1_ROOT = Path(
    "/kaggle/working/step2_rq1_dev"
)

assert RQ1_ROOT.exists()


target_strings = [
    "1176294",
    "3197488",
    "34431",
    "195499",
    "11717",
    "159039",
]


artifact_hits = []


allowed_suffixes = {
    ".json",
    ".jsonl",
    ".csv",
    ".txt",
    ".md",
}


for path in RQ1_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in allowed_suffixes:
        continue

    # Safety: don't scan huge graph files.
    try:
        if path.stat().st_size > 20 * 1024 * 1024:
            continue
    except Exception:
        continue

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    matched = [
        x
        for x in target_strings
        if x in text
    ]

    if matched:
        artifact_hits.append(
            (
                path,
                matched
            )
        )


print(
    "Files containing previously verified oracle values:"
)

for path, matched in artifact_hits:
    print(
        "\n ",
        path
    )
    print(
        "   matched:",
        matched
    )


# ----------------------------------------------------------------------
# 7. Search all small development manifests/results for oracle field names
# ----------------------------------------------------------------------

field_hits = []

for path in RQ1_ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in allowed_suffixes:
        continue

    try:
        if path.stat().st_size > 20 * 1024 * 1024:
            continue

        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue

    if (
        "oracle_avoidable" in text
        or
        "doomed_prefix" in text
        or
        "prunable_intermediate" in text
    ):
        field_hits.append(path)


print(
    "\nFiles containing oracle field names:"
)

for path in field_hits:
    print(" ", path)


# ----------------------------------------------------------------------
# 8. Critical numerical diagnosis
# ----------------------------------------------------------------------

current_cwq = 1341283
frozen_cwq = 1176294

difference = (
    current_cwq
    -
    frozen_cwq
)

print("\n" + "=" * 100)
print("E. NUMERICAL DIFFERENCE")
print("=" * 100)

print(
    "Current CWQ avoidable downstream edges:",
    current_cwq
)

print(
    "Frozen verified value:",
    frozen_cwq
)

print(
    "Difference:",
    difference
)

print(
    "Relative excess:",
    f"{difference / frozen_cwq:.6%}"
)


# ----------------------------------------------------------------------
# 9. Status
# ----------------------------------------------------------------------

print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE")
print("=" * 100)

print(
    "\nNo TEST result was accepted."
)

print(
    "No expected oracle value was changed."
)

print(
    "No AFP parameter was changed."
)

print(
    "No development result was overwritten."
)

print(
    "\nPASTE THE OUTPUT OF THIS CELL HERE."
)

CELL 16 ORACLE-MISMATCH DIAGNOSTIC

Freeze gate: PASSED

Previously verified CWQ oracle reference:
  total_edges: 5257272
  doomed_prefix_edges: 3197488
  oracle_avoidable_downstream_edges: 1176294
  candidate_branches: 247161
  intermediate_candidates: 41576
  prunable_intermediate_candidates: 34431
  doomed_final_protected: 195499

A. CURRENT LIVE ORACLE-RELATED FUNCTIONS
Found functions:
  make_oracle_functions_16
  oracle_validation_gate_16

----------------------------------------------------------------------------------------------------
FUNCTION: make_oracle_functions_16
SHA256: 9ad6a28e261c24931764ba05120d5a31d9f389a266ed059b79b5cab0aca4cf69
def make_oracle_functions_16(
    adjacency,
    gold_answers
):

    gold_set = {
        str(x)
        for x in gold_answers
    }


    reachable_cache = {}

    cost_cache = {}


    def suffix_reachable(
        entity,
        suffix
    ):

        entity = str(
            entity
        )

        suffix = tuple(
            suff

In [7]:
# ======================================================================
# CELL 16 ORACLE REPAIR — EXACT SOURCE EXTRACTION
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun Cell 16
#   - run Cell 17
#   - restart kernel
#   - change expected oracle values
#
# PURPOSE:
#   Recover the EXACT original oracle logic from development Cell 149
#   before modifying the reconstructed Cell-16 implementation.
# ======================================================================

import ast
import hashlib
import inspect


print("=" * 110)
print("EXACT ORIGINAL ORACLE SOURCE EXTRACTION")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Hard scientific gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Helpers
# ----------------------------------------------------------------------

def source_sha_16repair(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def get_cell_source_16repair(cell_index):

    cell = persisted_nb_16ar[
        "cells"
    ][cell_index]

    assert cell.get(
        "cell_type"
    ) == "code"

    return "".join(
        cell.get(
            "source",
            []
        )
    )


# ----------------------------------------------------------------------
# 3. Cell 149 exact function inventory
# ----------------------------------------------------------------------

cell149_source = (
    get_cell_source_16repair(
        149
    )
)

cell149_tree = ast.parse(
    cell149_source
)


cell149_functions = []


for node in cell149_tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        src = ast.get_source_segment(
            cell149_source,
            node
        )

        cell149_functions.append(
            {
                "name": node.name,
                "lineno": node.lineno,
                "end_lineno": getattr(
                    node,
                    "end_lineno",
                    None
                ),
                "sha256": source_sha_16repair(
                    src
                ),
                "source": src,
            }
        )


print(
    "\nCELL 149 FUNCTION INVENTORY"
)

for rec in cell149_functions:

    print(
        f"  {rec['name']:<40} "
        f"lines {rec['lineno']}-{rec['end_lineno']} "
        f"sha={rec['sha256']}"
    )


# ----------------------------------------------------------------------
# 4. Print ALL oracle-relevant Cell-149 functions verbatim
# ----------------------------------------------------------------------

oracle_relevant_149 = []


for rec in cell149_functions:

    src_lower = rec[
        "source"
    ].lower()

    if (
        "oracle" in src_lower
        or
        "productive" in src_lower
        or
        "doomed" in src_lower
        or
        "avoidable" in src_lower
        or
        "downstream" in src_lower
        or
        "suffix" in src_lower
    ):

        oracle_relevant_149.append(
            rec
        )


print(
    "\n"
    + "=" * 110
)

print(
    "ORIGINAL CELL 149 ORACLE-RELEVANT FUNCTIONS — VERBATIM"
)

print(
    "=" * 110
)


for rec in oracle_relevant_149:

    print(
        "\n"
        + "-" * 110
    )

    print(
        "FUNCTION:",
        rec["name"]
    )

    print(
        "LINES:",
        rec["lineno"],
        "-",
        rec["end_lineno"]
    )

    print(
        "SHA256:",
        rec["sha256"]
    )

    print(
        "-" * 110
    )

    print(
        rec["source"]
    )


# ----------------------------------------------------------------------
# 5. Cell 147 DP-function inventory
# ----------------------------------------------------------------------

cell147_source = (
    get_cell_source_16repair(
        147
    )
)

cell147_tree = ast.parse(
    cell147_source
)


cell147_functions = []


for node in cell147_tree.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        )
    ):

        src = ast.get_source_segment(
            cell147_source,
            node
        )

        cell147_functions.append(
            {
                "name": node.name,
                "lineno": node.lineno,
                "end_lineno": getattr(
                    node,
                    "end_lineno",
                    None
                ),
                "sha256": source_sha_16repair(
                    src
                ),
                "source": src,
            }
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 147 SUFFIX/REACHABILITY FUNCTIONS"
)

print(
    "=" * 110
)


for rec in cell147_functions:

    src_lower = rec[
        "source"
    ].lower()

    if (
        "suffix" in src_lower
        or
        "reachable" in src_lower
        or
        "relation" in src_lower
    ):

        print(
            "\n"
            + "-" * 110
        )

        print(
            "FUNCTION:",
            rec["name"]
        )

        print(
            "LINES:",
            rec["lineno"],
            "-",
            rec["end_lineno"]
        )

        print(
            "SHA256:",
            rec["sha256"]
        )

        print(
            "-" * 110
        )

        print(
            rec["source"]
        )


# ----------------------------------------------------------------------
# 6. Current Cell-16 profile function
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "CURRENT profile_rq1_dataset_16 — ORACLE USAGE"
)

print(
    "=" * 110
)


assert (
    "profile_rq1_dataset_16"
    in globals()
)


current_profile_src = (
    inspect.getsource(
        profile_rq1_dataset_16
    )
)


print(
    "SHA256:",
    source_sha_16repair(
        current_profile_src
    )
)


# Print full function because the interaction between
# traversal and oracle accounting is what we need to compare.
print(
    current_profile_src
)


# ----------------------------------------------------------------------
# 7. Current helper dependencies
# ----------------------------------------------------------------------

for fname in [
    "matching_neighbors_16",
    "degree_16",
    "make_oracle_functions_16",
]:

    print(
        "\n"
        + "=" * 110
    )

    print(
        "CURRENT:",
        fname
    )

    print(
        "=" * 110
    )

    assert fname in globals()

    src = inspect.getsource(
        globals()[fname]
    )

    print(
        "SHA256:",
        source_sha_16repair(
            src
        )
    )

    print(
        src
    )


# ----------------------------------------------------------------------
# 8. Saved development summary — exact contents
# ----------------------------------------------------------------------

import pandas as pd
from pathlib import Path


print(
    "\n"
    + "=" * 110
)

print(
    "AUTHORITATIVE SAVED DEVELOPMENT ORACLE SUMMARIES"
)

print(
    "=" * 110
)


for dataset_name in [
    "webqsp",
    "cwq",
]:

    path = Path(
        f"/kaggle/working/step2_rq1_dev/"
        f"oracle_headroom_{dataset_name}_dev_summary.csv"
    )

    assert path.exists()

    df = pd.read_csv(
        path
    )

    print(
        f"\n{dataset_name.upper()}"
    )

    print(
        df.to_string(
            index=False
        )
    )


print(
    "\n"
    + "=" * 110
)

print(
    "EXACT SOURCE EXTRACTION COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nNo TEST evaluation performed."
)

print(
    "No oracle value changed."
)

print(
    "No AFP setting changed."
)

print(
    "No artifact overwritten."
)

print(
    "\nPASTE THIS OUTPUT HERE."
)

EXACT ORIGINAL ORACLE SOURCE EXTRACTION

Freeze gate: PASSED

CELL 149 FUNCTION INVENTORY
  oracle_headroom_for_plan                 lines 18-93 sha=42190812ea03aead6ab97b564ed1829e73ab043fc9678bd3cadf6ab584caef1a
  compute_oracle_headroom                  lines 96-166 sha=295d9d8a0d838167f5b570a818c617806e8076b5dc43d3897ee346e294ec8309
  check_oracle_vs_rq1                      lines 207-239 sha=04050bf25473141a8ec2d7cbaa3e6388b144031acffac4716949aa5fbc1dc22a
  safe_pct                                 lines 258-259 sha=1cc1805de72a0ceb0b0c63511cf158e2933b6422322d38dac344ca2a7c8f1301
  summarize_headroom                       lines 262-474 sha=c46230458a4799d67e71b4969886c6d1ed73fbe03c581e8afc212a26f8e60954
  get_summary_value                        lines 532-533 sha=8761e5ff0ae691de7e42d29d06baefac37331f17c750d7c6c720dc809e03ba60

ORIGINAL CELL 149 ORACLE-RELEVANT FUNCTIONS — VERBATIM

----------------------------------------------------------------------------------------------------

In [9]:
# ======================================================================
# CELL 16-R1
# EXACT ORIGINAL ORACLE-SEMANTICS REPAIR + VALIDATION GATE
# ======================================================================
#
# RUN AS A NEW CELL after the failed Cell 16 oracle gate.
#
# DO NOT:
#   - rerun Cell 16
#   - run Cell 17
#   - restart kernel
#
# This repairs ONLY the reconstructed Cell-16 oracle accounting.
# It does not alter AFP, frozen plans, scorer, selector, or TEST data.
# ======================================================================

import inspect
import textwrap
import hashlib
import json
from pathlib import Path

import pandas as pd


# ======================================================================
# 1. HARD FREEZE GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "profile_rq1_dataset_16" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("Cell 16-R1 freeze gate: PASSED")


# ======================================================================
# 2. VERIFY THAT WE ARE PATCHING THE EXACT FAILED IMPLEMENTATION
# ======================================================================

def sha256_text_16r1(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


profile_source_16r1 = textwrap.dedent(
    inspect.getsource(
        profile_rq1_dataset_16
    )
)


CURRENT_FAILED_PROFILE_SHA = (
    "cf1f74072bb777653005682d48e65caf8544bcc9d636752ff918a5fec792e6c9"
)


actual_failed_sha = sha256_text_16r1(
    profile_source_16r1
)


print("\nCurrent failed profile SHA:")
print(" ", actual_failed_sha)


assert (
    actual_failed_sha
    ==
    CURRENT_FAILED_PROFILE_SHA
), (
    "profile_rq1_dataset_16 is not the exact implementation "
    "diagnosed previously. Stop rather than patch unknown code."
)


print("Failed Cell-16 profile identity: PASSED")


# ======================================================================
# 3. PATCH 1
#
# ORIGINAL CELL-149 SEMANTICS:
#
# If the CURRENT prefix is doomed and h > 0, every adjacency edge
# examined from that prefix is avoidable because that prefix could
# have been pruned after the previous intermediate candidate set.
#
# Therefore:
#
# avoidable_this_hop =
#     doomed_prefix_edges if hop > 0 else 0
#
# It is NOT recursive future subtree cost assigned to the candidate.
# ======================================================================

old_1 = '''                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges
'''


new_1 = '''                oracle_totals[
                    "doomed_prefix_edges"
                ] += doomed_prefix_edges


                # ------------------------------------------------------
                # EXACT ORIGINAL CELL-149 ORACLE SEMANTICS
                #
                # A doomed prefix at hop > 0 was generated at the
                # preceding intermediate hop and could therefore have
                # been removed before THIS expansion occurred.
                #
                # Count the actual adjacency work paid at this hop.
                # Do not recursively assign its whole future subtree
                # to the candidate that generated it.
                # ------------------------------------------------------

                oracle_avoidable_this_hop = int(
                    doomed_prefix_edges
                    if hop > 0
                    else 0
                )


                oracle_totals[
                    "oracle_avoidable_downstream_edges"
                ] += oracle_avoidable_this_hop


                plan_oracle_avoidable += (
                    oracle_avoidable_this_hop
                )
'''


assert profile_source_16r1.count(old_1) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_1,
        new_1,
        1
    )
)


# ======================================================================
# 4. PATCH 2
# Do not reset oracle_avoidable_this_hop after it was computed from
# the current doomed-prefix expansion.
# ======================================================================

old_2 = '''                oracle_prunable_this_hop = 0
                oracle_avoidable_this_hop = 0
                downstream_work_all_candidates = 0
'''


new_2 = '''                oracle_prunable_this_hop = 0

                # oracle_avoidable_this_hop was already computed above
                # from CURRENT doomed-prefix adjacency work, matching
                # the original Cell-149 definition.

                downstream_work_all_candidates = 0
'''


assert profile_source_16r1.count(old_2) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_2,
        new_2,
        1
    )
)


# ======================================================================
# 5. PATCH 3
# Candidate feasibility still determines whether an INTERMEDIATE
# candidate is oracle-prunable.
#
# But recursive future_cost must NOT be added to the authoritative
# oracle-avoidable edge counter.
# ======================================================================

old_3 = '''                        if not feasible:

                            oracle_prunable_this_hop += 1

                            oracle_avoidable_this_hop += (
                                future_cost
                            )
'''


new_3 = '''                        if not feasible:

                            oracle_prunable_this_hop += 1
'''


assert profile_source_16r1.count(old_3) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_3,
        new_3,
        1
    )
)


# ======================================================================
# 6. PATCH 4
# Remove the old recursive candidate-level addition to global oracle
# avoidable edges. It is now counted once at actual prefix expansion.
# ======================================================================

old_4 = '''                    oracle_totals[
                        "oracle_avoidable_downstream_edges"
                    ] += oracle_avoidable_this_hop
'''


assert profile_source_16r1.count(old_4) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_4,
        "",
        1
    )
)


# ======================================================================
# 7. PATCH 5
# Same correction for per-plan oracle avoidable work.
# ======================================================================

old_5 = '''                    plan_oracle_avoidable += (
                        oracle_avoidable_this_hop
                    )
'''


assert profile_source_16r1.count(old_5) == 1

profile_source_16r1 = (
    profile_source_16r1.replace(
        old_5,
        "",
        1
    )
)


# ======================================================================
# 8. INSTALL REPAIRED FUNCTION IN CURRENT KERNEL
# ======================================================================

CELL16_REPAIRED_PROFILE_SOURCE = (
    profile_source_16r1
)

CELL16_REPAIRED_PROFILE_SHA256 = (
    sha256_text_16r1(
        CELL16_REPAIRED_PROFILE_SOURCE
    )
)


exec(
    CELL16_REPAIRED_PROFILE_SOURCE,
    globals()
)


print("\nRepaired profile installed.")
print(
    "Repaired profile SHA:",
    CELL16_REPAIRED_PROFILE_SHA256
)


# ======================================================================
# 9. AUTHORITATIVE FROZEN VALIDATION REFERENCES
# ======================================================================

EXPECTED_WEB_16R1 = {
    "total_edges":
        341526,

    "doomed_prefix_edges":
        159039,

    "oracle_avoidable_downstream_edges":
        11717,

    "candidate_branches":
        7983,

    "intermediate_candidate_branches":
        1707,

    "oracle_prunable_intermediate_branches":
        1340,

    "doomed_final_protected_branches":
        4882,
}


EXPECTED_CWQ_16R1 = {
    "total_edges":
        5257272,

    "doomed_prefix_edges":
        3197488,

    "oracle_avoidable_downstream_edges":
        1176294,

    "candidate_branches":
        247161,

    "intermediate_candidate_branches":
        41576,

    "oracle_prunable_intermediate_branches":
        34431,

    "doomed_final_protected_branches":
        195499,
}


# ======================================================================
# 10. RUN REPAIRED WEBQSP VALIDATION
# ======================================================================

print(
    "\nRunning repaired WebQSP oracle validation..."
)


(
    web_hop_16r1,
    web_plan_16r1,
    web_question_16r1,
    web_oracle_16r1,
) = profile_rq1_dataset_16(
    "webqsp_validation_repair_gate",
    webqsp_val_plan_rows
)


for key, expected in EXPECTED_WEB_16R1.items():

    actual = int(
        web_oracle_16r1[
            key
        ]
    )

    assert actual == expected, (
        f"WebQSP repaired oracle mismatch: "
        f"{key}: {actual} != {expected}"
    )


print(
    "WEBQSP repaired oracle totals: EXACT MATCH"
)


# ======================================================================
# 11. RUN REPAIRED CWQ VALIDATION
# ======================================================================

print(
    "\nRunning repaired CWQ oracle validation..."
)


(
    cwq_hop_16r1,
    cwq_plan_16r1,
    cwq_question_16r1,
    cwq_oracle_16r1,
) = profile_rq1_dataset_16(
    "cwq_validation_repair_gate",
    cwq_val_plan_rows
)


for key, expected in EXPECTED_CWQ_16R1.items():

    actual = int(
        cwq_oracle_16r1[
            key
        ]
    )

    assert actual == expected, (
        f"CWQ repaired oracle mismatch: "
        f"{key}: {actual} != {expected}"
    )


print(
    "CWQ repaired oracle totals: EXACT MATCH"
)


# ======================================================================
# 12. PER-HOP FIDELITY AGAINST SAVED ORIGINAL CELL-149 ARTIFACTS
#
# This verifies not only the total, but the HOP ATTRIBUTION.
# ======================================================================

def verify_per_hop_16r1(
    dataset_name,
    new_hop_df
):

    old_path = Path(
        "/kaggle/working/step2_rq1_dev/"
        f"oracle_headroom_{dataset_name}_dev_hop.csv"
    )

    assert old_path.exists(), old_path

    old_df = pd.read_csv(
        old_path
    )


    required_old = {
        "hop",
        "oracle_avoidable_downstream_edges",
        "oracle_prunable_branches",
    }

    assert required_old.issubset(
        set(old_df.columns)
    )


    new_agg = (
        new_hop_df
        .groupby(
            "hop",
            as_index=False
        )
        .agg(
            oracle_avoidable_downstream_edges=(
                "oracle_avoidable_downstream_edges",
                "sum"
            ),

            oracle_prunable_branches=(
                "oracle_prunable_candidates",
                "sum"
            ),
        )
    )


    old_cmp = (
        old_df[
            [
                "hop",
                "oracle_avoidable_downstream_edges",
                "oracle_prunable_branches",
            ]
        ]
        .copy()
    )


    all_hops = sorted(
        set(
            old_cmp["hop"].tolist()
        )
        |
        set(
            new_agg["hop"].tolist()
        )
    )


    old_cmp = (
        old_cmp
        .set_index("hop")
        .reindex(
            all_hops,
            fill_value=0
        )
    )


    new_cmp = (
        new_agg
        .set_index("hop")
        .reindex(
            all_hops,
            fill_value=0
        )
    )


    for col in [
        "oracle_avoidable_downstream_edges",
        "oracle_prunable_branches",
    ]:

        old_values = (
            old_cmp[col]
            .astype("int64")
            .tolist()
        )

        new_values = (
            new_cmp[col]
            .astype("int64")
            .tolist()
        )

        assert old_values == new_values, (
            f"{dataset_name} per-hop mismatch "
            f"for {col}:\n"
            f"old={old_values}\n"
            f"new={new_values}"
        )


    print(
        f"{dataset_name.upper()} per-hop "
        "oracle fidelity: PASSED"
    )


verify_per_hop_16r1(
    "webqsp",
    web_hop_16r1
)


verify_per_hop_16r1(
    "cwq",
    cwq_hop_16r1
)


# ======================================================================
# 13. ORDINARY RoG TRAVERSAL MUST REMAIN UNCHANGED
# ======================================================================

assert int(
    web_hop_16r1[
        "edges_examined"
    ].sum()
) == 341526

assert int(
    web_hop_16r1[
        "candidate_branches"
    ].sum()
) == 7983


assert int(
    cwq_hop_16r1[
        "edges_examined"
    ].sum()
) == 5257272

assert int(
    cwq_hop_16r1[
        "candidate_branches"
    ].sum()
) == 247161


assert int(
    web_question_16r1[
        "reachable"
    ].sum()
) == 205

assert int(
    cwq_question_16r1[
        "reachable"
    ].sum()
) == 2425


print(
    "\nOrdinary RoG validation traversal unchanged: PASSED"
)


# ======================================================================
# 14. SAVE REPAIR AUDIT MANIFEST
# ======================================================================

repair_manifest_16r1 = {
    "repair":
        "cell16_exact_original_oracle_semantics",

    "development_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "failed_profile_sha256":
        CURRENT_FAILED_PROFILE_SHA,

    "repaired_profile_sha256":
        CELL16_REPAIRED_PROFILE_SHA256,

    "original_cell149_oracle_headroom_for_plan_sha256":
        "42190812ea03aead6ab97b564ed1829e73ab043fc9678bd3cadf6ab584caef1a",

    "original_cell147_suffix_reachable_dp_sha256":
        "85a1fea859e8eb9e262efe6a807d8845ea9e69a287890353a279d21295551325",

    "webqsp_validation_oracle":
        {
            k: int(v)
            for k, v
            in web_oracle_16r1.items()
        },

    "cwq_validation_oracle":
        {
            k: int(v)
            for k, v
            in cwq_oracle_16r1.items()
        },

    "test_evaluated_in_this_repair":
        False,

    "afp_changed":
        False,

    "test_tuning":
        False,
}


repair_manifest_path_16r1 = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_oracle_repair_manifest.json"
)


repair_manifest_path_16r1.parent.mkdir(
    parents=True,
    exist_ok=True
)


with open(
    repair_manifest_path_16r1,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        repair_manifest_16r1,
        f,
        indent=2,
        sort_keys=True
    )


CELL16_ORACLE_REPAIR_VALIDATED = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16 ORACLE REPAIR: VALIDATED"
)

print(
    "=" * 110
)


print(
    "\nWebQSP oracle avoidable edges:",
    web_oracle_16r1[
        "oracle_avoidable_downstream_edges"
    ]
)

print(
    "CWQ oracle avoidable edges:",
    cwq_oracle_16r1[
        "oracle_avoidable_downstream_edges"
    ]
)


print(
    "\nExpected:"
)

print(
    " WebQSP = 11717"
)

print(
    " CWQ    = 1176294"
)


print(
    "\nPer-hop original Cell-149 fidelity: PASSED"
)

print(
    "Ordinary RoG traversal fidelity:   PASSED"
)

print(
    "AFP changed:                       NO"
)

print(
    "TEST evaluated by repair cell:     NO"
)

print(
    "TEST tuning:                       NO"
)


print(
    "\nManifest:"
)

print(
    " ",
    repair_manifest_path_16r1
)


print(
    "\nIMPORTANT:"
)

print(
    "DO NOT RERUN THE ORIGINAL CELL 16 YET."
)

print(
    "It would redefine the old reconstructed profiler."
)

print(
    "Paste this cell's final output here."
)

Cell 16-R1 freeze gate: PASSED


OSError: could not get source code

In [10]:
# ======================================================================
# CELL 16-R1A
# RECOVER / RESUME AFTER inspect.getsource OSError
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun Cell 16-R1
#   - rerun original Cell 16
#   - run Cell 17
#   - restart kernel
# ======================================================================

from pathlib import Path
import hashlib
import json


print("=" * 100)
print("CELL 16-R1 PARTIAL-EXECUTION STATUS")
print("=" * 100)


# ----------------------------------------------------------------------
# 1. Freeze state
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. Did first Cell 16-R1 create repaired source?
# ----------------------------------------------------------------------

has_repaired_source = (
    "CELL16_REPAIRED_PROFILE_SOURCE"
    in globals()
)

has_repaired_sha = (
    "CELL16_REPAIRED_PROFILE_SHA256"
    in globals()
)

print(
    "\nCELL16_REPAIRED_PROFILE_SOURCE exists:",
    has_repaired_source
)

print(
    "CELL16_REPAIRED_PROFILE_SHA256 exists:",
    has_repaired_sha
)

if has_repaired_sha:

    print(
        "Repaired SHA:",
        CELL16_REPAIRED_PROFILE_SHA256
    )


# ----------------------------------------------------------------------
# 3. Is current profiler callable?
# ----------------------------------------------------------------------

print(
    "\nprofile_rq1_dataset_16 exists:",
    "profile_rq1_dataset_16"
    in globals()
)

print(
    "profile_rq1_dataset_16 callable:",
    callable(
        globals().get(
            "profile_rq1_dataset_16",
            None
        )
    )
)


# ----------------------------------------------------------------------
# 4. Did WebQSP/CWQ repaired validation already execute?
# ----------------------------------------------------------------------

for name in [
    "web_oracle_16r1",
    "cwq_oracle_16r1",
]:

    print(
        f"\n{name} exists:",
        name in globals()
    )

    if name in globals():

        print(
            globals()[name]
        )


print(
    "\nCELL16_ORACLE_REPAIR_VALIDATED:",
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
)


# ----------------------------------------------------------------------
# 5. Repair manifest
# ----------------------------------------------------------------------

manifest_path = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_oracle_repair_manifest.json"
)

print(
    "\nRepair manifest exists:",
    manifest_path.exists()
)

if manifest_path.exists():

    print(
        "Manifest path:",
        manifest_path
    )

    with open(
        manifest_path,
        "r",
        encoding="utf-8"
    ) as f:

        manifest = json.load(f)

    print(
        "Manifest test_evaluated_in_this_repair:",
        manifest.get(
            "test_evaluated_in_this_repair"
        )
    )

    print(
        "Manifest AFP changed:",
        manifest.get(
            "afp_changed"
        )
    )


# ----------------------------------------------------------------------
# 6. Frozen TEST plans still safe
# ----------------------------------------------------------------------

WEB_TEST = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)

print("\nFrozen TEST planning files:")
print(" WebQSP exists:", WEB_TEST.exists())
print(" CWQ exists:   ", CWQ_TEST.exists())


# ----------------------------------------------------------------------
# 7. If repaired source exists, verify its stored SHA directly
# ----------------------------------------------------------------------

if has_repaired_source:

    actual_sha = hashlib.sha256(
        CELL16_REPAIRED_PROFILE_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()

    print(
        "\nStored repaired source SHA:",
        actual_sha
    )

    if has_repaired_sha:

        assert (
            actual_sha
            ==
            CELL16_REPAIRED_PROFILE_SHA256
        )

        print(
            "Stored repaired-source integrity: PASSED"
        )


print(
    "\n"
    + "=" * 100
)

print(
    "STATUS CHECK COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nPaste this output here."
)

CELL 16-R1 PARTIAL-EXECUTION STATUS

Freeze gate: PASSED

CELL16_REPAIRED_PROFILE_SOURCE exists: True
CELL16_REPAIRED_PROFILE_SHA256 exists: True
Repaired SHA: 3413dc5efa00a1be3f9c56f79677bbd97232293b9fb4d15d5a562bacb434287f

profile_rq1_dataset_16 exists: True
profile_rq1_dataset_16 callable: True

web_oracle_16r1 exists: True
{'total_edges': 341526, 'doomed_prefix_edges': 159039, 'candidate_branches': 7983, 'intermediate_candidate_branches': 1707, 'oracle_prunable_intermediate_branches': 1340, 'doomed_final_protected_branches': 4882, 'oracle_avoidable_downstream_edges': 11717}

cwq_oracle_16r1 exists: True
{'total_edges': 5257272, 'doomed_prefix_edges': 3197488, 'candidate_branches': 247161, 'intermediate_candidate_branches': 41576, 'oracle_prunable_intermediate_branches': 34431, 'doomed_final_protected_branches': 195499, 'oracle_avoidable_downstream_edges': 1176294}

CELL16_ORACLE_REPAIR_VALIDATED: True

Repair manifest exists: True
Manifest path: /kaggle/working/step2_rq1_test/cell

In [11]:
# ======================================================================
# CELL 16-R2
# RESUME ORIGINAL CELL 16 AFTER VALIDATED ORACLE REPAIR
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun original Cell 16
#   - rerun Cell 16-R1
#   - rerun M3
#   - restart kernel
#
# This executes ONLY the original Cell-16 statements that came AFTER
# the CWQ validation oracle gate.
# ======================================================================

import ast
import hashlib
from pathlib import Path


print("=" * 110)
print("CELL 16-R2 — SAFE RESUME")
print("=" * 110)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

assert "profile_rq1_dataset_16" in globals()
assert callable(profile_rq1_dataset_16)

assert "web_oracle_16r1" in globals()
assert "cwq_oracle_16r1" in globals()


# Exact repaired validation evidence
assert (
    int(
        web_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 11717
)

assert (
    int(
        cwq_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 1176294
)


print("Freeze gate:                 PASSED")
print("Oracle repair validation:    PASSED")
print("Repaired profiler in memory: YES")


# ======================================================================
# 2. VERIFY FROZEN TEST PLANNING ARTIFACTS
# ======================================================================

WEB_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)


assert WEB_TEST_PATH.exists()
assert CWQ_TEST_PATH.exists()


def file_sha256_16r2(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


WEB_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


web_sha_16r2 = file_sha256_16r2(
    WEB_TEST_PATH
)

cwq_sha_16r2 = file_sha256_16r2(
    CWQ_TEST_PATH
)


assert web_sha_16r2 == WEB_EXPECTED_SHA
assert cwq_sha_16r2 == CWQ_EXPECTED_SHA


print("\nFrozen TEST planning SHA gate: PASSED")
print(" WebQSP:", web_sha_16r2)
print(" CWQ:   ", cwq_sha_16r2)


# ======================================================================
# 3. RECOVER EXACT ORIGINAL CELL 16 FROM IPYTHON HISTORY
# ======================================================================

ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


cell16_candidates = []


for history_index, source in enumerate(
    history
):

    if not isinstance(source, str):
        continue

    # Strong signature of the original Cell 16.
    if (
        "def profile_rq1_dataset_16" in source
        and
        "def oracle_validation_gate_16" in source
        and
        "EXPECTED_VAL_ORACLE_16" in source
        and
        "oracle_validation_gate_16" in source
    ):

        cell16_candidates.append(
            (
                history_index,
                source
            )
        )


assert cell16_candidates, (
    "Could not find the executed original Cell 16 "
    "in IPython input history."
)


# Use the latest matching execution.
CELL16_HISTORY_INDEX, CELL16_ORIGINAL_SOURCE = (
    cell16_candidates[-1]
)


CELL16_ORIGINAL_SOURCE_SHA256 = (
    hashlib.sha256(
        CELL16_ORIGINAL_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nOriginal Cell 16 recovered from history:"
)

print(
    " history index:",
    CELL16_HISTORY_INDEX
)

print(
    " source SHA256:",
    CELL16_ORIGINAL_SOURCE_SHA256
)


# ======================================================================
# 4. FIND THE TWO TOP-LEVEL VALIDATION-GATE CALLS
# ======================================================================

cell16_tree = ast.parse(
    CELL16_ORIGINAL_SOURCE
)


gate_nodes = []


for body_index, node in enumerate(
    cell16_tree.body
):

    if not isinstance(
        node,
        ast.Expr
    ):
        continue

    call = node.value

    if not isinstance(
        call,
        ast.Call
    ):
        continue

    func = call.func

    if (
        isinstance(
            func,
            ast.Name
        )
        and
        func.id
        ==
        "oracle_validation_gate_16"
    ):

        gate_nodes.append(
            (
                body_index,
                node
            )
        )


print(
    "\nTop-level validation gates found:",
    len(gate_nodes)
)


assert len(gate_nodes) == 2, (
    "Expected exactly two top-level validation "
    "oracle gates (WebQSP and CWQ)."
)


first_gate_idx, first_gate_node = (
    gate_nodes[0]
)

second_gate_idx, second_gate_node = (
    gate_nodes[1]
)


print(
    " WebQSP gate line:",
    first_gate_node.lineno
)

print(
    " CWQ gate line:",
    second_gate_node.lineno
)


# ======================================================================
# 5. BUILD EXACT POST-GATE RESUME MODULE
# ======================================================================
#
# Original Cell 16 failed while executing the SECOND validation gate.
#
# Everything before it already executed successfully.
#
# We therefore execute only the top-level statements AFTER that gate.
# ======================================================================

resume_nodes = (
    cell16_tree.body[
        second_gate_idx + 1:
    ]
)


assert len(resume_nodes) > 0, (
    "No Cell-16 statements exist after the CWQ gate."
)


resume_module = ast.Module(
    body=resume_nodes,
    type_ignores=[]
)


ast.fix_missing_locations(
    resume_module
)


# Raw source after the exact second gate, for audit only.
source_lines = (
    CELL16_ORIGINAL_SOURCE
    .splitlines(
        keepends=True
    )
)


CELL16_POST_GATE_RAW_SOURCE = "".join(
    source_lines[
        second_gate_node.end_lineno:
    ]
)


CELL16_POST_GATE_SHA256 = (
    hashlib.sha256(
        CELL16_POST_GATE_RAW_SOURCE.encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nPost-CWQ-gate resume source:"
)

print(
    " statements:",
    len(resume_nodes)
)

print(
    " SHA256:",
    CELL16_POST_GATE_SHA256
)


# ======================================================================
# 6. SAFETY CHECKS ON REMAINDER
# ======================================================================

# The remainder must NOT redefine the profiler we just repaired.
assert (
    "def profile_rq1_dataset_16"
    not in
    CELL16_POST_GATE_RAW_SOURCE
), (
    "Unsafe resume: remainder unexpectedly "
    "redefines profile_rq1_dataset_16."
)


# It must not redefine the oracle repair.
assert (
    "def make_oracle_functions_16"
    not in
    CELL16_POST_GATE_RAW_SOURCE
)


print(
    "Resume-source safety checks: PASSED"
)


# ======================================================================
# 7. EXECUTE THE ORIGINAL CELL-16 REMAINDER
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "RESUMING ORIGINAL CELL 16 AFTER CWQ VALIDATION GATE"
)

print(
    "=" * 110
)

print(
    "\nUsing repaired, validation-proven "
    "profile_rq1_dataset_16."
)


CELL16_R2_RESUME_STARTED = True


exec(
    compile(
        resume_module,
        filename="<CELL16_R2_EXACT_RESUME>",
        mode="exec"
    ),
    globals()
)


CELL16_R2_RESUME_COMPLETE = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2 RESUME COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nOriginal Cell 16 rerun: NO"
)

print(
    "M3 rerun:              NO"
)

print(
    "AFP changed:           NO"
)

print(
    "TEST tuning:           NO"
)

print(
    "Frozen TEST plans:     UNCHANGED"
)

print(
    "\nNEXT ACTION:"
)

print(
    "Paste the final Cell 16-R2 output here "
    "before running Cell 17."
)

CELL 16-R2 — SAFE RESUME
Freeze gate:                 PASSED
Oracle repair validation:    PASSED
Repaired profiler in memory: YES

Frozen TEST planning SHA gate: PASSED
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ:    95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

Original Cell 16 recovered from history:
 history index: 11
 source SHA256: 036e3f0d2e06d59edc9de544a2d92d08cf9d7fe6aa3337b5db46b9ab60f5b67a

Top-level validation gates found: 0


AssertionError: Expected exactly two top-level validation oracle gates (WebQSP and CWQ).

In [12]:
# ======================================================================
# CELL 16-R2A
# LOCATE ORIGINAL VALIDATION-GATE CALLS EXACTLY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# This is diagnostic only.
# No TEST profiling.
# No AFP changes.
# No files overwritten.
# ======================================================================

import ast
import hashlib


print("=" * 110)
print("CELL 16-R2A — EXACT VALIDATION-GATE LOCATION")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Scientific-state gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

print("\nFreeze + oracle repair gates: PASSED")


# ----------------------------------------------------------------------
# 2. Reuse exact Cell-16 source already recovered by R2
# ----------------------------------------------------------------------

if "CELL16_ORIGINAL_SOURCE" in globals():

    source = CELL16_ORIGINAL_SOURCE

    print(
        "\nUsing CELL16_ORIGINAL_SOURCE already recovered."
    )

else:

    ip = get_ipython()

    history = list(
        ip.history_manager.input_hist_raw
    )

    candidates = []

    for history_index, src in enumerate(history):

        if not isinstance(src, str):
            continue

        if (
            "def profile_rq1_dataset_16" in src
            and
            "def oracle_validation_gate_16" in src
            and
            "EXPECTED_VAL_ORACLE_16" in src
        ):

            candidates.append(
                (history_index, src)
            )

    assert candidates, (
        "Could not recover original Cell 16 source."
    )

    CELL16_HISTORY_INDEX, source = (
        candidates[-1]
    )

    CELL16_ORIGINAL_SOURCE = source


print(
    "Cell-16 SHA256:",
    hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()
)


# ----------------------------------------------------------------------
# 3. Parse source and build parent map
# ----------------------------------------------------------------------

tree = ast.parse(source)

parent = {}

for node in ast.walk(tree):

    for child in ast.iter_child_nodes(node):

        parent[child] = node


# ----------------------------------------------------------------------
# 4. Find ALL calls to oracle_validation_gate_16 anywhere
# ----------------------------------------------------------------------

gate_calls = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    if (
        isinstance(func, ast.Name)
        and
        func.id == "oracle_validation_gate_16"
    ):

        gate_calls.append(node)


gate_calls = sorted(
    gate_calls,
    key=lambda n: (
        n.lineno,
        n.col_offset
    )
)


print(
    "\nTotal oracle_validation_gate_16 calls found:",
    len(gate_calls)
)


# ----------------------------------------------------------------------
# 5. Find enclosing top-level statement for each call
# ----------------------------------------------------------------------

def enclosing_top_level(node):

    current = node

    while current in parent:

        p = parent[current]

        if p is tree:

            return current

        current = p

    return None


lines = source.splitlines()


for i, call in enumerate(
    gate_calls,
    start=1
):

    top = enclosing_top_level(call)

    print(
        "\n"
        + "-" * 110
    )

    print(
        f"GATE CALL #{i}"
    )

    print(
        "call lines:",
        call.lineno,
        "-",
        getattr(
            call,
            "end_lineno",
            call.lineno
        )
    )

    print(
        "column:",
        call.col_offset
    )

    if top is not None:

        print(
            "enclosing TOP-LEVEL type:",
            type(top).__name__
        )

        print(
            "top-level lines:",
            top.lineno,
            "-",
            getattr(
                top,
                "end_lineno",
                top.lineno
            )
        )


    # Ancestor chain
    chain = []

    cur = call

    while cur in parent:

        cur = parent[cur]

        chain.append(
            type(cur).__name__
        )

        if cur is tree:
            break

    print(
        "ancestor chain:",
        " -> ".join(chain)
    )


    # Exact call source
    call_src = ast.get_source_segment(
        source,
        call
    )

    print(
        "\nExact call:"
    )

    print(
        call_src
    )


    # Surrounding source
    start = max(
        0,
        call.lineno - 15
    )

    end = min(
        len(lines),
        getattr(
            call,
            "end_lineno",
            call.lineno
        ) + 20
    )

    print(
        "\nSurrounding source:"
    )

    for line_no in range(
        start,
        end
    ):

        marker = (
            ">>>"
            if (
                call.lineno - 1
                <= line_no
                <=
                getattr(
                    call,
                    "end_lineno",
                    call.lineno
                ) - 1
            )
            else "   "
        )

        print(
            f"{marker} "
            f"{line_no + 1:04d}: "
            f"{lines[line_no]}"
        )


# ----------------------------------------------------------------------
# 6. Raw text occurrence check
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "RAW SOURCE OCCURRENCES"
)

print(
    "=" * 110
)


for line_no, line in enumerate(
    lines,
    start=1
):

    if (
        "oracle_validation_gate_16("
        in line
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ----------------------------------------------------------------------
# 7. Check what comes immediately after the second gate
# ----------------------------------------------------------------------

if len(gate_calls) >= 2:

    second = gate_calls[1]

    second_end = getattr(
        second,
        "end_lineno",
        second.lineno
    )

    print(
        "\n"
        + "=" * 110
    )

    print(
        "SOURCE AFTER SECOND VALIDATION GATE"
    )

    print(
        "=" * 110
    )

    for line_no in range(
        second_end,
        min(
            len(lines),
            second_end + 60
        )
    ):

        print(
            f"{line_no + 1:04d}: "
            f"{lines[line_no]}"
        )


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2A COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nNo TEST profiling executed."
)

print(
    "No AFP setting changed."
)

print(
    "No artifact overwritten."
)

print(
    "\nPaste this output here."
)

CELL 16-R2A — EXACT VALIDATION-GATE LOCATION

Freeze + oracle repair gates: PASSED

Using CELL16_ORIGINAL_SOURCE already recovered.
Cell-16 SHA256: 036e3f0d2e06d59edc9de544a2d92d08cf9d7fe6aa3337b5db46b9ab60f5b67a

Total oracle_validation_gate_16 calls found: 0

RAW SOURCE OCCURRENCES

CELL 16-R2A COMPLETE

No TEST profiling executed.
No AFP setting changed.
No artifact overwritten.

Paste this output here.


In [13]:
# ======================================================================
# CELL 16-R2B
# ROBUST ORIGINAL CELL-16 LOCATOR + SAFE POST-GATE RESUME
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun original Cell 16
#   - rerun Cell 16-R1
#   - rerun Cell 16-R2 / R2A
#   - rerun M3
#   - restart kernel
#
# This locates the REAL original Cell 16 using AST structure,
# not string matching.
# ======================================================================

import ast
import hashlib
from pathlib import Path


print("=" * 110)
print("CELL 16-R2B — ROBUST SAFE RESUME")
print("=" * 110)


# ======================================================================
# 1. SCIENTIFIC STATE GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    globals().get(
        "CELL16_ORACLE_REPAIR_VALIDATED",
        False
    )
    is True
)

assert "profile_rq1_dataset_16" in globals()
assert callable(profile_rq1_dataset_16)

assert "web_oracle_16r1" in globals()
assert "cwq_oracle_16r1" in globals()


assert (
    int(
        web_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 11717
)

assert (
    int(
        cwq_oracle_16r1[
            "oracle_avoidable_downstream_edges"
        ]
    )
    == 1176294
)


print("Freeze gate:              PASSED")
print("Oracle repair validation: PASSED")


# ======================================================================
# 2. VERIFY FROZEN TEST PLAN FILES AGAIN
# ======================================================================

WEB_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_TEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)


assert WEB_TEST_PATH.exists()
assert CWQ_TEST_PATH.exists()


def file_sha256_r2b(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


WEB_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


assert (
    file_sha256_r2b(WEB_TEST_PATH)
    ==
    WEB_EXPECTED_SHA
)

assert (
    file_sha256_r2b(CWQ_TEST_PATH)
    ==
    CWQ_EXPECTED_SHA
)


print("Frozen TEST plan SHA gate: PASSED")


# ======================================================================
# 3. SEARCH IPYTHON HISTORY BY ACTUAL AST DEFINITIONS
# ======================================================================

ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


real_candidates = []


for hist_index, source in enumerate(history):

    if not isinstance(source, str):
        continue

    if len(source) < 1000:
        continue

    try:

        tree = ast.parse(source)

    except Exception:

        continue


    # --------------------------------------------------------------
    # REAL function definitions, not text appearing inside strings
    # --------------------------------------------------------------

    top_level_function_names = {
        node.name
        for node in tree.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
    }


    if (
        "profile_rq1_dataset_16"
        not in
        top_level_function_names
    ):
        continue


    if (
        "oracle_validation_gate_16"
        not in
        top_level_function_names
    ):
        continue


    # --------------------------------------------------------------
    # REAL function calls anywhere in AST
    # --------------------------------------------------------------

    gate_calls = []


    for node in ast.walk(tree):

        if not isinstance(
            node,
            ast.Call
        ):
            continue

        func = node.func

        if (
            isinstance(
                func,
                ast.Name
            )
            and
            func.id
            ==
            "oracle_validation_gate_16"
        ):

            gate_calls.append(
                node
            )


    gate_calls = sorted(
        gate_calls,
        key=lambda x:
            (
                x.lineno,
                x.col_offset
            )
    )


    if len(gate_calls) != 2:
        continue


    real_candidates.append(
        {
            "history_index":
                hist_index,

            "source":
                source,

            "tree":
                tree,

            "gate_calls":
                gate_calls,

            "sha256":
                hashlib.sha256(
                    source.encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "chars":
                len(source),
        }
    )


print(
    "\nREAL Cell-16 candidates found:",
    len(real_candidates)
)


for candidate in real_candidates:

    print(
        " history=",
        candidate[
            "history_index"
        ],

        "| chars=",
        candidate[
            "chars"
        ],

        "| SHA=",
        candidate[
            "sha256"
        ],

        "| gates=",
        [
            x.lineno
            for x in
            candidate[
                "gate_calls"
            ]
        ]
    )


assert real_candidates, (
    "No IPython history entry actually defines "
    "profile_rq1_dataset_16 + oracle_validation_gate_16 "
    "and contains exactly two real gate calls."
)


# Prefer the latest actual definition cell.
original = sorted(
    real_candidates,
    key=lambda x:
        x["history_index"]
)[-1]


CELL16_TRUE_HISTORY_INDEX = (
    original[
        "history_index"
    ]
)

CELL16_TRUE_ORIGINAL_SOURCE = (
    original[
        "source"
    ]
)

CELL16_TRUE_ORIGINAL_SHA256 = (
    original[
        "sha256"
    ]
)

cell16_true_tree = (
    original[
        "tree"
    ]
)

gate_calls = (
    original[
        "gate_calls"
    ]
)


print(
    "\nSelected REAL original Cell 16:"
)

print(
    " history index:",
    CELL16_TRUE_HISTORY_INDEX
)

print(
    " characters:",
    len(
        CELL16_TRUE_ORIGINAL_SOURCE
    )
)

print(
    " SHA256:",
    CELL16_TRUE_ORIGINAL_SHA256
)

print(
    " gate lines:",
    [
        x.lineno
        for x in gate_calls
    ]
)


# ======================================================================
# 4. VERIFY BOTH GATE CALLS ARE TOP-LEVEL EXPRESSIONS
# ======================================================================

parent = {}


for node in ast.walk(
    cell16_true_tree
):

    for child in ast.iter_child_nodes(
        node
    ):

        parent[
            child
        ] = node


def enclosing_top_level_r2b(
    node
):

    current = node

    while current in parent:

        p = parent[
            current
        ]

        if p is cell16_true_tree:
            return current

        current = p

    return None


gate_top_nodes = []


for i, call in enumerate(
    gate_calls,
    start=1
):

    top = enclosing_top_level_r2b(
        call
    )

    assert top is not None

    gate_top_nodes.append(
        top
    )

    print(
        f"\nGate #{i}:"
    )

    print(
        " call line:",
        call.lineno
    )

    print(
        " top-level type:",
        type(top).__name__
    )

    print(
        " top-level lines:",
        top.lineno,
        "-",
        getattr(
            top,
            "end_lineno",
            top.lineno
        )
    )

    print(
        " exact call:"
    )

    print(
        ast.get_source_segment(
            CELL16_TRUE_ORIGINAL_SOURCE,
            call
        )
    )


# For the original Cell 16 design these validation calls should be
# standalone top-level expressions.
assert all(
    isinstance(
        x,
        ast.Expr
    )
    for x in gate_top_nodes
), (
    "Validation gate is nested inside a compound "
    "statement. Stop rather than resume ambiguously."
)


# ======================================================================
# 5. FIND SECOND GATE'S TOP-LEVEL BODY INDEX
# ======================================================================

second_gate_top = (
    gate_top_nodes[1]
)


second_gate_body_index = None


for idx, node in enumerate(
    cell16_true_tree.body
):

    if node is second_gate_top:

        second_gate_body_index = idx
        break


assert second_gate_body_index is not None


print(
    "\nSecond validation gate top-level body index:",
    second_gate_body_index
)


# ======================================================================
# 6. BUILD EXACT REMAINDER AFTER SECOND VALIDATION GATE
# ======================================================================

resume_nodes = (
    cell16_true_tree.body[
        second_gate_body_index + 1:
    ]
)


assert resume_nodes, (
    "Nothing remains after the second validation gate."
)


resume_module = ast.Module(
    body=resume_nodes,
    type_ignores=[]
)


ast.fix_missing_locations(
    resume_module
)


source_lines = (
    CELL16_TRUE_ORIGINAL_SOURCE
    .splitlines(
        keepends=True
    )
)


second_gate_end_line = getattr(
    second_gate_top,
    "end_lineno",
    second_gate_top.lineno
)


CELL16_R2B_POST_GATE_SOURCE = "".join(
    source_lines[
        second_gate_end_line:
    ]
)


CELL16_R2B_POST_GATE_SHA256 = (
    hashlib.sha256(
        CELL16_R2B_POST_GATE_SOURCE
        .encode(
            "utf-8"
        )
    ).hexdigest()
)


print(
    "\nPost-gate remainder:"
)

print(
    " statements:",
    len(
        resume_nodes
    )
)

print(
    " characters:",
    len(
        CELL16_R2B_POST_GATE_SOURCE
    )
)

print(
    " SHA256:",
    CELL16_R2B_POST_GATE_SHA256
)


# ======================================================================
# 7. CRITICAL SAFETY CHECKS
# ======================================================================

assert (
    "def profile_rq1_dataset_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)

assert (
    "def make_oracle_functions_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)

assert (
    "def oracle_validation_gate_16"
    not in
    CELL16_R2B_POST_GATE_SOURCE
)


# It should contain references to final TEST processing.
assert (
    "test"
    in
    CELL16_R2B_POST_GATE_SOURCE.lower()
), (
    "Post-gate source does not appear to contain "
    "the final TEST section."
)


print(
    "Post-gate safety checks: PASSED"
)


# ======================================================================
# 8. CONFIRM REPAIRED FUNCTION IS STILL ACTIVE
# ======================================================================

assert (
    globals().get(
        "CELL16_REPAIRED_PROFILE_SHA256"
    )
    ==
    "3413dc5efa00a1be3f9c56f79677bbd97232293b9fb4d15d5a562bacb434287f"
)


print(
    "Repaired profiler identity: PASSED"
)


# ======================================================================
# 9. RESUME ORIGINAL CELL 16
# ======================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "RESUMING ORIGINAL CELL 16 AFTER VALIDATED CWQ GATE"
)

print(
    "=" * 110
)

print(
    "\nThis will now execute the frozen TEST RQ1 section."
)

print(
    "No planner regeneration."
)

print(
    "No AFP retuning."
)


CELL16_R2B_RESUME_STARTED = True


exec(
    compile(
        resume_module,
        filename="<CELL16_R2B_TRUE_POST_GATE_RESUME>",
        mode="exec"
    ),
    globals()
)


CELL16_R2B_RESUME_COMPLETE = True


print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R2B RESUME COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nOriginal Cell 16 fully rerun: NO"
)

print(
    "M3 rerun:                   NO"
)

print(
    "Frozen TEST plans changed:  NO"
)

print(
    "AFP changed:                NO"
)

print(
    "TEST tuning:                NO"
)

print(
    "\nNEXT ACTION:"
)

print(
    "PASTE THE FINAL OUTPUT HERE BEFORE RUNNING CELL 17."
)

CELL 16-R2B — ROBUST SAFE RESUME
Freeze gate:              PASSED
Oracle repair validation: PASSED
Frozen TEST plan SHA gate: PASSED

REAL Cell-16 candidates found: 1
 history= 5 | chars= 51180 | SHA= 2f72f6ab2ac9ec11ac7d97780bd4ae580dc75908c1fca5391da39c9aec6a61e3 | gates= [1677, 1683]

Selected REAL original Cell 16:
 history index: 5
 characters: 51180
 SHA256: 2f72f6ab2ac9ec11ac7d97780bd4ae580dc75908c1fca5391da39c9aec6a61e3
 gate lines: [1677, 1683]

Gate #1:
 call line: 1677
 top-level type: Expr
 top-level lines: 1677 - 1680
 exact call:
oracle_validation_gate_16(
    "webqsp",
    webqsp_val_plan_rows
)

Gate #2:
 call line: 1683
 top-level type: Expr
 top-level lines: 1683 - 1686
 exact call:
oracle_validation_gate_16(
    "cwq",
    cwq_val_plan_rows
)

Second validation gate top-level body index: 41

Post-gate remainder:
 statements: 35
 characters: 16333
 SHA256: ceb29540b9522fd9b88706c8f99564cab7c96e44a4d329dfa1613e5eabbe61c8
Post-gate safety checks: PASSED
Repaired profile

webqsp final TEST RQ1:   0%|          | 0/1628 [00:00<?, ?it/s]

cwq final TEST RQ1:   0%|          | 0/3531 [00:00<?, ?it/s]


WEBQSP FINAL FROZEN-TEST RQ1
questions: 1628
plans: 4880
active hop rows: 4880
active prefixes: 4978
edges examined: 2254484
candidate branches: 0
RoG reachable questions: 0 (0.0000%)
decision hops: 0 (0.0000% of intermediate-active hops)

POST-FREEZE ORACLE
doomed-prefix edges: 2254484 (100.0000%)
oracle-prunable intermediate branches: 0 / 0 (0.0000%)
oracle-avoidable downstream edges: 0 (0.0000%)
doomed final-hop protected branches: 0

CWQ FINAL FROZEN-TEST RQ1
questions: 3531
plans: 10537
active hop rows: 10537
active prefixes: 16344
edges examined: 3461479
candidate branches: 0
RoG reachable questions: 0 (0.0000%)
decision hops: 0 (0.0000% of intermediate-active hops)

POST-FREEZE ORACLE
doomed-prefix edges: 3461479 (100.0000%)
oracle-prunable intermediate branches: 0 / 0 (0.0000%)
oracle-avoidable downstream edges: 0 (0.0000%)
doomed final-hop protected branches: 0

=== CELL 16: FINAL FROZEN-TEST RQ1 PROFILING COMPLETE ===
AFP changed after freeze: NO
TEST used for tuning: NO
POS

In [14]:
# ======================================================================
# CELL 16-R3
# TEST RELATION-MATCHING DIAGNOSTIC
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun Cell 16
#   - rerun M3
#   - restart kernel
#
# No tuning. No TEST plans changed. No artifacts overwritten.
# ======================================================================

from collections import Counter
import json
import re


print("=" * 110)
print("CELL 16-R3 — TEST RELATION-MATCHING DIAGNOSTIC")
print("=" * 110)


# ----------------------------------------------------------------------
# 1. Scientific gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()

print("\nFreeze / oracle repair gates: PASSED")


# ----------------------------------------------------------------------
# 2. Relation normalization probes ONLY
# ----------------------------------------------------------------------

def graph_relation_set(rec):

    return {
        str(triple[1]).strip()
        for triple in rec["graph"]
    }


def raw_plan_tokens(rec):

    plans = rec.get(
        "predicted_paths",
        []
    )

    out = []

    for p in plans:

        if isinstance(p, (list, tuple)):
            out.extend(
                str(x)
                for x in p
            )
        else:
            out.append(
                str(p)
            )

    return out


def token_variants(token):

    s = str(token)

    variants = {
        "raw":
            s,

        "strip":
            s.strip(),

        "strip_quotes":
            s.strip()
             .strip("'\""),

        "remove_path_tags":
            s.replace(
                "<PATH>",
                ""
            ).replace(
                "</PATH>",
                ""
            ).strip(),

        "remove_brackets":
            s.strip()
             .strip("[](){}")
             .strip(),

        "last_space_token":
            s.strip().split()[-1]
            if s.strip()
            else "",
    }

    return variants


# ----------------------------------------------------------------------
# 3. Print several raw TEST examples
# ----------------------------------------------------------------------

def inspect_examples(
    dataset_name,
    rows,
    n=8
):

    print(
        "\n"
        + "=" * 110
    )

    print(
        dataset_name,
        "RAW EXAMPLES"
    )

    print(
        "=" * 110
    )

    shown = 0

    for rec in rows:

        tokens = raw_plan_tokens(
            rec
        )

        if not tokens:
            continue

        graph_rels = graph_relation_set(
            rec
        )

        print(
            "\nQuestion ID:",
            rec["id"]
        )

        print(
            "Question:",
            rec["question"][:180]
        )

        print(
            "predicted_paths:",
            repr(
                rec["predicted_paths"]
            )
        )

        print(
            "\nFirst planner token variants:"
        )

        first = tokens[0]

        for k, v in token_variants(
            first
        ).items():

            print(
                f"  {k:<20}: {repr(v)} "
                f"| exact graph match={v in graph_rels}"
            )

        print(
            "\nSample graph relations:"
        )

        for x in list(
            graph_rels
        )[:12]:
            print(
                " ",
                repr(x)
            )

        shown += 1

        if shown >= n:
            break


inspect_examples(
    "WEBQSP",
    webqsp_test_plan_rows
)

inspect_examples(
    "CWQ",
    cwq_test_plan_rows
)


# ----------------------------------------------------------------------
# 4. Dataset-wide overlap statistics
# ----------------------------------------------------------------------

def overlap_stats(
    dataset_name,
    rows
):

    stats = Counter()

    token_examples = {
        "raw_miss_strip_hit": [],
        "all_simple_miss": [],
    }


    for rec in rows:

        graph_rels = graph_relation_set(
            rec
        )

        tokens = raw_plan_tokens(
            rec
        )

        for token in tokens:

            stats[
                "tokens"
            ] += 1

            variants = token_variants(
                token
            )

            if variants[
                "raw"
            ] in graph_rels:

                stats[
                    "raw_match"
                ] += 1

            if variants[
                "strip"
            ] in graph_rels:

                stats[
                    "strip_match"
                ] += 1

            if variants[
                "strip_quotes"
            ] in graph_rels:

                stats[
                    "strip_quotes_match"
                ] += 1

            if variants[
                "remove_path_tags"
            ] in graph_rels:

                stats[
                    "remove_path_tags_match"
                ] += 1

            if variants[
                "remove_brackets"
            ] in graph_rels:

                stats[
                    "remove_brackets_match"
                ] += 1


            if (
                variants["raw"]
                not in graph_rels
                and
                variants["strip"]
                in graph_rels
                and
                len(
                    token_examples[
                        "raw_miss_strip_hit"
                    ]
                ) < 10
            ):

                token_examples[
                    "raw_miss_strip_hit"
                ].append(
                    (
                        token,
                        variants["strip"]
                    )
                )


            if not any(
                variants[k] in graph_rels

                for k in [
                    "raw",
                    "strip",
                    "strip_quotes",
                    "remove_path_tags",
                    "remove_brackets",
                ]
            ):

                if len(
                    token_examples[
                        "all_simple_miss"
                    ]
                ) < 15:

                    token_examples[
                        "all_simple_miss"
                    ].append(
                        {
                            "id":
                                rec["id"],

                            "token":
                                token,

                            "question":
                                rec["question"][:120],
                        }
                    )


    print(
        "\n"
        + "=" * 110
    )

    print(
        dataset_name,
        "RELATION OVERLAP"
    )

    print(
        "=" * 110
    )

    for key in [
        "tokens",
        "raw_match",
        "strip_match",
        "strip_quotes_match",
        "remove_path_tags_match",
        "remove_brackets_match",
    ]:

        print(
            f"{key:<28}:",
            stats[key]
        )


    denom = max(
        1,
        stats["tokens"]
    )

    print(
        "\nRaw match %:",
        100
        * stats["raw_match"]
        / denom
    )

    print(
        "Strip match %:",
        100
        * stats["strip_match"]
        / denom
    )


    print(
        "\nraw miss / strip hit examples:"
    )

    for x in token_examples[
        "raw_miss_strip_hit"
    ]:
        print(
            repr(x)
        )


    print(
        "\nSimple-normalization misses:"
    )

    for x in token_examples[
        "all_simple_miss"
    ]:
        print(
            x
        )


    return stats


WEB_REL_STATS_R3 = overlap_stats(
    "WEBQSP",
    webqsp_test_plan_rows
)

CWQ_REL_STATS_R3 = overlap_stats(
    "CWQ",
    cwq_test_plan_rows
)


# ----------------------------------------------------------------------
# 5. Compare TEST token shape to frozen VALIDATION token shape
# ----------------------------------------------------------------------

def token_shape_summary(
    rows,
    label
):

    examples = []

    lengths = Counter()

    contains_arrow = 0
    contains_space = 0
    contains_path_tag = 0
    contains_brackets = 0

    total = 0


    for rec in rows:

        for token in raw_plan_tokens(
            rec
        ):

            total += 1

            lengths[
                len(token)
            ] += 1

            if " -> " in token:
                contains_arrow += 1

            if " " in token:
                contains_space += 1

            if (
                "<PATH>" in token
                or
                "</PATH>" in token
            ):
                contains_path_tag += 1

            if any(
                x in token
                for x in "[](){}"
            ):
                contains_brackets += 1

            if len(examples) < 12:
                examples.append(
                    token
                )


    print(
        "\n"
        + "=" * 110
    )

    print(
        label,
        "TOKEN SHAPE"
    )

    print(
        "=" * 110
    )

    print(
        "tokens:",
        total
    )

    print(
        "contains ' -> ':",
        contains_arrow
    )

    print(
        "contains spaces:",
        contains_space
    )

    print(
        "contains PATH tags:",
        contains_path_tag
    )

    print(
        "contains brackets:",
        contains_brackets
    )

    print(
        "\nexamples:"
    )

    for x in examples:
        print(
            " ",
            repr(x)
        )


token_shape_summary(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

token_shape_summary(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

token_shape_summary(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)

token_shape_summary(
    cwq_test_plan_rows,
    "CWQ TEST"
)


# ----------------------------------------------------------------------
# 6. Final status
# ----------------------------------------------------------------------

print(
    "\n"
    + "=" * 110
)

print(
    "CELL 16-R3 DIAGNOSTIC COMPLETE"
)

print(
    "=" * 110
)

print(
    "\nCurrent Cell-16 TEST result is NOT accepted."
)

print(
    "No TEST plan regenerated."
)

print(
    "No tuning performed."
)

print(
    "No AFP parameter changed."
)

print(
    "\nPASTE THIS OUTPUT HERE."
)

CELL 16-R3 — TEST RELATION-MATCHING DIAGNOSTIC

Freeze / oracle repair gates: PASSED

WEBQSP RAW EXAMPLES

Question ID: WebQTest-0
Question: what does jamaican people speak
predicted_paths: [['location. location. l anguages _ sp oken'], ['location. language. count ries _ sp oken _ in'], ['location. language. main _ country']]

First planner token variants:
  raw                 : 'location. location. l anguages _ sp oken' | exact graph match=False
  strip               : 'location. location. l anguages _ sp oken' | exact graph match=False
  strip_quotes        : 'location. location. l anguages _ sp oken' | exact graph match=False
  remove_path_tags    : 'location. location. l anguages _ sp oken' | exact graph match=False
  remove_brackets     : 'location. location. l anguages _ sp oken' | exact graph match=False
  last_space_token    : 'oken' | exact graph match=False

Sample graph relations:
  'symbols.namesake.named_after'
  'olympics.olympic_participating_country.olympics_participat

In [17]:
# ======================================================================
# CELL 16-R4
# TOKENIZER / WHITESPACE-CANONICALIZATION FIDELITY DIAGNOSTIC
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - rewrite TEST JSONL files
#   - restart kernel
#
# This is diagnostic only.
# ======================================================================

import re
import sys
from collections import Counter, defaultdict

import transformers


print("=" * 115)
print("CELL 16-R4 — PLANNER DECODING / RELATION-ID FIDELITY DIAGNOSTIC")
print("=" * 115)


# ======================================================================
# 1. HARD GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert "tokenizer" in globals()

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()

assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()


print("\nFreeze / repair gates: PASSED")


# ======================================================================
# 2. CURRENT TOKENIZER ENVIRONMENT
# ======================================================================

print("\n" + "=" * 115)
print("A. CURRENT TOKENIZER ENVIRONMENT")
print("=" * 115)

print("Python:", sys.version.split()[0])
print("transformers:", transformers.__version__)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer module:", tokenizer.__class__.__module__)
print("is_fast:", getattr(tokenizer, "is_fast", None))
print("legacy:", getattr(tokenizer, "legacy", "ATTRIBUTE_NOT_PRESENT"))
print(
    "clean_up_tokenization_spaces:",
    getattr(
        tokenizer,
        "clean_up_tokenization_spaces",
        "ATTRIBUTE_NOT_PRESENT"
    )
)

print(
    "name_or_path:",
    getattr(
        tokenizer,
        "name_or_path",
        None
    )
)


# ======================================================================
# 3. CANONICAL RELATION-ID RULE
#
# Freebase relation identifiers in the graph are schema identifiers,
# not natural-language strings. We first establish empirically whether
# graph relation IDs contain whitespace.
# ======================================================================

def compact_relation_id_16r4(x):

    return re.sub(
        r"\s+",
        "",
        str(x)
    )


def graph_relations_16r4(rec):

    return {
        str(t[1]).strip()
        for t in rec["graph"]
    }


def plan_tokens_16r4(rec):

    out = []

    for plan in rec.get(
        "predicted_paths",
        []
    ):

        if isinstance(
            plan,
            (list, tuple)
        ):

            out.extend(
                str(x)
                for x in plan
            )

        else:

            out.append(
                str(plan)
            )

    return out


# ======================================================================
# 4. VERIFY GRAPH RELATION VOCABULARY ITSELF
# ======================================================================

def graph_relation_schema_audit(
    rows,
    label
):

    total = 0
    with_whitespace = 0
    examples = []


    for rec in rows:

        for triple in rec["graph"]:

            relation = str(
                triple[1]
            )

            total += 1

            if re.search(
                r"\s",
                relation
            ):

                with_whitespace += 1

                if len(examples) < 10:

                    examples.append(
                        relation
                    )


    print(f"\n{label}")
    print(" graph edge relations:", total)
    print(" relations containing whitespace:", with_whitespace)

    if examples:

        print(" examples:", examples)


    return {
        "total":
            total,

        "with_whitespace":
            with_whitespace,
    }


print("\n" + "=" * 115)
print("B. GRAPH RELATION-ID SCHEMA AUDIT")
print("=" * 115)


WEB_GRAPH_SCHEMA = graph_relation_schema_audit(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

CWQ_GRAPH_SCHEMA = graph_relation_schema_audit(
    cwq_test_plan_rows,
    "CWQ TEST"
)


assert WEB_GRAPH_SCHEMA[
    "with_whitespace"
] == 0

assert CWQ_GRAPH_SCHEMA[
    "with_whitespace"
] == 0


print(
    "\nGraph relation IDs are whitespace-free: PASSED"
)


# ======================================================================
# 5. RAW vs WHITESPACE-COMPACTED MATCHING
# ======================================================================

def compact_match_audit(
    rows,
    label
):

    stats = Counter()

    examples = []

    question_any_raw = 0
    question_any_compact = 0

    first_hop_raw = 0
    first_hop_compact = 0

    n_questions_with_plans = 0


    for rec in rows:

        relations = graph_relations_16r4(
            rec
        )

        tokens = plan_tokens_16r4(
            rec
        )


        any_raw = False
        any_compact = False


        if tokens:

            n_questions_with_plans += 1


        for token in tokens:

            raw = str(token)

            compact = compact_relation_id_16r4(
                raw
            )

            stats[
                "tokens"
            ] += 1


            if raw in relations:

                stats[
                    "raw_exact"
                ] += 1

                any_raw = True


            if compact in relations:

                stats[
                    "compact_exact"
                ] += 1

                any_compact = True


            if (
                raw not in relations
                and
                compact in relations
            ):

                stats[
                    "recovered_by_compaction"
                ] += 1

                if len(examples) < 20:

                    examples.append(
                        (
                            raw,
                            compact
                        )
                    )


        # --------------------------------------------------------------
        # First relation of each predicted plan
        # --------------------------------------------------------------

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):

                continue

            if len(plan) == 0:

                continue


            raw_first = str(
                plan[0]
            )

            compact_first = (
                compact_relation_id_16r4(
                    raw_first
                )
            )


            stats[
                "first_hop_tokens"
            ] += 1


            if raw_first in relations:

                first_hop_raw += 1


            if compact_first in relations:

                first_hop_compact += 1


        if any_raw:
            question_any_raw += 1

        if any_compact:
            question_any_compact += 1


    print("\n" + "-" * 115)
    print(label)
    print("-" * 115)

    for key in [
        "tokens",
        "raw_exact",
        "compact_exact",
        "recovered_by_compaction",
        "first_hop_tokens",
    ]:

        print(
            f"{key:<30}:",
            stats[key]
        )


    denom = max(
        1,
        stats["tokens"]
    )

    first_denom = max(
        1,
        stats["first_hop_tokens"]
    )


    print(
        f"raw exact %:      "
        f"{100 * stats['raw_exact'] / denom:.4f}%"
    )

    print(
        f"compact exact %:  "
        f"{100 * stats['compact_exact'] / denom:.4f}%"
    )

    print(
        f"first-hop raw %:  "
        f"{100 * first_hop_raw / first_denom:.4f}%"
    )

    print(
        f"first-hop compact %: "
        f"{100 * first_hop_compact / first_denom:.4f}%"
    )

    print(
        "questions with any raw graph relation:",
        question_any_raw,
        "/",
        n_questions_with_plans
    )

    print(
        "questions with any compact graph relation:",
        question_any_compact,
        "/",
        n_questions_with_plans
    )


    print(
        "\nRecovered examples:"
    )

    for raw, compact in examples:

        print(
            " RAW:    ",
            repr(raw)
        )

        print(
            " COMPACT:",
            repr(compact)
        )

        print()


    return {
        "stats":
            stats,

        "question_any_raw":
            question_any_raw,

        "question_any_compact":
            question_any_compact,

        "first_hop_raw":
            first_hop_raw,

        "first_hop_compact":
            first_hop_compact,
    }


print("\n" + "=" * 115)
print("C. TEST RAW-vs-COMPACT MATCH AUDIT")
print("=" * 115)


WEB_COMPACT_TEST = compact_match_audit(
    webqsp_test_plan_rows,
    "WEBQSP TEST"
)

CWQ_COMPACT_TEST = compact_match_audit(
    cwq_test_plan_rows,
    "CWQ TEST"
)


# ======================================================================
# 6. VALIDATION INVARIANCE CHECK
#
# A legitimate representation repair should essentially be a no-op on
# the already-correct frozen validation planner outputs.
# ======================================================================

def validation_compaction_audit(
    rows,
    label
):

    tokens = 0
    changed = 0

    changed_examples = []


    for rec in rows:

        for token in plan_tokens_16r4(
            rec
        ):

            tokens += 1

            compact = compact_relation_id_16r4(
                token
            )

            if compact != token:

                changed += 1

                if len(
                    changed_examples
                ) < 20:

                    changed_examples.append(
                        (
                            token,
                            compact
                        )
                    )


    print(f"\n{label}")
    print(" tokens:", tokens)
    print(" tokens altered by whitespace compaction:", changed)

    if tokens:

        print(
            " changed %:",
            f"{100 * changed / tokens:.6f}%"
        )


    if changed_examples:

        print(
            "\n changed examples:"
        )

        for x in changed_examples:

            print(
                " ",
                repr(x[0]),
                "->",
                repr(x[1])
            )


    return {
        "tokens":
            tokens,

        "changed":
            changed,

        "examples":
            changed_examples,
    }


print("\n" + "=" * 115)
print("D. FROZEN VALIDATION INVARIANCE")
print("=" * 115)


WEB_VAL_COMPACT = (
    validation_compaction_audit(
        webqsp_val_plan_rows,
        "WEBQSP VALIDATION"
    )
)

CWQ_VAL_COMPACT = (
    validation_compaction_audit(
        cwq_val_plan_rows,
        "CWQ VALIDATION"
    )
)


# ======================================================================
# 7. TOKENIZER ROUND-TRIP PROBES
# ======================================================================

print("\n" + "=" * 115)
print("E. CURRENT TOKENIZER ROUND-TRIP PROBES")
print("=" * 115)


probe_relations = [
    "people.person.place_of_birth",
    "people.person.profession",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "sports.sports_team.team_mascot",
    "education.educational_institution.sports_teams",
    "olympics.olympic_participating_country.olympics_participated_in",
]


TOKENIZER_PROBE_16R4 = []


for relation in probe_relations:

    ids = tokenizer.encode(
        relation,
        add_special_tokens=False
    )

    tokens = tokenizer.convert_ids_to_tokens(
        ids
    )

    decoded = tokenizer.decode(
        ids,
        skip_special_tokens=True
    )

    rec = {
        "relation":
            relation,

        "ids":
            ids,

        "tokens":
            tokens,

        "decoded":
            decoded,

        "exact_roundtrip":
            decoded == relation,

        "compact_roundtrip":
            compact_relation_id_16r4(
                decoded
            )
            ==
            relation,
    }

    TOKENIZER_PROBE_16R4.append(
        rec
    )


    print("\nRELATION:")
    print(" ", relation)

    print("TOKENS:")
    print(" ", tokens)

    print("DECODED:")
    print(" ", repr(decoded))

    print(
        "exact round-trip:",
        decoded == relation
    )

    print(
        "compact round-trip:",
        compact_relation_id_16r4(
            decoded
        )
        ==
        relation
    )


# ======================================================================
# 8. CHECK WHETHER TEST CORRUPTION IS CHARACTER-DESTRUCTIVE
#
# If removing whitespace produces syntactically normal relation IDs,
# then characters themselves survived and only token-boundary spacing
# was injected.
# ======================================================================

relation_pattern = re.compile(
    r"^[A-Za-z0-9_#.\-]+$"
)


def compact_syntax_audit(
    rows,
    label
):

    total = 0
    valid_compact_syntax = 0
    invalid_examples = []


    for rec in rows:

        for token in plan_tokens_16r4(
            rec
        ):

            total += 1

            compact = (
                compact_relation_id_16r4(
                    token
                )
            )


            if relation_pattern.fullmatch(
                compact
            ):

                valid_compact_syntax += 1

            elif len(
                invalid_examples
            ) < 20:

                invalid_examples.append(
                    (
                        token,
                        compact
                    )
                )


    print(f"\n{label}")
    print(" tokens:", total)
    print(
        "valid relation-ID syntax after compaction:",
        valid_compact_syntax
    )

    print(
        "percentage:",
        (
            100
            * valid_compact_syntax
            / max(1, total)
        )
    )


    if invalid_examples:

        print(
            "\nInvalid compacted examples:"
        )

        for x in invalid_examples:
            print(x)


    return {
        "total":
            total,

        "valid":
            valid_compact_syntax,

        "invalid_examples":
            invalid_examples,
    }


print("\n" + "=" * 115)
print("F. CHARACTER-PRESERVATION / SYNTAX AUDIT")
print("=" * 115)


WEB_SYNTAX_16R4 = (
    compact_syntax_audit(
        webqsp_test_plan_rows,
        "WEBQSP TEST"
    )
)

CWQ_SYNTAX_16R4 = (
    compact_syntax_audit(
        cwq_test_plan_rows,
        "CWQ TEST"
    )
)


# ======================================================================
# 9. STATUS
# ======================================================================

CELL16_R4_COMPLETE = True


print(
    "\n"
    + "=" * 115
)

print(
    "CELL 16-R4 DIAGNOSTIC COMPLETE"
)

print(
    "=" * 115
)

print(
    "\nTEST JSONLs modified: NO"
)

print(
    "TEST plans regenerated: NO"
)

print(
    "AFP changed: NO"
)

print(
    "TEST tuning: NO"
)

print(
    "\nPASTE THE OUTPUT HERE BEFORE ANY REPAIR."
)

CELL 16-R4 — PLANNER DECODING / RELATION-ID FIDELITY DIAGNOSTIC

Freeze / repair gates: PASSED

A. CURRENT TOKENIZER ENVIRONMENT
Python: 3.12.13
transformers: 5.0.0
Tokenizer class: TokenizersBackend
Tokenizer module: transformers.tokenization_utils_tokenizers
is_fast: True
legacy: ATTRIBUTE_NOT_PRESENT
clean_up_tokenization_spaces: True
name_or_path: rmanluo/RoG

B. GRAPH RELATION-ID SCHEMA AUDIT

WEBQSP TEST
 graph edge relations: 7015502
 relations containing whitespace: 0

CWQ TEST
 graph edge relations: 15086427
 relations containing whitespace: 0

Graph relation IDs are whitespace-free: PASSED

C. TEST RAW-vs-COMPACT MATCH AUDIT

-------------------------------------------------------------------------------------------------------------------
WEBQSP TEST
-------------------------------------------------------------------------------------------------------------------
tokens                        : 7817
raw_exact                     : 0
compact_exact                 : 4160
reco

In [18]:
# ======================================================================
# CELL 16-R5
# ORIGINAL SLOW-TOKENIZER COMPATIBILITY / INPUT-ID FIDELITY AUDIT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - rewrite TEST JSONLs
#   - restart kernel
#
# This cell performs NO TEST evaluation and NO tuning.
# ======================================================================

from pathlib import Path
import hashlib
import re
import sys

import sentencepiece as spm


print("=" * 118)
print("CELL 16-R5 — ORIGINAL SLOW-TOKENIZER COMPATIBILITY AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "tokenizer" in globals()
assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("\nFreeze gate: PASSED")


# ======================================================================
# 2. VERIFY OFFICIAL RoG REQUIREMENT FILE
# ======================================================================

req_candidates = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "requirements.txt"
    ),
    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/requirements.txt"
    ),
]

REQ_PATH = next(
    (p for p in req_candidates if p.exists()),
    None
)

assert REQ_PATH is not None, (
    "Official local RoG requirements.txt not found."
)

requirements_text = REQ_PATH.read_text(
    encoding="utf-8"
)

assert "transformers==4.32.0" in requirements_text
assert "sentencepiece==0.1.99" in requirements_text

print("\nOfficial RoG dependency specification: PASSED")
print(" requirements:", REQ_PATH)
print(" expected transformers: 4.32.0")
print(" expected sentencepiece: 0.1.99")

print("\nCurrent environment:")
print(
    " transformers:",
    __import__("transformers").__version__
)
print(
    " sentencepiece:",
    spm.__version__
)
print(
    " tokenizer class:",
    tokenizer.__class__.__name__
)
print(
    " is_fast:",
    getattr(tokenizer, "is_fast", None)
)


# ======================================================================
# 3. FIND EXACT rmanluo/RoG tokenizer.model
# ======================================================================

tokenizer_model_candidates = []

# HF cache
hf_cache = Path.home() / ".cache/huggingface/hub"

if hf_cache.exists():

    tokenizer_model_candidates.extend(
        hf_cache.glob(
            "models--rmanluo--RoG/"
            "snapshots/*/tokenizer.model"
        )
    )

# Kaggle caches sometimes live elsewhere.
for root in [
    Path("/root/.cache/huggingface"),
    Path("/kaggle/working"),
]:

    if root.exists():

        tokenizer_model_candidates.extend(
            root.rglob("tokenizer.model")
        )


# Keep plausible RoG files, then verify vocab size later.
tokenizer_model_candidates = list(
    dict.fromkeys(
        tokenizer_model_candidates
    )
)


print(
    "\nFound tokenizer.model candidates:",
    len(tokenizer_model_candidates)
)


for p in tokenizer_model_candidates[:20]:
    print(" ", p)


assert tokenizer_model_candidates, (
    "Could not locate tokenizer.model downloaded for rmanluo/RoG."
)


# Prefer explicit rmanluo RoG cache path.
preferred = [
    p
    for p in tokenizer_model_candidates
    if (
        "models--rmanluo--RoG"
        in str(p)
    )
]


TOKENIZER_MODEL_PATH = (
    preferred[0]
    if preferred
    else tokenizer_model_candidates[0]
)


print(
    "\nSelected SentencePiece model:",
    TOKENIZER_MODEL_PATH
)


# ======================================================================
# 4. LOAD RAW SENTENCEPIECE MODEL
# ======================================================================

sp = spm.SentencePieceProcessor()

loaded = sp.load(
    str(TOKENIZER_MODEL_PATH)
)

assert loaded


print("\nSentencePiece model loaded: PASSED")
print(" vocab size:", sp.get_piece_size())
print(" bos id:", sp.bos_id())
print(" eos id:", sp.eos_id())
print(" unk id:", sp.unk_id())


print("\nTransformers tokenizer:")
print(
    " vocab size:",
    getattr(tokenizer, "vocab_size", None)
)
print(
    " bos_token_id:",
    getattr(tokenizer, "bos_token_id", None)
)
print(
    " eos_token_id:",
    getattr(tokenizer, "eos_token_id", None)
)


# ======================================================================
# 5. ID-ENCODING COMPARISON
#
# The critical question:
# Does Transformers 5 produce the same SentencePiece IDs?
# ======================================================================

def current_ids_no_special(text):

    return list(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )


def slow_sp_ids_no_special(text):

    return list(
        sp.encode(
            str(text),
            out_type=int
        )
    )


def compare_id_encoding(text):

    current = current_ids_no_special(
        text
    )

    slow = slow_sp_ids_no_special(
        text
    )

    return {
        "same":
            current == slow,

        "current":
            current,

        "slow":
            slow,
    }


probe_texts = [
    "people.person.place_of_birth",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "what does jamaican people speak",
    "what did james k polk do before he was president",
    (
        "Please generate a valid relation path that can be "
        "helpful for answering the following question: "
        "what does jamaican people speak"
    ),
]


print("\n" + "=" * 118)
print("A. DIRECT TOKEN-ID PROBES")
print("=" * 118)


for text in probe_texts:

    cmp = compare_id_encoding(
        text
    )

    print("\nTEXT:")
    print(repr(text))

    print(
        "same token IDs:",
        cmp["same"]
    )

    if not cmp["same"]:

        print(
            " current:",
            cmp["current"][:80]
        )
        print(
            " slow SP:",
            cmp["slow"][:80]
        )


# ======================================================================
# 6. LARGE-SAMPLE QUESTION ENCODING AUDIT
# ======================================================================

def audit_question_ids(
    rows,
    label,
    limit=500
):

    checked = 0
    identical = 0
    mismatched = 0
    examples = []


    for rec in rows[:limit]:

        text = str(
            rec["question"]
        )

        cmp = compare_id_encoding(
            text
        )

        checked += 1

        if cmp["same"]:

            identical += 1

        else:

            mismatched += 1

            if len(examples) < 10:

                examples.append(
                    {
                        "id":
                            rec["id"],

                        "question":
                            text,

                        "current":
                            cmp["current"],

                        "slow":
                            cmp["slow"],
                    }
                )


    print(f"\n{label}")
    print(" checked:", checked)
    print(" identical IDs:", identical)
    print(" mismatched IDs:", mismatched)

    print(
        " identity rate:",
        f"{100 * identical / max(1, checked):.6f}%"
    )

    if examples:

        print("\nMismatch examples:")

        for x in examples:
            print(" ", x["id"])
            print("   ", x["question"][:150])
            print("   current:", x["current"][:40])
            print("   slow:   ", x["slow"][:40])


    return {
        "checked":
            checked,

        "identical":
            identical,

        "mismatched":
            mismatched,
    }


print("\n" + "=" * 118)
print("B. QUESTION INPUT-ID FIDELITY")
print("=" * 118)


WEB_Q_ID_AUDIT = audit_question_ids(
    webqsp_test_plan_rows,
    "WEBQSP TEST questions",
    limit=500
)

CWQ_Q_ID_AUDIT = audit_question_ids(
    cwq_test_plan_rows,
    "CWQ TEST questions",
    limit=500
)


# ======================================================================
# 7. FULL PLANNER-PROMPT INPUT-ID AUDIT
#
# Use the exact recovered prompter if it survived M3.
# ======================================================================

print("\n" + "=" * 118)
print("C. FULL PLANNER-PROMPT INPUT-ID FIDELITY")
print("=" * 118)


PROMPT_AUDIT_AVAILABLE = (
    "prompter" in globals()
    and
    "INSTRUCTION" in globals()
)


print(
    "Exact prompter available:",
    PROMPT_AUDIT_AVAILABLE
)


def planner_prompt_for(question):

    return prompter.format(
        instruction=INSTRUCTION,
        message=question
    )


def audit_full_prompts(
    rows,
    label,
    limit=200
):

    checked = 0
    identical = 0
    mismatched = 0

    examples = []


    for rec in rows[:limit]:

        prompt = planner_prompt_for(
            rec["question"]
        )

        cmp = compare_id_encoding(
            prompt
        )

        checked += 1

        if cmp["same"]:

            identical += 1

        else:

            mismatched += 1

            if len(examples) < 5:

                examples.append(
                    (
                        rec["id"],
                        cmp["current"][:60],
                        cmp["slow"][:60],
                    )
                )


    print(f"\n{label}")
    print(" prompts checked:", checked)
    print(" identical IDs:", identical)
    print(" mismatched IDs:", mismatched)

    print(
        " identity rate:",
        f"{100 * identical / max(1, checked):.6f}%"
    )


    if examples:

        print("\nMismatch examples:")

        for x in examples:
            print(" ", x)


    return {
        "checked":
            checked,

        "identical":
            identical,

        "mismatched":
            mismatched,
    }


if PROMPT_AUDIT_AVAILABLE:

    WEB_PROMPT_ID_AUDIT = audit_full_prompts(
        webqsp_test_plan_rows,
        "WEBQSP full planner prompts",
        limit=200
    )

    CWQ_PROMPT_ID_AUDIT = audit_full_prompts(
        cwq_test_plan_rows,
        "CWQ full planner prompts",
        limit=200
    )

else:

    WEB_PROMPT_ID_AUDIT = None
    CWQ_PROMPT_ID_AUDIT = None

    print(
        "\nFull-prompt audit skipped because "
        "prompter/INSTRUCTION is not currently in RAM."
    )


# ======================================================================
# 8. DECODING COMPARISON
# ======================================================================

def compact_ws(x):

    return re.sub(
        r"\s+",
        "",
        str(x)
    )


print("\n" + "=" * 118)
print("D. FAST-v5 vs RAW SENTENCEPIECE DECODING")
print("=" * 118)


relations = [
    "people.person.place_of_birth",
    "people.person.profession",
    "government.government_position_held.office_holder",
    "location.location.containedby",
    "sports.sports_team.team_mascot",
    "education.educational_institution.sports_teams",
]


for relation in relations:

    ids = slow_sp_ids_no_special(
        relation
    )

    sp_decoded = sp.decode(
        ids
    )

    fast_decoded = tokenizer.decode(
        ids,
        skip_special_tokens=True
    )


    print("\nRELATION:")
    print(" ", relation)

    print("SentencePiece decode:")
    print(" ", repr(sp_decoded))

    print("Transformers-5 decode:")
    print(" ", repr(fast_decoded))

    print(
        "SP exact:",
        sp_decoded == relation
    )

    print(
        "v5 compact exact:",
        compact_ws(
            fast_decoded
        )
        ==
        relation
    )


# ======================================================================
# 9. DATASET-WIDE VALIDATION RELATION ROUND-TRIP
#
# Known-good frozen validation relation strings are used as the
# representation-fidelity reference, not TEST gold labels.
# ======================================================================

def relation_roundtrip_audit(
    rows,
    label
):

    total = 0
    whitespace_free = 0

    sp_exact = 0
    v5_exact = 0
    v5_compact_exact = 0

    failures = []


    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue


            for relation in plan:

                relation = str(
                    relation
                )

                total += 1


                # Primary reference is already-canonical
                # validation relation strings.
                if re.search(
                    r"\s",
                    relation
                ):
                    continue


                whitespace_free += 1

                ids = slow_sp_ids_no_special(
                    relation
                )

                sp_decoded = sp.decode(
                    ids
                )

                v5_decoded = tokenizer.decode(
                    ids,
                    skip_special_tokens=True
                )


                if sp_decoded == relation:
                    sp_exact += 1

                if v5_decoded == relation:
                    v5_exact += 1

                if (
                    compact_ws(
                        v5_decoded
                    )
                    ==
                    relation
                ):
                    v5_compact_exact += 1

                elif len(failures) < 15:

                    failures.append(
                        (
                            relation,
                            v5_decoded,
                            compact_ws(
                                v5_decoded
                            ),
                        )
                    )


    print(f"\n{label}")
    print(" all tokens:", total)
    print(
        " canonical whitespace-free tokens:",
        whitespace_free
    )

    print(
        " raw SentencePiece exact:",
        sp_exact
    )

    print(
        " Transformers-5 exact:",
        v5_exact
    )

    print(
        " Transformers-5 compact exact:",
        v5_compact_exact
    )

    print(
        " compact recovery rate:",
        f"{100 * v5_compact_exact / max(1, whitespace_free):.6f}%"
    )


    if failures:

        print("\nCompaction failures:")

        for item in failures:
            print(item)


    return {
        "total":
            total,

        "canonical":
            whitespace_free,

        "sp_exact":
            sp_exact,

        "v5_exact":
            v5_exact,

        "v5_compact_exact":
            v5_compact_exact,
    }


print("\n" + "=" * 118)
print("E. FROZEN-VALIDATION ROUND-TRIP FIDELITY")
print("=" * 118)


WEB_VAL_ROUNDTRIP = relation_roundtrip_audit(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

CWQ_VAL_ROUNDTRIP = relation_roundtrip_audit(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)


# ======================================================================
# 10. VALIDATION GRAPH-OVERLAP REFERENCE
#
# Compare TEST-compacted overlap later against the natural raw overlap
# level of frozen validation plans.
# ======================================================================

def graph_relations_r5(rec):

    return {
        str(t[1]).strip()
        for t in rec["graph"]
    }


def overlap_reference(
    rows,
    label
):

    tokens = 0
    exact = 0

    first_tokens = 0
    first_exact = 0

    q_with_plan = 0
    q_any = 0


    for rec in rows:

        graph_rels = graph_relations_r5(
            rec
        )

        any_match = False
        had_plan = False


        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue

            if len(plan) == 0:
                continue

            had_plan = True

            first_tokens += 1

            if str(plan[0]) in graph_rels:
                first_exact += 1


            for relation in plan:

                tokens += 1

                if str(relation) in graph_rels:

                    exact += 1
                    any_match = True


        if had_plan:
            q_with_plan += 1

        if any_match:
            q_any += 1


    print(f"\n{label}")
    print(" relation tokens:", tokens)
    print(" exact graph relations:", exact)

    print(
        " exact token %:",
        f"{100 * exact / max(1, tokens):.4f}%"
    )

    print(
        " first-hop exact %:",
        f"{100 * first_exact / max(1, first_tokens):.4f}%"
    )

    print(
        " questions with any exact relation:",
        q_any,
        "/",
        q_with_plan
    )


    return {
        "tokens":
            tokens,

        "exact":
            exact,

        "first_tokens":
            first_tokens,

        "first_exact":
            first_exact,

        "q_with_plan":
            q_with_plan,

        "q_any":
            q_any,
    }


print("\n" + "=" * 118)
print("F. FROZEN VALIDATION GRAPH-OVERLAP REFERENCE")
print("=" * 118)


WEB_VAL_OVERLAP_R5 = overlap_reference(
    webqsp_val_plan_rows,
    "WEBQSP VALIDATION"
)

CWQ_VAL_OVERLAP_R5 = overlap_reference(
    cwq_val_plan_rows,
    "CWQ VALIDATION"
)


# ======================================================================
# 11. FINAL STATUS — DO NOT AUTO-REPAIR
# ======================================================================

CELL16_R5_COMPLETE = True


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R5 AUDIT COMPLETE"
)

print(
    "=" * 118
)

print("\nNo TEST JSONL modified.")
print("No TEST plan regenerated.")
print("No AFP parameter changed.")
print("No TEST tuning performed.")

print(
    "\nPASTE THIS OUTPUT HERE."
)

print(
    "Do NOT run Cell 17 yet."
)

CELL 16-R5 — ORIGINAL SLOW-TOKENIZER COMPATIBILITY AUDIT

Freeze gate: PASSED

Official RoG dependency specification: PASSED
 requirements: /kaggle/working/reasoning-on-graphs/requirements.txt
 expected transformers: 4.32.0
 expected sentencepiece: 0.1.99

Current environment:
 transformers: 5.0.0
 sentencepiece: 0.2.1
 tokenizer class: TokenizersBackend
 is_fast: True

Found tokenizer.model candidates: 1
  /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model

Selected SentencePiece model: /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model

SentencePiece model loaded: PASSED
 vocab size: 32000
 bos id: 1
 eos id: 2
 unk id: 0

Transformers tokenizer:
 vocab size: 32000
 bos_token_id: 1
 eos_token_id: 2

A. DIRECT TOKEN-ID PROBES

TEXT:
'people.person.place_of_birth'
same token IDs: False
 current: [412, 459, 280, 29889, 546, 1100, 29889, 572, 815, 29918, 974

In [19]:
# ======================================================================
# CELL 16-R6
# ORIGINAL ENVIRONMENT + TRUE SLOW LLAMA TOKENIZER RECOVERY AUDIT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - modify TEST JSONLs
#   - restart kernel
#
# Diagnostic only.
# ======================================================================

import ast
import re
from pathlib import Path

import sentencepiece as spm
import transformers


print("=" * 118)
print("CELL 16-R6 — ORIGINAL ENVIRONMENT / SLOW TOKENIZER AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()
assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()

print("\nFreeze gate: PASSED")


# ======================================================================
# 2. SEARCH ORIGINAL NOTEBOOK FOR DEPENDENCY-INSTALL CELLS
# ======================================================================

print("\n" + "=" * 118)
print("A. ORIGINAL NOTEBOOK DEPENDENCY SETUP")
print("=" * 118)


dependency_terms = [
    "pip install",
    "pip3 install",
    "requirements.txt",
    "transformers==",
    "sentencepiece",
    "graph-walker",
]


dependency_cells = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    lowered = source.lower()

    if any(
        term.lower() in lowered
        for term in dependency_terms
    ):

        dependency_cells.append(
            (
                cell_idx,
                source
            )
        )


print(
    "Relevant dependency/setup cells:",
    [x[0] for x in dependency_cells]
)


for cell_idx, source in dependency_cells:

    print(
        "\n"
        + "-" * 118
    )

    print(
        f"NOTEBOOK CELL {cell_idx}"
    )

    print(
        "-" * 118
    )

    for line in source.splitlines():

        if any(
            term.lower() in line.lower()
            for term in dependency_terms
        ):

            print(line)


# ======================================================================
# 3. OFFICIAL LOCAL REQUIREMENTS
# ======================================================================

req_candidates = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "requirements.txt"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/requirements.txt"
    ),
]


REQ_PATH = next(
    (
        p
        for p in req_candidates
        if p.exists()
    ),
    None
)


assert REQ_PATH is not None


requirements_text = (
    REQ_PATH.read_text(
        encoding="utf-8"
    )
)


print("\n" + "=" * 118)
print("B. OFFICIAL RoG REQUIREMENTS")
print("=" * 118)

for line in requirements_text.splitlines():

    if (
        "transformers" in line.lower()
        or
        "sentencepiece" in line.lower()
        or
        "tokenizers" in line.lower()
    ):

        print(line)


assert "transformers==4.32.0" in requirements_text
assert "sentencepiece==0.1.99" in requirements_text


print("\nCurrent environment:")
print(" transformers:", transformers.__version__)
print(" sentencepiece:", spm.__version__)


# ======================================================================
# 4. FIND RAW RoG SENTENCEPIECE MODEL
# ======================================================================

cache_root = Path(
    "/root/.cache/huggingface/hub"
)

sp_candidates = list(
    cache_root.glob(
        "models--rmanluo--RoG/"
        "snapshots/*/tokenizer.model"
    )
)


assert sp_candidates, (
    "RoG tokenizer.model not found."
)


SP_MODEL_PATH = sp_candidates[0]


sp = spm.SentencePieceProcessor()

assert sp.load(
    str(SP_MODEL_PATH)
)


print("\nSentencePiece model:")
print(" ", SP_MODEL_PATH)
print(" vocab:", sp.get_piece_size())


# ======================================================================
# 5. TRY TRUE SLOW LLAMA TOKENIZER DIRECTLY
#
# AutoTokenizer is the component that returned TokenizersBackend.
# Here we explicitly request the SentencePiece LlamaTokenizer.
# ======================================================================

print("\n" + "=" * 118)
print("C. DIRECT SLOW LLAMA TOKENIZER")
print("=" * 118)


SLOW_TOKENIZER_AVAILABLE = False
slow_tokenizer = None
slow_tokenizer_error = None


try:

    from transformers.models.llama.tokenization_llama import (
        LlamaTokenizer
    )

    try:

        slow_tokenizer = (
            LlamaTokenizer.from_pretrained(
                "rmanluo/RoG",
                legacy=True,
                clean_up_tokenization_spaces=True,
            )
        )

    except TypeError:

        # Some versions do not expose legacy as a constructor kwarg.
        slow_tokenizer = (
            LlamaTokenizer.from_pretrained(
                "rmanluo/RoG",
                clean_up_tokenization_spaces=True,
            )
        )


    SLOW_TOKENIZER_AVAILABLE = True


except Exception as e:

    slow_tokenizer_error = repr(e)


print(
    "Slow tokenizer available:",
    SLOW_TOKENIZER_AVAILABLE
)


if SLOW_TOKENIZER_AVAILABLE:

    print(
        " class:",
        slow_tokenizer.__class__.__name__
    )

    print(
        " module:",
        slow_tokenizer.__class__.__module__
    )

    print(
        " is_fast:",
        getattr(
            slow_tokenizer,
            "is_fast",
            False
        )
    )

    print(
        " vocab size:",
        slow_tokenizer.vocab_size
    )

else:

    print(
        " error:",
        slow_tokenizer_error
    )


# ======================================================================
# 6. RAW SP vs DIRECT SLOW-TOKENIZER ID FIDELITY
# ======================================================================

def raw_sp_ids(text):

    return list(
        sp.encode(
            str(text),
            out_type=int
        )
    )


def slow_ids_no_special(text):

    return list(
        slow_tokenizer.encode(
            str(text),
            add_special_tokens=False
        )
    )


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("D. DIRECT TOKEN-ID FIDELITY")
    print("=" * 118)


    probes = [
        "people.person.place_of_birth",
        "government.government_position_held.office_holder",
        "location.location.containedby",
        "what does jamaican people speak",
        "what did james k polk do before he was president",
    ]


    for text in probes:

        a = raw_sp_ids(
            text
        )

        b = slow_ids_no_special(
            text
        )

        print(
            "\n",
            repr(text)
        )

        print(
            " IDs identical:",
            a == b
        )

        if a != b:

            print(
                " raw SP:",
                a
            )

            print(
                " slow:  ",
                b
            )


# ======================================================================
# 7. DATASET-WIDE QUESTION ID FIDELITY
# ======================================================================

def question_id_fidelity(
    rows,
    label,
    limit=500
):

    checked = 0
    exact = 0

    mismatches = []


    for rec in rows[:limit]:

        question = str(
            rec["question"]
        )

        a = raw_sp_ids(
            question
        )

        b = slow_ids_no_special(
            question
        )

        checked += 1


        if a == b:

            exact += 1

        elif len(mismatches) < 5:

            mismatches.append(
                rec["id"]
            )


    print(f"\n{label}")
    print(" checked:", checked)
    print(" exact IDs:", exact)

    print(
        " identity rate:",
        f"{100 * exact / max(1, checked):.6f}%"
    )

    print(
        " mismatch examples:",
        mismatches
    )


    return {
        "checked": checked,
        "exact": exact,
    }


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("E. QUESTION INPUT-ID FIDELITY")
    print("=" * 118)


    WEB_SLOW_ID_AUDIT = (
        question_id_fidelity(
            webqsp_test_plan_rows,
            "WEBQSP TEST",
            500
        )
    )


    CWQ_SLOW_ID_AUDIT = (
        question_id_fidelity(
            cwq_test_plan_rows,
            "CWQ TEST",
            500
        )
    )


# ======================================================================
# 8. VALIDATION RELATION ROUND-TRIP
# ======================================================================

def validation_roundtrip(
    rows,
    label
):

    checked = 0
    exact = 0

    failures = []


    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):

                continue


            for relation in plan:

                relation = str(
                    relation
                )

                # Keep this audit on already-canonical relation IDs.
                if re.search(
                    r"\s",
                    relation
                ):

                    continue


                ids = slow_ids_no_special(
                    relation
                )

                decoded = (
                    slow_tokenizer.decode(
                        ids,
                        skip_special_tokens=True
                    )
                )


                checked += 1


                if decoded == relation:

                    exact += 1

                elif len(failures) < 10:

                    failures.append(
                        (
                            relation,
                            decoded
                        )
                    )


    print(f"\n{label}")
    print(
        " canonical relations checked:",
        checked
    )

    print(
        " exact round-trips:",
        exact
    )

    print(
        " exact rate:",
        f"{100 * exact / max(1, checked):.6f}%"
    )


    if failures:

        print(
            " failures:",
            failures
        )


    return {
        "checked": checked,
        "exact": exact,
    }


if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("F. FROZEN VALIDATION RELATION ROUND-TRIP")
    print("=" * 118)


    WEB_SLOW_ROUNDTRIP = (
        validation_roundtrip(
            webqsp_val_plan_rows,
            "WEBQSP VALIDATION"
        )
    )


    CWQ_SLOW_ROUNDTRIP = (
        validation_roundtrip(
            cwq_val_plan_rows,
            "CWQ VALIDATION"
        )
    )


# ======================================================================
# 9. SPECIAL-TOKEN BEHAVIOR USED BY generate_seq()
#
# generate_seq uses tokenizer.encode(..., return_tensors="pt")
# without add_special_tokens=False.
# ======================================================================

if SLOW_TOKENIZER_AVAILABLE:

    print("\n" + "=" * 118)
    print("G. generate_seq ENCODE BEHAVIOR")
    print("=" * 118)


    probe = (
        "Please generate a valid relation path that can be "
        "helpful for answering the following question: "
        "what does jamaican people speak"
    )


    no_special = slow_ids_no_special(
        probe
    )

    with_special = list(
        slow_tokenizer.encode(
            probe
        )
    )


    print(
        "slow add_bos_token:",
        getattr(
            slow_tokenizer,
            "add_bos_token",
            None
        )
    )

    print(
        "slow add_eos_token:",
        getattr(
            slow_tokenizer,
            "add_eos_token",
            None
        )
    )

    print(
        "BOS id:",
        slow_tokenizer.bos_token_id
    )

    print(
        "EOS id:",
        slow_tokenizer.eos_token_id
    )

    print(
        "no-special length:",
        len(no_special)
    )

    print(
        "default encode length:",
        len(with_special)
    )

    print(
        "default encode prefix:",
        with_special[:10]
    )

    print(
        "expected raw-SP prefix:",
        (
            [slow_tokenizer.bos_token_id]
            + raw_sp_ids(probe)
        )[:10]
    )

    print(
        "default == BOS + raw SP:",
        with_special
        ==
        (
            [slow_tokenizer.bos_token_id]
            + raw_sp_ids(probe)
        )
    )


# ======================================================================
# 10. FINAL DECISION GATE
# ======================================================================

if SLOW_TOKENIZER_AVAILABLE:

    web_ids_ok = (
        WEB_SLOW_ID_AUDIT["exact"]
        ==
        WEB_SLOW_ID_AUDIT["checked"]
    )

    cwq_ids_ok = (
        CWQ_SLOW_ID_AUDIT["exact"]
        ==
        CWQ_SLOW_ID_AUDIT["checked"]
    )

    web_decode_ok = (
        WEB_SLOW_ROUNDTRIP["exact"]
        ==
        WEB_SLOW_ROUNDTRIP["checked"]
    )

    cwq_decode_ok = (
        CWQ_SLOW_ROUNDTRIP["exact"]
        ==
        CWQ_SLOW_ROUNDTRIP["checked"]
    )

    SLOW_TOKENIZER_FIDELITY_PASSED = (
        web_ids_ok
        and
        cwq_ids_ok
        and
        web_decode_ok
        and
        cwq_decode_ok
    )

else:

    SLOW_TOKENIZER_FIDELITY_PASSED = False


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R6 AUDIT COMPLETE"
)

print(
    "=" * 118
)

print(
    "\nSlow tokenizer fidelity passed:",
    SLOW_TOKENIZER_FIDELITY_PASSED
)

print("\nTEST JSONLs modified: NO")
print("TEST plans regenerated: NO")
print("AFP changed: NO")
print("TEST tuning: NO")

print(
    "\nPASTE THIS OUTPUT HERE."
)

print(
    "DO NOT RUN CELL 17."
)

CELL 16-R6 — ORIGINAL ENVIRONMENT / SLOW TOKENIZER AUDIT

Freeze gate: PASSED

A. ORIGINAL NOTEBOOK DEPENDENCY SETUP
Relevant dependency/setup cells: [1]

----------------------------------------------------------------------------------------------------------------------
NOTEBOOK CELL 1
----------------------------------------------------------------------------------------------------------------------
!pip install -q \
    --only-binary=transformers,tokenizers,peft,sentencepiece,accelerate,datasets,numpy,torch \
    "transformers==4.44.2" "tokenizers==0.19.1" "peft==0.12.0" \
    "sentencepiece==0.2.0" "accelerate==0.34.2" "datasets==2.20.0" \
    "numpy>=2.0,<2.1" "graph-walker==1.0.6" "{_torch_pin}"

B. OFFICIAL RoG REQUIREMENTS
transformers==4.32.0
sentencepiece==0.1.99

Current environment:
 transformers: 5.0.0
 sentencepiece: 0.2.1

SentencePiece model:
  /root/.cache/huggingface/hub/models--rmanluo--RoG/snapshots/c73cb678c9d0318f9d1eeeda61cfebd040c7ea11/tokenizer.model
 vocab

In [20]:
# ======================================================================
# CELL 16-R7
# EXACT ORIGINAL NOTEBOOK TOKENIZER ENVIRONMENT AUDIT
# transformers 4.44.2 / tokenizers 0.19.1 / sentencepiece 0.2.0
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - run Cell 17
#   - rerun M3
#   - restart kernel
#   - modify TEST planning files
#
# This creates an isolated Python package directory and runs the
# tokenizer audit in a subprocess, leaving the current kernel untouched.
# ======================================================================

import json
import os
import subprocess
import sys
import textwrap
from pathlib import Path


print("=" * 118)
print("CELL 16-R7 — EXACT ORIGINAL TOKENIZER ENVIRONMENT AUDIT")
print("=" * 118)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "webqsp_test_plan_rows" in globals()
assert "cwq_test_plan_rows" in globals()
assert "webqsp_val_plan_rows" in globals()
assert "cwq_val_plan_rows" in globals()


print("\nFreeze gate: PASSED")


# ======================================================================
# 2. EXACT ORIGINAL NOTEBOOK SOFTWARE SPEC
# ======================================================================

TARGET_TRANSFORMERS = "4.44.2"
TARGET_TOKENIZERS = "0.19.1"
TARGET_SENTENCEPIECE = "0.2.0"


print("\nExact original notebook environment:")
print(" transformers:  ", TARGET_TRANSFORMERS)
print(" tokenizers:    ", TARGET_TOKENIZERS)
print(" sentencepiece: ", TARGET_SENTENCEPIECE)


# ======================================================================
# 3. PREPARE ISOLATED ENVIRONMENT
# ======================================================================

ENV_DIR = Path(
    "/kaggle/working/rog_exact_tokenizer_env_4442"
)

ENV_DIR.mkdir(
    parents=True,
    exist_ok=True
)


marker = (
    ENV_DIR
    / "_INSTALL_COMPLETE.json"
)


need_install = True


if marker.exists():

    try:

        marker_data = json.loads(
            marker.read_text(
                encoding="utf-8"
            )
        )

        need_install = (
            marker_data.get("transformers")
            != TARGET_TRANSFORMERS
            or
            marker_data.get("tokenizers")
            != TARGET_TOKENIZERS
            or
            marker_data.get("sentencepiece")
            != TARGET_SENTENCEPIECE
        )

    except Exception:

        need_install = True


if need_install:

    print(
        "\nInstalling exact tokenizer environment "
        "into isolated directory..."
    )

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--target",
        str(ENV_DIR),

        "transformers==4.44.2",
        "tokenizers==0.19.1",
        "sentencepiece==0.2.0",

        # Notebook Cell 1 also pinned this numerical range.
        "numpy>=2.0,<2.1",
    ]


    subprocess.run(
        cmd,
        check=True
    )


    marker.write_text(
        json.dumps(
            {
                "transformers":
                    TARGET_TRANSFORMERS,

                "tokenizers":
                    TARGET_TOKENIZERS,

                "sentencepiece":
                    TARGET_SENTENCEPIECE,
            },
            indent=2
        ),
        encoding="utf-8"
    )


    print("Isolated installation: COMPLETE")

else:

    print(
        "\nExisting exact isolated environment reused."
    )


# ======================================================================
# 4. SERIALIZE NON-GOLD AUDIT INPUTS
#
# No answer labels are required.
# ======================================================================

AUDIT_DIR = Path(
    "/kaggle/working/step2_rq1_test/"
    "tokenizer_fidelity_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


audit_input_path = (
    AUDIT_DIR
    / "cell16_r7_inputs.json"
)


def canonical_relation_tokens(rows):

    out = []

    for rec in rows:

        for plan in rec.get(
            "predicted_paths",
            []
        ):

            if not isinstance(
                plan,
                (list, tuple)
            ):
                continue

            for relation in plan:

                relation = str(
                    relation
                )

                # Only already-canonical validation tokens.
                if not any(
                    c.isspace()
                    for c in relation
                ):
                    out.append(
                        relation
                    )

    return out


audit_payload = {
    "webqsp_questions": [
        {
            "id": str(x["id"]),
            "question": str(x["question"]),
        }
        for x in webqsp_test_plan_rows[:1000]
    ],

    "cwq_questions": [
        {
            "id": str(x["id"]),
            "question": str(x["question"]),
        }
        for x in cwq_test_plan_rows[:1000]
    ],

    "webqsp_validation_relations":
        canonical_relation_tokens(
            webqsp_val_plan_rows
        ),

    "cwq_validation_relations":
        canonical_relation_tokens(
            cwq_val_plan_rows
        ),
}


audit_input_path.write_text(
    json.dumps(
        audit_payload,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nAudit payload written:",
    audit_input_path
)


# ======================================================================
# 5. BUILD ISOLATED SUBPROCESS SCRIPT
# ======================================================================

audit_output_path = (
    AUDIT_DIR
    / "cell16_r7_results.json"
)


script_path = (
    AUDIT_DIR
    / "cell16_r7_subprocess.py"
)


script = r'''
import json
import sys
from pathlib import Path

import transformers
import tokenizers
import sentencepiece

from transformers import AutoTokenizer


INPUT_PATH = Path(sys.argv[1])
OUTPUT_PATH = Path(sys.argv[2])


payload = json.loads(
    INPUT_PATH.read_text(
        encoding="utf-8"
    )
)


tokenizer = AutoTokenizer.from_pretrained(
    "rmanluo/RoG",
    use_fast=False,
    clean_up_tokenization_spaces=True,
    local_files_only=True,
)


def audit_questions(rows):

    checked = 0
    unk_questions = 0
    unk_token_total = 0

    lengths = []

    examples_with_unk = []


    for rec in rows:

        ids = tokenizer.encode(
            rec["question"],
            add_special_tokens=False
        )

        checked += 1
        lengths.append(len(ids))

        unk_count = sum(
            int(x == tokenizer.unk_token_id)
            for x in ids
        )

        unk_token_total += unk_count


        if unk_count > 0:

            unk_questions += 1

            if len(examples_with_unk) < 10:

                examples_with_unk.append(
                    {
                        "id":
                            rec["id"],

                        "question":
                            rec["question"],

                        "unk_count":
                            unk_count,

                        "ids":
                            ids,
                    }
                )


    return {
        "checked":
            checked,

        "questions_with_unk":
            unk_questions,

        "unk_token_total":
            unk_token_total,

        "mean_length":
            (
                sum(lengths) / len(lengths)
                if lengths
                else 0
            ),

        "examples_with_unk":
            examples_with_unk,
    }


def audit_relations(relations):

    checked = 0
    exact = 0
    failures = []


    for relation in relations:

        ids = tokenizer.encode(
            relation,
            add_special_tokens=False
        )

        decoded = tokenizer.decode(
            ids,
            skip_special_tokens=True
        )

        checked += 1


        if decoded == relation:

            exact += 1

        elif len(failures) < 20:

            failures.append(
                {
                    "relation":
                        relation,

                    "decoded":
                        decoded,

                    "ids":
                        ids,
                }
            )


    return {
        "checked":
            checked,

        "exact":
            exact,

        "exact_rate":
            (
                exact / checked
                if checked
                else 0
            ),

        "failures":
            failures,
    }


probe = (
    "people.person.place_of_birth"
)

probe_ids = tokenizer.encode(
    probe,
    add_special_tokens=False
)

probe_decoded = tokenizer.decode(
    probe_ids,
    skip_special_tokens=True
)


results = {
    "software": {
        "transformers":
            transformers.__version__,

        "tokenizers":
            tokenizers.__version__,

        "sentencepiece":
            sentencepiece.__version__,

        "tokenizer_class":
            tokenizer.__class__.__name__,

        "tokenizer_module":
            tokenizer.__class__.__module__,

        "is_fast":
            getattr(
                tokenizer,
                "is_fast",
                None
            ),

        "name_or_path":
            getattr(
                tokenizer,
                "name_or_path",
                None
            ),
    },

    "probe": {
        "input":
            probe,

        "ids":
            probe_ids,

        "decoded":
            probe_decoded,

        "exact":
            probe_decoded == probe,
    },

    "webqsp_questions":
        audit_questions(
            payload[
                "webqsp_questions"
            ]
        ),

    "cwq_questions":
        audit_questions(
            payload[
                "cwq_questions"
            ]
        ),

    "webqsp_validation_relations":
        audit_relations(
            payload[
                "webqsp_validation_relations"
            ]
        ),

    "cwq_validation_relations":
        audit_relations(
            payload[
                "cwq_validation_relations"
            ]
        ),

    "generate_seq_special_tokens": {
        "add_bos_token":
            getattr(
                tokenizer,
                "add_bos_token",
                None
            ),

        "add_eos_token":
            getattr(
                tokenizer,
                "add_eos_token",
                None
            ),

        "bos_token_id":
            tokenizer.bos_token_id,

        "eos_token_id":
            tokenizer.eos_token_id,
    },
}


OUTPUT_PATH.write_text(
    json.dumps(
        results,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)
'''


script_path.write_text(
    script,
    encoding="utf-8"
)


# ======================================================================
# 6. RUN EXACT ENVIRONMENT SUBPROCESS
# ======================================================================

env = os.environ.copy()


existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)


env[
    "PYTHONPATH"
] = (
    str(ENV_DIR)
    +
    (
        os.pathsep + existing_pythonpath
        if existing_pythonpath
        else ""
    )
)


print(
    "\nRunning isolated exact-environment audit..."
)


proc = subprocess.run(
    [
        sys.executable,
        str(script_path),
        str(audit_input_path),
        str(audit_output_path),
    ],
    env=env,
    text=True,
    capture_output=True,
)


print(
    "Subprocess return code:",
    proc.returncode
)


if proc.stdout.strip():

    print(
        "\nSUBPROCESS STDOUT:"
    )

    print(
        proc.stdout
    )


if proc.stderr.strip():

    print(
        "\nSUBPROCESS STDERR:"
    )

    print(
        proc.stderr[-5000:]
    )


assert proc.returncode == 0, (
    "Exact-environment subprocess failed. "
    "Do NOT change the main environment."
)


# ======================================================================
# 7. LOAD RESULTS
# ======================================================================

results = json.loads(
    audit_output_path.read_text(
        encoding="utf-8"
    )
)


print(
    "\n"
    + "=" * 118
)

print(
    "A. EXACT ENVIRONMENT"
)

print(
    "=" * 118
)


for k, v in results[
    "software"
].items():

    print(
        f"{k:<25}:",
        v
    )


# Exact software identity.
assert (
    results["software"]["transformers"]
    ==
    "4.44.2"
)

assert (
    results["software"]["tokenizers"]
    ==
    "0.19.1"
)

assert (
    results["software"]["sentencepiece"]
    ==
    "0.2.0"
)


# CRITICAL:
# use_fast=False must produce a genuine slow tokenizer.
assert (
    results[
        "software"
    ][
        "is_fast"
    ]
    is False
), (
    "Even exact 4.44.2 did not return a slow tokenizer."
)


print(
    "\nExact software + slow-tokenizer identity: PASSED"
)


# ======================================================================
# 8. PROBE
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "B. RELATION ROUND-TRIP PROBE"
)

print(
    "=" * 118
)


print(
    "input:",
    repr(
        results[
            "probe"
        ][
            "input"
        ]
    )
)

print(
    "decoded:",
    repr(
        results[
            "probe"
        ][
            "decoded"
        ]
    )
)

print(
    "exact:",
    results[
        "probe"
    ][
        "exact"
    ]
)


assert results[
    "probe"
][
    "exact"
] is True


# ======================================================================
# 9. VALIDATION RELATION FIDELITY
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "C. FROZEN VALIDATION RELATION FIDELITY"
)

print(
    "=" * 118
)


for name in [
    "webqsp_validation_relations",
    "cwq_validation_relations",
]:

    x = results[
        name
    ]

    print(
        f"\n{name}:"
    )

    print(
        " checked:",
        x["checked"]
    )

    print(
        " exact:",
        x["exact"]
    )

    print(
        " exact rate:",
        f"{100 * x['exact_rate']:.6f}%"
    )

    if x[
        "failures"
    ]:

        print(
            " failures:",
            x["failures"][:5]
        )


# These are the canonical validation tokens.
assert (
    results[
        "webqsp_validation_relations"
    ][
        "exact"
    ]
    ==
    results[
        "webqsp_validation_relations"
    ][
        "checked"
    ]
)

assert (
    results[
        "cwq_validation_relations"
    ][
        "exact"
    ]
    ==
    results[
        "cwq_validation_relations"
    ][
        "checked"
    ]
)


print(
    "\nFrozen validation canonical-token fidelity: PASSED"
)


# ======================================================================
# 10. QUESTION TOKENIZATION HEALTH
#
# We do NOT compare against raw SentencePiece here.
# The exact 4.44.2 AutoTokenizer is the reference implementation.
# ======================================================================

print(
    "\n"
    + "=" * 118
)

print(
    "D. QUESTION TOKENIZATION HEALTH"
)

print(
    "=" * 118
)


for name in [
    "webqsp_questions",
    "cwq_questions",
]:

    x = results[
        name
    ]

    print(
        f"\n{name}:"
    )

    print(
        " checked:",
        x["checked"]
    )

    print(
        " questions with UNK:",
        x["questions_with_unk"]
    )

    print(
        " total UNK tokens:",
        x["unk_token_total"]
    )

    print(
        " mean tokenized length:",
        x["mean_length"]
    )

    if x[
        "examples_with_unk"
    ]:

        print(
            " UNK examples:",
            x["examples_with_unk"][:5]
        )


# SentencePiece should not explode ordinary natural-language questions
# into the v5 behavior we saw.
assert (
    results[
        "webqsp_questions"
    ][
        "questions_with_unk"
    ]
    == 0
)

assert (
    results[
        "cwq_questions"
    ][
        "questions_with_unk"
    ]
    == 0
)


print(
    "\nQuestion-tokenization health gate: PASSED"
)


# ======================================================================
# 11. SPECIAL TOKEN GATE
# ======================================================================

special = results[
    "generate_seq_special_tokens"
]


print(
    "\n"
    + "=" * 118
)

print(
    "E. generate_seq SPECIAL-TOKEN BEHAVIOR"
)

print(
    "=" * 118
)


for k, v in special.items():

    print(
        f"{k:<20}:",
        v
    )


assert special[
    "add_bos_token"
] is True

assert special[
    "add_eos_token"
] is False

assert special[
    "bos_token_id"
] == 1

assert special[
    "eos_token_id"
] == 2


print(
    "\ngenerate_seq tokenization behavior gate: PASSED"
)


# ======================================================================
# 12. FINAL DECISION
# ======================================================================

EXACT_4442_TOKENIZER_FIDELITY_PASSED = True


print(
    "\n"
    + "=" * 118
)

print(
    "CELL 16-R7 EXACT TOKENIZER AUDIT: PASSED"
)

print(
    "=" * 118
)

print(
    "\nExperimental reference environment:"
)

print(
    " transformers==4.44.2"
)

print(
    " tokenizers==0.19.1"
)

print(
    " sentencepiece==0.2.0"
)

print(
    "\nCurrent v5 M3 TEST realization valid:",
    "NO"
)

print(
    "Whitespace repair of M3 accepted:",
    "NO"
)

print(
    "TEST regeneration required:",
    "YES"
)

print(
    "\nTEST JSONLs modified:",
    "NO"
)

print(
    "AFP changed:",
    "NO"
)

print(
    "TEST tuning:",
    "NO"
)

print(
    "\nResults:",
    audit_output_path
)

print(
    "\nNEXT:"
)

print(
    "Paste this output here."
)

print(
    "Do NOT regenerate TEST plans until the "
    "exact generation environment is prepared."
)

CELL 16-R7 — EXACT ORIGINAL TOKENIZER ENVIRONMENT AUDIT

Freeze gate: PASSED

Exact original notebook environment:
 transformers:   4.44.2
 tokenizers:     0.19.1
 sentencepiece:  0.2.0

Installing exact tokenizer environment into isolated directory...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 5.0.0 requires fsspec[http]<=2026.4.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, bu

Isolated installation: COMPLETE

Audit payload written: /kaggle/working/step2_rq1_test/tokenizer_fidelity_audit/cell16_r7_inputs.json

Running isolated exact-environment audit...
Subprocess return code: 0

SUBPROCESS STDERR:
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.

0it [00:00, ?it/s]
0it [00:00, ?it/s]


A. EXACT ENVIRONMENT
transformers             : 4.44.2
tokenizers               : 0.19.1
sentencepiece            : 0.2.0
tokenizer_class          : LlamaTokenizer
tokenizer_module         : transformers.models.llama.tokenization_llama
is_fast                  : False
name_or_path             : rmanluo/RoG

Exact software + slow-tokenizer identity: PASSED

B. RELATION ROUND-TRIP PROBE
input: 'people.person.place_of_birth'
decoded: 'people.person.place_of_birth'
exact: True

C. FROZEN VALIDATION RE

In [21]:
# ======================================================================
# CELL 16-R8
# QUARANTINE INVALID v5 M3 + EXACT GENERATION-ENVIRONMENT PREFLIGHT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M3
#   - run Cell 17
#   - restart kernel
#   - overwrite either original M3 JSONL
#
# This cell:
#   1. verifies the invalid M3 artifacts by their frozen hashes
#   2. copies them into an explicit audit quarantine
#   3. records why they were rejected
#   4. recovers the original notebook software setup
#   5. audits current/exact subprocess Torch+CUDA state
#   6. recovers the ACTUAL executed M3 source from history if available
#
# NO model generation occurs.
# ======================================================================

import ast
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch


print("=" * 120)
print("CELL 16-R8 — INVALID-M3 QUARANTINE + GENERATION PREFLIGHT")
print("=" * 120)


# ======================================================================
# 1. HARD SCIENTIFIC GATES
# ======================================================================

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert globals().get(
    "CELL16_ORACLE_REPAIR_VALIDATED",
    False
) is True

assert globals().get(
    "EXACT_4442_TOKENIZER_FIDELITY_PASSED",
    False
) is True

assert "persisted_nb_16ar" in globals()


print("\nFreeze gate:                    PASSED")
print("Oracle repair gate:             PASSED")
print("Exact 4.44.2 tokenizer gate:    PASSED")


# ======================================================================
# 2. ORIGINAL INVALID M3 FILES
# ======================================================================

WEB_M3_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_webqsp_test.jsonl"
)

CWQ_M3_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "planning_cwq_test.jsonl"
)

M3_MANIFEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16a_m3_materialization_manifest.json"
)


assert WEB_M3_PATH.exists()
assert CWQ_M3_PATH.exists()


WEB_M3_EXPECTED_SHA = (
    "ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790"
)

CWQ_M3_EXPECTED_SHA = (
    "95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53"
)


def sha256_file_r8(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


web_m3_sha = sha256_file_r8(
    WEB_M3_PATH
)

cwq_m3_sha = sha256_file_r8(
    CWQ_M3_PATH
)


assert web_m3_sha == WEB_M3_EXPECTED_SHA
assert cwq_m3_sha == CWQ_M3_EXPECTED_SHA


print("\nInvalid M3 artifact identity: PASSED")
print(" WebQSP:", web_m3_sha)
print(" CWQ:   ", cwq_m3_sha)


# ======================================================================
# 3. LOAD R7 AUDIT EVIDENCE
# ======================================================================

R7_RESULTS_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "tokenizer_fidelity_audit/"
    "cell16_r7_results.json"
)

assert R7_RESULTS_PATH.exists()


r7 = json.loads(
    R7_RESULTS_PATH.read_text(
        encoding="utf-8"
    )
)


assert r7["software"]["transformers"] == "4.44.2"
assert r7["software"]["tokenizers"] == "0.19.1"
assert r7["software"]["sentencepiece"] == "0.2.0"
assert r7["software"]["is_fast"] is False

assert (
    r7["webqsp_validation_relations"]["exact"]
    ==
    r7["webqsp_validation_relations"]["checked"]
)

assert (
    r7["cwq_validation_relations"]["exact"]
    ==
    r7["cwq_validation_relations"]["checked"]
)


print("\nR7 evidence file: PASSED")


# ======================================================================
# 4. QUARANTINE COPY — NEVER DELETE ORIGINAL FAILED RUN
# ======================================================================

QUARANTINE_DIR = Path(
    "/kaggle/working/step2_rq1_test/"
    "invalid_transformers5_m3"
)

QUARANTINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


WEB_QUARANTINE = (
    QUARANTINE_DIR
    /
    "planning_webqsp_test.INVALID_TRANSFORMERS5.jsonl"
)

CWQ_QUARANTINE = (
    QUARANTINE_DIR
    /
    "planning_cwq_test.INVALID_TRANSFORMERS5.jsonl"
)


# copy2 is idempotent here; verify hashes afterwards.
shutil.copy2(
    WEB_M3_PATH,
    WEB_QUARANTINE
)

shutil.copy2(
    CWQ_M3_PATH,
    CWQ_QUARANTINE
)


assert (
    sha256_file_r8(
        WEB_QUARANTINE
    )
    ==
    WEB_M3_EXPECTED_SHA
)

assert (
    sha256_file_r8(
        CWQ_QUARANTINE
    )
    ==
    CWQ_M3_EXPECTED_SHA
)


if M3_MANIFEST_PATH.exists():

    shutil.copy2(
        M3_MANIFEST_PATH,
        QUARANTINE_DIR
        /
        "cell16a_m3_materialization_manifest.INVALID_TRANSFORMERS5.json"
    )


print("\nInvalid M3 quarantine copies: VERIFIED")


# ======================================================================
# 5. WRITE SOFTWARE-FIDELITY FAILURE MANIFEST
# ======================================================================

FAILURE_MANIFEST_PATH = (
    QUARANTINE_DIR
    /
    "invalid_m3_software_fidelity_manifest.json"
)


failure_manifest = {
    "status":
        "INVALID_DO_NOT_USE_FOR_RESULTS",

    "reason":
        (
            "TEST plans were generated under transformers 5.0.0, "
            "where AutoTokenizer(use_fast=False) resolved to "
            "TokenizersBackend / fast behavior rather than the "
            "experiment notebook's transformers 4.44.2 slow "
            "LlamaTokenizer behavior."
        ),

    "scientific_consequence":
        (
            "Planner INPUT token IDs differed materially from the "
            "original experiment tokenizer. Therefore this is not "
            "a decoding-only defect and whitespace post-processing "
            "is not an acceptable repair."
        ),

    "invalid_artifacts": {
        "webqsp": {
            "path":
                str(WEB_M3_PATH),

            "sha256":
                web_m3_sha,
        },

        "cwq": {
            "path":
                str(CWQ_M3_PATH),

            "sha256":
                cwq_m3_sha,
        },
    },

    "invalid_generation_environment": {
        "transformers":
            "5.0.0",

        "observed_tokenizer_class":
            "TokenizersBackend",

        "observed_is_fast":
            True,
    },

    "experimental_reference_environment": {
        "transformers":
            "4.44.2",

        "tokenizers":
            "0.19.1",

        "sentencepiece":
            "0.2.0",

        "tokenizer_class":
            "LlamaTokenizer",

        "is_fast":
            False,
    },

    "r7_evidence":
        str(R7_RESULTS_PATH),

    "afp_changed":
        False,

    "test_used_for_tuning":
        False,

    "failed_test_results_accepted":
        False,

    "whitespace_repair_accepted":
        False,

    "regeneration_required":
        True,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


FAILURE_MANIFEST_PATH.write_text(
    json.dumps(
        failure_manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "Failure manifest:",
    FAILURE_MANIFEST_PATH
)


# ======================================================================
# 6. RECOVER ORIGINAL NOTEBOOK CELL 1 SOFTWARE SETUP
# ======================================================================

setup_source = "".join(
    persisted_nb_16ar[
        "cells"
    ][1].get(
        "source",
        []
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "A. ORIGINAL NOTEBOOK SOFTWARE SETUP"
)

print(
    "=" * 120
)


print(
    setup_source
)


# ----------------------------------------------------------------------
# Try to recover _torch_pin without executing pip.
# ----------------------------------------------------------------------

torch_pin_expression = None

for line in setup_source.splitlines():

    if re.search(
        r"^\s*_torch_pin\s*=",
        line
    ):

        torch_pin_expression = line.strip()

        break


print(
    "\nRecovered _torch_pin assignment:"
)

print(
    torch_pin_expression
)


# ======================================================================
# 7. CURRENT MAIN-KERNEL TORCH/CUDA STATE
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "B. CURRENT MAIN-KERNEL GPU SOFTWARE"
)

print(
    "=" * 120
)


print(
    "Python:",
    sys.version.split()[0]
)

print(
    "torch:",
    torch.__version__
)

print(
    "torch CUDA build:",
    torch.version.cuda
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU count:",
        torch.cuda.device_count()
    )

    print(
        "GPU 0:",
        torch.cuda.get_device_name(0)
    )

    props = torch.cuda.get_device_properties(
        0
    )

    print(
        "GPU memory GiB:",
        round(
            props.total_memory
            /
            (1024 ** 3),
            3
        )
    )


# ======================================================================
# 8. EXACT ISOLATED ENVIRONMENT GPU PREFLIGHT
# ======================================================================

EXACT_ENV_DIR = Path(
    "/kaggle/working/"
    "rog_exact_tokenizer_env_4442"
)

assert EXACT_ENV_DIR.exists()


preflight_script = r'''
import json

import torch
import transformers
import tokenizers
import sentencepiece

from transformers import AutoTokenizer


tok = AutoTokenizer.from_pretrained(
    "rmanluo/RoG",
    use_fast=False,
    clean_up_tokenization_spaces=True,
    local_files_only=True,
)


result = {
    "torch":
        torch.__version__,

    "torch_cuda_build":
        torch.version.cuda,

    "cuda_available":
        torch.cuda.is_available(),

    "gpu_name":
        (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),

    "transformers":
        transformers.__version__,

    "tokenizers":
        tokenizers.__version__,

    "sentencepiece":
        sentencepiece.__version__,

    "tokenizer_class":
        tok.__class__.__name__,

    "is_fast":
        getattr(
            tok,
            "is_fast",
            None
        ),

    "probe_decode":
        tok.decode(
            tok.encode(
                "people.person.place_of_birth",
                add_special_tokens=False
            ),
            skip_special_tokens=True
        ),
}


print(
    json.dumps(
        result
    )
)
'''


env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = (
    str(EXACT_ENV_DIR)
    +
    (
        os.pathsep
        +
        existing_pythonpath

        if existing_pythonpath
        else ""
    )
)


gpu_preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        preflight_script,
    ],
    env=env,
    text=True,
    capture_output=True,
)


print(
    "\n"
    + "=" * 120
)

print(
    "C. EXACT 4.44.2 SUBPROCESS GPU PREFLIGHT"
)

print(
    "=" * 120
)


print(
    "return code:",
    gpu_preflight.returncode
)


if gpu_preflight.stderr.strip():

    print(
        "\nSTDERR:"
    )

    print(
        gpu_preflight.stderr[-4000:]
    )


assert gpu_preflight.returncode == 0


preflight_lines = [
    x.strip()
    for x in gpu_preflight.stdout.splitlines()
    if x.strip()
]


assert preflight_lines


gpu_exact = json.loads(
    preflight_lines[-1]
)


for k, v in gpu_exact.items():

    print(
        f"{k:<24}:",
        v
    )


assert gpu_exact["transformers"] == "4.44.2"
assert gpu_exact["tokenizers"] == "0.19.1"
assert gpu_exact["sentencepiece"] == "0.2.0"
assert gpu_exact["tokenizer_class"] == "LlamaTokenizer"
assert gpu_exact["is_fast"] is False

assert (
    gpu_exact["probe_decode"]
    ==
    "people.person.place_of_birth"
)

assert gpu_exact["cuda_available"] is True


print(
    "\nExact-env CUDA/tokenizer preflight: PASSED"
)


# ======================================================================
# 9. RECOVER THE ACTUAL EXECUTED M3 SOURCE FROM IPYTHON HISTORY
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "D. EXACT EXECUTED M3 SOURCE RECOVERY"
)

print(
    "=" * 120
)


ip = get_ipython()

assert ip is not None


history = list(
    ip.history_manager.input_hist_raw
)


def assigned_names_r8(tree):

    names = set()

    for node in ast.walk(tree):

        if isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            targets = (
                node.targets

                if isinstance(
                    node,
                    ast.Assign
                )

                else [
                    node.target
                ]
            )

            for target in targets:

                if isinstance(
                    target,
                    ast.Name
                ):

                    names.add(
                        target.id
                    )

    return names


m3_candidates = []


for hist_idx, source in enumerate(
    history
):

    if not isinstance(
        source,
        str
    ):
        continue

    if len(source) < 1000:
        continue

    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    names = assigned_names_r8(
        tree
    )


    # Strong structural signatures of the actual M3 materialization cell.
    if (
        "CELL16AM3_COMPLETE"
        in names
        and
        "FROZEN_TEST_PLANS_MATERIALIZED"
        in names
    ):

        m3_candidates.append(
            {
                "history_index":
                    hist_idx,

                "source":
                    source,

                "sha256":
                    hashlib.sha256(
                        source.encode(
                            "utf-8"
                        )
                    ).hexdigest(),

                "chars":
                    len(source),
            }
        )


print(
    "M3 history candidates found:",
    len(m3_candidates)
)


for x in m3_candidates:

    print(
        " history=",
        x["history_index"],
        "| chars=",
        x["chars"],
        "| sha=",
        x["sha256"]
    )


M3_SOURCE_RECOVERED = (
    len(m3_candidates) >= 1
)


if M3_SOURCE_RECOVERED:

    selected_m3 = sorted(
        m3_candidates,
        key=lambda x:
            x["history_index"]
    )[-1]


    EXACT_EXECUTED_M3_SOURCE = (
        selected_m3[
            "source"
        ]
    )


    EXACT_EXECUTED_M3_SOURCE_SHA256 = (
        selected_m3[
            "sha256"
        ]
    )


    M3_SOURCE_PATH = (
        QUARANTINE_DIR
        /
        "executed_m3_source_transformers5_failure.py"
    )


    M3_SOURCE_PATH.write_text(
        EXACT_EXECUTED_M3_SOURCE,
        encoding="utf-8"
    )


    print(
        "\nSelected M3 history index:",
        selected_m3[
            "history_index"
        ]
    )

    print(
        "M3 source SHA256:",
        EXACT_EXECUTED_M3_SOURCE_SHA256
    )

    print(
        "M3 source saved:",
        M3_SOURCE_PATH
    )


else:

    EXACT_EXECUTED_M3_SOURCE = None
    EXACT_EXECUTED_M3_SOURCE_SHA256 = None

    print(
        "\nActual M3 source not found in current "
        "IPython history."
    )

    print(
        "This does NOT affect the quarantined artifacts."
    )


# ======================================================================
# 10. SAVE PREFLIGHT MANIFEST
# ======================================================================

PREFLIGHT_MANIFEST_PATH = Path(
    "/kaggle/working/step2_rq1_test/"
    "cell16_r8_generation_environment_preflight.json"
)


preflight_manifest = {
    "invalid_m3_quarantined":
        True,

    "invalid_m3_webqsp_sha256":
        web_m3_sha,

    "invalid_m3_cwq_sha256":
        cwq_m3_sha,

    "exact_tokenizer_environment_passed":
        True,

    "exact_reference_environment": {
        "transformers":
            "4.44.2",

        "tokenizers":
            "0.19.1",

        "sentencepiece":
            "0.2.0",
    },

    "main_kernel": {
        "torch":
            torch.__version__,

        "torch_cuda_build":
            torch.version.cuda,

        "cuda_available":
            torch.cuda.is_available(),

        "gpu":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
    },

    "exact_subprocess": gpu_exact,

    "torch_pin_expression":
        torch_pin_expression,

    "m3_source_recovered":
        M3_SOURCE_RECOVERED,

    "m3_source_sha256":
        EXACT_EXECUTED_M3_SOURCE_SHA256,

    "model_generation_executed_in_r8":
        False,

    "test_tuning":
        False,

    "afp_changed":
        False,
}


PREFLIGHT_MANIFEST_PATH.write_text(
    json.dumps(
        preflight_manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


CELL16_R8_PREFLIGHT_PASSED = True


print(
    "\n"
    + "=" * 120
)

print(
    "CELL 16-R8 PREFLIGHT COMPLETE"
)

print(
    "=" * 120
)

print(
    "\nInvalid Transformers-5 M3: QUARANTINED"
)

print(
    "Original failed files deleted: NO"
)

print(
    "Original failed files overwritten: NO"
)

print(
    "Exact 4.44.2 tokenizer: PASSED"
)

print(
    "Exact-env CUDA access: PASSED"
)

print(
    "M3 source recovered:",
    M3_SOURCE_RECOVERED
)

print(
    "Model generation executed: NO"
)

print(
    "TEST tuning: NO"
)

print(
    "AFP changed: NO"
)

print(
    "\nPreflight manifest:",
    PREFLIGHT_MANIFEST_PATH
)

print(
    "\nNEXT ACTION:"
)

print(
    "Paste the full R8 output here."
)

print(
    "Do NOT run Cell 17 or regenerate TEST yet."
)

CELL 16-R8 — INVALID-M3 QUARANTINE + GENERATION PREFLIGHT

Freeze gate:                    PASSED
Oracle repair gate:             PASSED
Exact 4.44.2 tokenizer gate:    PASSED

Invalid M3 artifact identity: PASSED
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ:    95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

R7 evidence file: PASSED

Invalid M3 quarantine copies: VERIFIED
Failure manifest: /kaggle/working/step2_rq1_test/invalid_transformers5_m3/invalid_m3_software_fidelity_manifest.json

A. ORIGINAL NOTEBOOK SOFTWARE SETUP
import torch
_torch_pin = f"torch=={torch.__version__.split('+')[0]}"
print("Keeping installed PyTorch:", _torch_pin)

# accelerate==0.33.0 caps numpy<2.0, which drags Kaggle's numpy back from 2.x
# to 1.26.4 -- but Kaggle's pandas wheel is built against numpy 2.x's C ABI,
# so that downgrade breaks pandas at import time ("numpy.dtype size changed").
# accelerate>=0.34 dropped that cap, so bump it and pin numpy exp

In [22]:
# ======================================================================
# KAGGLE ACCOUNT MIGRATION BACKUP — PROJECT STATE
# ======================================================================
#
# RUN AS A NEW CELL IN THE CURRENT ACCOUNT.
#
# Does NOT:
#   - run experiments
#   - modify AFP
#   - modify TEST plans
#   - restart kernel
#
# Creates:
#   /kaggle/working/AdaPruner_KGQA_MIGRATION_20260901.zip
# ======================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile


ROOT = Path("/kaggle/working")

OUT_ZIP = ROOT / "AdaPruner_KGQA_MIGRATION_20260901.zip"

MANIFEST_PATH = ROOT / "AdaPruner_KGQA_MIGRATION_MANIFEST.json"


# ----------------------------------------------------------------------
# Critical project directories/files
# ----------------------------------------------------------------------

targets = [
    ROOT / "step2_rq1_dev",
    ROOT / "step2_rq1_test",
    ROOT / "step3_rq2_dev_v1",
    ROOT / "reasoning-on-graphs",
    ROOT / "rog_exact_tokenizer_env_4442",
]


# Include only things that actually exist.
targets = [
    p for p in targets
    if p.exists()
]


print("=" * 100)
print("ADAPRUNER-KGQA MIGRATION BACKUP")
print("=" * 100)

print("\nTargets:")

for p in targets:
    print(" ", p)


# ----------------------------------------------------------------------
# SHA helper
# ----------------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ----------------------------------------------------------------------
# Inventory files
# ----------------------------------------------------------------------

files = []

for target in targets:

    if target.is_file():

        files.append(target)

    else:

        files.extend(
            p
            for p in target.rglob("*")
            if p.is_file()
        )


files = sorted(
    set(files),
    key=lambda p: str(p)
)


print("\nFiles to archive:", len(files))


# ----------------------------------------------------------------------
# Build manifest BEFORE ZIP
# ----------------------------------------------------------------------

manifest_files = []

total_bytes = 0


for i, path in enumerate(files, start=1):

    size = path.stat().st_size

    total_bytes += size

    rel = path.relative_to(ROOT)

    manifest_files.append(
        {
            "path": str(rel),
            "size_bytes": size,
            "sha256": sha256_file(path),
        }
    )

    if (
        i % 100 == 0
        or i == len(files)
    ):
        print(
            f"Hashed {i}/{len(files)} files"
        )


manifest = {
    "project": "AdaPruner-KGQA",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "root": str(ROOT),
    "file_count": len(files),
    "total_bytes": total_bytes,

    "scientific_state": {
        "afp_development_frozen": globals().get(
            "AFP_DEVELOPMENT_FROZEN"
        ),

        "final_afp_freeze_sha256": globals().get(
            "FINAL_AFP_FREEZE_SHA256"
        ),

        "oracle_repair_validated": globals().get(
            "CELL16_ORACLE_REPAIR_VALIDATED"
        ),

        "exact_4442_tokenizer_fidelity_passed": globals().get(
            "EXACT_4442_TOKENIZER_FIDELITY_PASSED"
        ),

        "invalid_transformers5_m3_not_accepted": True,

        "cell17_started": False,
    },

    "files": manifest_files,
}


MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


print(
    "\nManifest:",
    MANIFEST_PATH
)


# ----------------------------------------------------------------------
# Add manifest itself
# ----------------------------------------------------------------------

files_with_manifest = (
    files
    +
    [MANIFEST_PATH]
)


# ----------------------------------------------------------------------
# ZIP
# ----------------------------------------------------------------------

if OUT_ZIP.exists():
    OUT_ZIP.unlink()


print("\nCreating ZIP...")


with zipfile.ZipFile(
    OUT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for i, path in enumerate(
        files_with_manifest,
        start=1
    ):

        arcname = path.relative_to(ROOT)

        zf.write(
            path,
            arcname=str(arcname)
        )

        if (
            i % 100 == 0
            or i == len(files_with_manifest)
        ):

            print(
                f"Archived {i}/"
                f"{len(files_with_manifest)} files"
            )


# ----------------------------------------------------------------------
# Verify archive can be read
# ----------------------------------------------------------------------

with zipfile.ZipFile(
    OUT_ZIP,
    "r"
) as zf:

    bad = zf.testzip()


assert bad is None, (
    f"ZIP integrity failure at {bad}"
)


zip_sha = sha256_file(
    OUT_ZIP
)


print("\n" + "=" * 100)
print("MIGRATION BACKUP COMPLETE")
print("=" * 100)

print(
    "\nZIP:",
    OUT_ZIP
)

print(
    "ZIP size GiB:",
    round(
        OUT_ZIP.stat().st_size
        / (1024 ** 3),
        3
    )
)

print(
    "ZIP SHA256:",
    zip_sha
)

print(
    "Archived files:",
    len(files_with_manifest)
)

print(
    "\nZIP integrity: PASSED"
)

print(
    "\nDOWNLOAD THIS ZIP BEFORE LEAVING THIS ACCOUNT."
)

ADAPRUNER-KGQA MIGRATION BACKUP

Targets:
  /kaggle/working/step2_rq1_dev
  /kaggle/working/step2_rq1_test
  /kaggle/working/step3_rq2_dev_v1
  /kaggle/working/reasoning-on-graphs
  /kaggle/working/rog_exact_tokenizer_env_4442

Files to archive: 6576
Hashed 100/6576 files
Hashed 200/6576 files
Hashed 300/6576 files
Hashed 400/6576 files
Hashed 500/6576 files
Hashed 600/6576 files
Hashed 700/6576 files
Hashed 800/6576 files
Hashed 900/6576 files
Hashed 1000/6576 files
Hashed 1100/6576 files
Hashed 1200/6576 files
Hashed 1300/6576 files
Hashed 1400/6576 files
Hashed 1500/6576 files
Hashed 1600/6576 files
Hashed 1700/6576 files
Hashed 1800/6576 files
Hashed 1900/6576 files
Hashed 2000/6576 files
Hashed 2100/6576 files
Hashed 2200/6576 files
Hashed 2300/6576 files
Hashed 2400/6576 files
Hashed 2500/6576 files
Hashed 2600/6576 files
Hashed 2700/6576 files
Hashed 2800/6576 files
Hashed 2900/6576 files
Hashed 3000/6576 files
Hashed 3100/6576 files
Hashed 3200/6576 files
Hashed 3300/6576 files

In [16]:
# ======================================================================
# CELL 16-R4A
# EXACT TOKENIZER-ONLY RECOVERY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M3
#   - rerun Cell 16
#   - restart kernel
#
# After this PASSES, rerun Cell 16-R4 unchanged.
# ======================================================================

import ast
import os
from pathlib import Path

import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


print("=" * 105)
print("CELL 16-R4A — TOKENIZER-ONLY RECOVERY")
print("=" * 105)


# ----------------------------------------------------------------------
# 1. Scientific-state gates
# ----------------------------------------------------------------------

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert "persisted_nb_16ar" in globals()

print("\nFreeze gate: PASSED")


# ----------------------------------------------------------------------
# 2. If tokenizer somehow already returned, simply reuse it
# ----------------------------------------------------------------------

if (
    "tokenizer" in globals()
    and
    globals()["tokenizer"] is not None
):

    print("\nTokenizer already exists in globals.")

else:

    # ------------------------------------------------------------------
    # 3. Recover EXACT original tokenizer assignment from initial
    #    planner setup. We previously verified tokenizer assignment
    #    was in persisted notebook cell 13.
    # ------------------------------------------------------------------

    TARGET_CELL = 13

    cell_source = "".join(
        persisted_nb_16ar[
            "cells"
        ][TARGET_CELL].get(
            "source",
            []
        )
    )

    tree = ast.parse(
        cell_source
    )


    tokenizer_assignment = None


    for node in tree.body:

        targets = []

        if isinstance(
            node,
            ast.Assign
        ):
            targets = node.targets

        elif isinstance(
            node,
            ast.AnnAssign
        ):
            targets = [
                node.target
            ]


        for target in targets:

            if (
                isinstance(
                    target,
                    ast.Name
                )
                and
                target.id == "tokenizer"
            ):

                tokenizer_assignment = (
                    ast.get_source_segment(
                        cell_source,
                        node
                    )
                )

                break


        if tokenizer_assignment is not None:
            break


    assert tokenizer_assignment is not None, (
        "Exact tokenizer assignment was not found "
        "in persisted planner setup cell 13."
    )


    print(
        "\nExact tokenizer assignment recovered "
        "from persisted cell 13:"
    )

    print(
        tokenizer_assignment
    )


    # ------------------------------------------------------------------
    # 4. Build assignment index for preceding setup variables
    # ------------------------------------------------------------------

    assignment_index = {}


    for cell_idx in range(
        0,
        TARGET_CELL + 1
    ):

        src = "".join(
            persisted_nb_16ar[
                "cells"
            ][cell_idx].get(
                "source",
                []
            )
        )

        try:
            parsed = ast.parse(
                src
            )

        except Exception:
            continue


        for node in parsed.body:

            if isinstance(
                node,
                ast.Assign
            ):

                for target in node.targets:

                    if isinstance(
                        target,
                        ast.Name
                    ):

                        assignment_index[
                            target.id
                        ] = (
                            cell_idx,
                            ast.get_source_segment(
                                src,
                                node
                            )
                        )


            elif isinstance(
                node,
                ast.AnnAssign
            ):

                if isinstance(
                    node.target,
                    ast.Name
                ):

                    assignment_index[
                        node.target.id
                    ] = (
                        cell_idx,
                        ast.get_source_segment(
                            src,
                            node
                        )
                    )


    # ------------------------------------------------------------------
    # 5. Minimal exact-execution namespace
    # ------------------------------------------------------------------

    ns = {
        "os":
            os,

        "Path":
            Path,

        "torch":
            torch,

        "transformers":
            transformers,

        "AutoTokenizer":
            AutoTokenizer,

        "AutoModelForCausalLM":
            AutoModelForCausalLM,
    }


    # Reuse already surviving globals where appropriate.
    for name in [
        "MODEL_NAME",
        "model_name",
        "model_path",
        "MODEL_PATH",
    ]:

        if name in globals():

            ns[
                name
            ] = globals()[
                name
            ]


    # ------------------------------------------------------------------
    # 6. Recover missing assignment dependencies on demand
    # ------------------------------------------------------------------

    recovering = set()


    def recover_assignment_symbol(
        symbol
    ):

        if symbol in ns:
            return


        if symbol in globals():

            ns[
                symbol
            ] = globals()[
                symbol
            ]

            return


        assert symbol not in recovering, (
            f"Circular setup dependency while "
            f"recovering {symbol}"
        )


        assert symbol in assignment_index, (
            f"Could not recover setup dependency: "
            f"{symbol}"
        )


        recovering.add(
            symbol
        )


        cell_idx, src = (
            assignment_index[
                symbol
            ]
        )


        # Try execution. If another setup variable is missing,
        # recover that exact variable first.
        while True:

            try:

                exec(
                    src,
                    ns
                )

                break

            except NameError as e:

                missing_name = getattr(
                    e,
                    "name",
                    None
                )

                assert missing_name, (
                    f"Unresolved NameError while "
                    f"recovering {symbol}: {e}"
                )

                recover_assignment_symbol(
                    missing_name
                )


        recovering.remove(
            symbol
        )


    # ------------------------------------------------------------------
    # 7. Execute exact tokenizer assignment
    # ------------------------------------------------------------------

    while True:

        try:

            exec(
                tokenizer_assignment,
                ns
            )

            break

        except NameError as e:

            missing_name = getattr(
                e,
                "name",
                None
            )

            assert missing_name, (
                f"Tokenizer recovery failed: {e}"
            )

            recover_assignment_symbol(
                missing_name
            )


    assert "tokenizer" in ns

    tokenizer = ns[
        "tokenizer"
    ]

    globals()[
        "tokenizer"
    ] = tokenizer


# ----------------------------------------------------------------------
# 8. Identity audit
# ----------------------------------------------------------------------

assert tokenizer is not None


tokenizer_name = str(
    getattr(
        tokenizer,
        "name_or_path",
        ""
    )
)


print("\nTokenizer recovered:")
print(
    " class:",
    tokenizer.__class__.__name__
)

print(
    " module:",
    tokenizer.__class__.__module__
)

print(
    " name_or_path:",
    tokenizer_name
)

print(
    " is_fast:",
    getattr(
        tokenizer,
        "is_fast",
        None
    )
)


assert (
    "rmanluo/RoG".lower()
    in
    tokenizer_name.lower()
), (
    f"Unexpected tokenizer identity: "
    f"{tokenizer_name}"
)


print(
    "\nTokenizer model identity: PASSED"
)


# ----------------------------------------------------------------------
# 9. Small non-destructive round-trip probe
# ----------------------------------------------------------------------

probe = (
    "people.person.place_of_birth"
)

ids = tokenizer.encode(
    probe,
    add_special_tokens=False
)

decoded = tokenizer.decode(
    ids,
    skip_special_tokens=True
)


print("\nRound-trip probe:")
print(" original:", repr(probe))
print(" decoded: ", repr(decoded))


print(
    "\n"
    + "=" * 105
)

print(
    "CELL 16-R4A TOKENIZER RECOVERY COMPLETE"
)

print(
    "=" * 105
)

print("\nModel loaded: NO")
print("TEST plans modified: NO")
print("TEST plans regenerated: NO")
print("AFP changed: NO")
print("TEST tuning: NO")

print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16-R4 UNCHANGED."
)

CELL 16-R4A — TOKENIZER-ONLY RECOVERY

Freeze gate: PASSED

Exact tokenizer assignment recovered from persisted cell 13:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, clean_up_tokenization_spaces=True)

Tokenizer recovered:
 class: TokenizersBackend
 module: transformers.tokenization_utils_tokenizers
 name_or_path: rmanluo/RoG
 is_fast: True

Tokenizer model identity: PASSED

Round-trip probe:
 original: 'people.person.place_of_birth'
 decoded:  'pe op le. per son. pl ace _ of _ bir th'

CELL 16-R4A TOKENIZER RECOVERY COMPLETE

Model loaded: NO
TEST plans modified: NO
TEST plans regenerated: NO
AFP changed: NO
TEST tuning: NO

NEXT ACTION:
RERUN CELL 16-R4 UNCHANGED.


In [35]:
# ======================================================================
# RECOVER + MATERIALIZE EXACT FROZEN RoG TEST PLANS
# ======================================================================
# DO NOT:
#   - modify AFP
#   - tune anything
#   - change planner settings
#   - use TEST gold answers in planner input
#
# GOAL
# ----
# 1. Recover exact frozen RoG planner implementation/configuration.
# 2. Locate exact WebQSP/CWQ TEST examples.
# 3. Verify planner software against the frozen VALIDATION plan artifact.
# 4. Generate TEST predicted_paths.
# 5. Save:
#
#    planning_webqsp_test.jsonl
#    planning_cwq_test.jsonl
#
# Then rerun Cell 16 UNCHANGED.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import os
import re
import time
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE GATE
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    # Existing frozen validation planning rows
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing frozen objects:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16A = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16A
), (
    "AFP freeze SHA mismatch."
)


print("Cell 16A freeze gate: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. FROZEN RoG PLANNER IDENTITY
# ======================================================================
#
# These are DEVELOPMENT-FROZEN facts already established earlier.
# They are NOT being selected using TEST.
# ======================================================================

FROZEN_ROG_MODEL_ID = (
    "rmanluo/RoG"
)

FROZEN_ROG_TOP_K = 3

FROZEN_INSTRUCTION_SHA256 = (
    "e3687b4a5081c22c"
)

FROZEN_PLANNER_CONFIG_SHA_PREFIX = (
    "2c36bd"
)


print(
    "\nFrozen planner identity:"
)

print(
    " model:",
    FROZEN_ROG_MODEL_ID
)

print(
    " Top-K:",
    FROZEN_ROG_TOP_K
)

print(
    " instruction SHA prefix:",
    FROZEN_INSTRUCTION_SHA256
)

print(
    " planner config SHA prefix:",
    FROZEN_PLANNER_CONFIG_SHA_PREFIX
)


# ======================================================================
# 3. PATHS
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists(), (
    f"Persisted notebook not found: {NOTEBOOK_PATH}"
)


PLANNING_OUT_DIR = Path(
    "/kaggle/working/step2_rq1_test"
)

PLANNING_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR
    / "planning_cwq_test.jsonl"
)


# ======================================================================
# 4. LOAD PERSISTED NOTEBOOK
# ======================================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16a = json.load(f)


print(
    "\nPersisted notebook loaded:",
    NOTEBOOK_PATH
)


# ======================================================================
# 5. SEARCH FOR EXACT PLANNING CELLS
# ======================================================================
#
# We locate cells containing the known validation planning artifact
# names and/or the frozen model ID.
# ======================================================================

planning_cell_hits = []


SEARCH_TERMS_16A = [
    "planning_webqsp_validation",
    "planning_cwq_validation",
    "rmanluo/RoG",
    "predicted_paths",
    "planning_time_sec",
]


for cell_idx, cell in enumerate(
    persisted_nb_16a[
        "cells"
    ]
):

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    score = sum(
        term in source
        for term in SEARCH_TERMS_16A
    )


    if score >= 2:

        planning_cell_hits.append(
            {
                "cell_idx":
                    int(
                        cell_idx
                    ),

                "score":
                    int(
                        score
                    ),

                "source":
                    source,
            }
        )


assert len(
    planning_cell_hits
) >= 1, (
    "Could not locate the persisted RoG planning implementation."
)


planning_cell_hits = sorted(
    planning_cell_hits,
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "cell_idx"
            ],
        )
)


print(
    "\nCandidate persisted planning cells:"
)


for hit in planning_cell_hits[
    :10
]:

    print(
        f"  cell {hit['cell_idx']}: "
        f"score={hit['score']}"
    )


# ======================================================================
# 6. RECOVER STRING LITERALS AND FIND FROZEN INSTRUCTION
# ======================================================================

def iter_string_literals_16a(
    source
):

    try:

        tree = ast.parse(
            source
        )

    except Exception:

        return []


    values = []


    for node in ast.walk(
        tree
    ):

        if (
            isinstance(
                node,
                ast.Constant
            )
            and
            isinstance(
                node.value,
                str
            )
        ):

            values.append(
                node.value
            )


    return values


def sha256_text_16a(
    text
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


instruction_candidates = []


for cell in persisted_nb_16a[
    "cells"
]:

    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    for text in iter_string_literals_16a(
        source
    ):

        digest = sha256_text_16a(
            text
        )


        if digest.startswith(
            FROZEN_INSTRUCTION_SHA256
        ):

            instruction_candidates.append(
                text
            )


instruction_candidates = list(
    dict.fromkeys(
        instruction_candidates
    )
)


print(
    "\nFrozen instruction literal matches:",
    len(
        instruction_candidates
    )
)


if len(
    instruction_candidates
) == 1:

    FROZEN_ROG_INSTRUCTION = (
        instruction_candidates[
            0
        ]
    )


    print(
        "Exact frozen instruction recovered: PASSED"
    )


else:

    FROZEN_ROG_INSTRUCTION = None


# ======================================================================
# 7. RECOVER PLANNER-RELATED FUNCTION DEFINITIONS
# ======================================================================

PLANNER_FUNCTION_KEYWORDS = [
    "plan",
    "predict",
    "generate",
    "path",
    "relation",
    "parse",
]


recovered_functions_16a = []


for hit in planning_cell_hits:

    source = hit[
        "source"
    ]


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        if not isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            continue


        name_lower = (
            node.name.lower()
        )


        function_source = (
            ast.get_source_segment(
                source,
                node
            )
        )


        if function_source is None:

            continue


        keyword_score = sum(
            keyword
            in
            name_lower
            for keyword
            in PLANNER_FUNCTION_KEYWORDS
        )


        body_score = sum(
            token
            in
            function_source
            for token
            in [
                "generate(",
                "predicted_paths",
                "relation",
                "num_return_sequences",
                "num_beams",
                "tokenizer",
            ]
        )


        if (
            keyword_score
            +
            body_score
            >=
            2
        ):

            recovered_functions_16a.append(
                {
                    "cell_idx":
                        hit[
                            "cell_idx"
                        ],

                    "name":
                        node.name,

                    "source":
                        function_source,

                    "score":
                        keyword_score
                        +
                        body_score,
                }
            )


# Deduplicate by exact name/source.
dedup = {}

for row in recovered_functions_16a:

    key = (
        row[
            "name"
        ],
        row[
            "source"
        ],
    )

    dedup[
        key
    ] = row


recovered_functions_16a = sorted(
    dedup.values(),
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "cell_idx"
            ],
            x[
                "name"
            ],
        )
)


print(
    "\nRecovered planner-related function candidates:"
)


for row in recovered_functions_16a[
    :20
]:

    print(
        f"  cell {row['cell_idx']}: "
        f"{row['name']} "
        f"(score={row['score']})"
    )


# ======================================================================
# 8. SEARCH SURVIVING GLOBAL PLANNER FUNCTIONS
# ======================================================================

global_planner_candidates = []


for name, obj in list(
    globals().items()
):

    if not callable(
        obj
    ):

        continue


    lname = name.lower()


    if not any(
        keyword
        in
        lname
        for keyword
        in [
            "plan",
            "predict",
            "generate",
            "relation",
            "rog",
        ]
    ):

        continue


    try:

        signature = str(
            inspect.signature(
                obj
            )
        )

    except Exception:

        signature = "?"


    global_planner_candidates.append(
        (
            name,
            signature
        )
    )


print(
    "\nSurviving planner-like globals:"
)


for name, signature in (
    global_planner_candidates[
        :30
    ]
):

    print(
        f"  {name}{signature}"
    )


# ======================================================================
# 9. LOCATE RAW TEST DATA — GLOBALS FIRST
# ======================================================================
#
# Required raw TEST row fields:
#
#   id
#   question
#   q_entity
#   a_entity
#   graph
#
# predicted_paths must NOT be required yet.
# ======================================================================

RAW_REQUIRED_16A = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
}


def valid_raw_rows_16a(
    rows
):

    if not isinstance(
        rows,
        list
    ):

        return False


    if len(
        rows
    ) == 0:

        return False


    probes = sorted(
        set(
            [
                0,
                len(
                    rows
                )
                //
                2,
                len(
                    rows
                )
                -
                1,
            ]
        )
    )


    for idx in probes:

        rec = rows[
            idx
        ]


        if not isinstance(
            rec,
            dict
        ):

            return False


        if not RAW_REQUIRED_16A.issubset(
            rec.keys()
        ):

            return False


    return True


def discover_raw_test_global_16a(
    dataset_name
):

    dataset_name = (
        dataset_name.lower()
    )


    preferred_names = [
        f"{dataset_name}_test",
        f"{dataset_name}_test_rows",
        f"{dataset_name}_test_data",
        f"{dataset_name}_raw_test",
        f"{dataset_name}_test_examples",
    ]


    matches = []


    for name in preferred_names:

        if name not in globals():

            continue


        value = globals()[
            name
        ]


        if valid_raw_rows_16a(
            value
        ):

            matches.append(
                (
                    name,
                    value
                )
            )


    if len(
        matches
    ) == 1:

        print(
            f"{dataset_name.upper()} raw TEST "
            f"from global: {matches[0][0]}"
        )

        return matches[
            0
        ][
            1
        ]


    if len(
        matches
    ) > 1:

        id_sets = [
            {
                str(
                    row[
                        "id"
                    ]
                )
                for row in rows
            }
            for _, rows
            in matches
        ]


        assert all(
            id_sets[
                0
            ]
            ==
            ids
            for ids
            in id_sets[
                1:
            ]
        ), (
            f"{dataset_name}: multiple non-identical "
            "raw TEST globals."
        )


        print(
            f"{dataset_name.upper()}: multiple "
            "equivalent raw TEST globals found."
        )


        return matches[
            0
        ][
            1
        ]


    return None


webqsp_raw_test_16a = (
    discover_raw_test_global_16a(
        "webqsp"
    )
)


cwq_raw_test_16a = (
    discover_raw_test_global_16a(
        "cwq"
    )
)


# ======================================================================
# 10. LOCATE RAW TEST FILES IF GLOBALS ARE ABSENT
# ======================================================================

def read_json_or_jsonl_16a(
    path
):

    suffix = (
        path.suffix.lower()
    )


    if suffix == ".jsonl":

        rows = []

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            for line in f:

                line = (
                    line.strip()
                )

                if not line:

                    continue

                rows.append(
                    json.loads(
                        line
                    )
                )


        return rows


    if suffix == ".json":

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(
                f
            )


        if isinstance(
            obj,
            list
        ):

            return obj


        if isinstance(
            obj,
            dict
        ):

            for key in [
                "data",
                "test",
                "examples",
                "rows",
            ]:

                if (
                    key in obj
                    and
                    isinstance(
                        obj[
                            key
                        ],
                        list
                    )
                ):

                    return obj[
                        key
                    ]


    return None


def discover_raw_test_file_16a(
    dataset_name
):

    dataset_name = (
        dataset_name.lower()
    )


    candidate_paths = []


    search_roots = [
        Path(
            "/kaggle/working"
        ),
        Path(
            "/kaggle/input"
        ),
    ]


    for root in search_roots:

        if not root.exists():

            continue


        for pattern in [
            f"*{dataset_name}*test*.json",
            f"*{dataset_name}*test*.jsonl",
            f"*{dataset_name.upper()}*test*.json",
            f"*{dataset_name.upper()}*test*.jsonl",
        ]:

            for path in root.rglob(
                pattern
            ):

                candidate_paths.append(
                    path
                )


    candidate_paths = sorted(
        set(
            candidate_paths
        )
    )


    valid = []


    for path in candidate_paths:

        # Skip output plan files themselves.
        if (
            "planning_"
            in
            path.name.lower()
        ):

            continue


        try:

            rows = (
                read_json_or_jsonl_16a(
                    path
                )
            )

        except Exception:

            continue


        if valid_raw_rows_16a(
            rows
        ):

            valid.append(
                (
                    path,
                    rows
                )
            )


    if len(
        valid
    ) == 0:

        return (
            None,
            None
        )


    print(
        f"\n{dataset_name.upper()} raw TEST "
        "file candidate(s):"
    )


    for path, rows in valid:

        print(
            " ",
            path,
            "| rows:",
            len(
                rows
            )
        )


    id_sets = [
        {
            str(
                row[
                    "id"
                ]
            )
            for row in rows
        }

        for _,
        rows in valid
    ]


    if len(
        valid
    ) > 1:

        assert all(
            id_sets[
                0
            ]
            ==
            ids
            for ids in id_sets[
                1:
            ]
        ), (
            f"{dataset_name}: multiple raw TEST files "
            "have different question sets. "
            "STOP rather than guessing."
        )


    return valid[
        0
    ]


if webqsp_raw_test_16a is None:

    (
        WEBQSP_RAW_TEST_PATH_16A,
        webqsp_raw_test_16a
    ) = discover_raw_test_file_16a(
        "webqsp"
    )

else:

    WEBQSP_RAW_TEST_PATH_16A = None


if cwq_raw_test_16a is None:

    (
        CWQ_RAW_TEST_PATH_16A,
        cwq_raw_test_16a
    ) = discover_raw_test_file_16a(
        "cwq"
    )

else:

    CWQ_RAW_TEST_PATH_16A = None


# ======================================================================
# 11. TEST DATA GATES
# ======================================================================

assert webqsp_raw_test_16a is not None, (
    "WebQSP raw TEST rows were not found."
)


assert cwq_raw_test_16a is not None, (
    "CWQ raw TEST rows were not found."
)


print(
    "\nRaw TEST datasets located:"
)

print(
    " WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    " CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


# Validation/test overlap check.
web_val_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in webqsp_val_plan_rows
}


cwq_val_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in cwq_val_plan_rows
}


web_test_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in webqsp_raw_test_16a
}


cwq_test_ids_16a = {
    str(
        row[
            "id"
        ]
    )
    for row in cwq_raw_test_16a
}


assert not (
    web_val_ids_16a
    &
    web_test_ids_16a
), (
    "WebQSP validation / TEST overlap."
)


assert not (
    cwq_val_ids_16a
    &
    cwq_test_ids_16a
), (
    "CWQ validation / TEST overlap."
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 12. FIND THE EXACT PLANNER CALLABLE
# ======================================================================
#
# Preferred path:
#   reuse a surviving callable that the notebook used.
#
# We inspect candidate names and require exactly one compatible callable
# or recover exact functions from the persisted notebook.
# ======================================================================

def callable_source_16a(
    obj
):

    try:

        return inspect.getsource(
            obj
        )

    except Exception:

        return ""


surviving_exact_candidates = []


for name, signature in (
    global_planner_candidates
):

    obj = globals()[
        name
    ]


    source = (
        callable_source_16a(
            obj
        )
    )


    combined = (
        name.lower()
        +
        " "
        +
        source.lower()
    )


    score = sum(
        token in combined
        for token in [
            "predicted_paths",
            "generate",
            "num_return_sequences",
            "relation",
            "tokenizer",
            "planning",
        ]
    )


    if score >= 2:

        surviving_exact_candidates.append(
            {
                "name":
                    name,

                "object":
                    obj,

                "signature":
                    signature,

                "score":
                    score,
            }
        )


surviving_exact_candidates = sorted(
    surviving_exact_candidates,
    key=lambda x:
        -x[
            "score"
        ]
)


print(
    "\nHigh-confidence surviving planner callable candidate(s):"
)


for row in surviving_exact_candidates[
    :20
]:

    print(
        " ",
        row[
            "name"
        ],
        row[
            "signature"
        ],
        "score=",
        row[
            "score"
        ]
    )


# ======================================================================
# 13. RECOVER EXACT PLANNER NAMESPACE FROM PERSISTED FUNCTION SOURCES
# ======================================================================
#
# We execute ONLY function/class definitions and literal assignments from
# the high-confidence persisted planning cells.
#
# We do NOT execute notebook top-level planning loops.
# ======================================================================

planner_ns_16a = dict(
    globals()
)


for hit in planning_cell_hits:

    source = hit[
        "source"
    ]


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    safe_nodes = []


    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
                ast.Import,
                ast.ImportFrom,
            )
        ):

            safe_nodes.append(
                node
            )


        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            # Literal-only constants.
            value_node = (
                node.value
            )


            try:

                ast.literal_eval(
                    value_node
                )

            except Exception:

                continue


            safe_nodes.append(
                node
            )


    if not safe_nodes:

        continue


    safe_module = ast.Module(
        body=
            safe_nodes,

        type_ignores=[]
    )


    ast.fix_missing_locations(
        safe_module
    )


    try:

        exec(
            compile(
                safe_module,
                filename=(
                    f"<planner_cell_"
                    f"{hit['cell_idx']}>"
                ),
                mode="exec"
            ),
            planner_ns_16a
        )

    except Exception as exc:

        print(
            f"Planner definition recovery warning "
            f"for cell {hit['cell_idx']}: "
            f"{type(exc).__name__}: {exc}"
        )


# ======================================================================
# 14. IDENTIFY EXACT SINGLE-QUESTION PLANNER FUNCTION
# ======================================================================

recovered_callable_candidates = []


for name, obj in (
    planner_ns_16a.items()
):

    if not callable(
        obj
    ):

        continue


    lname = (
        str(
            name
        ).lower()
    )


    if not any(
        token
        in
        lname
        for token in [
            "plan",
            "predict",
            "generate",
            "path",
            "relation",
        ]
    ):

        continue


    try:

        sig = inspect.signature(
            obj
        )

    except Exception:

        continue


    params = set(
        sig.parameters.keys()
    )


    useful_inputs = {
        "question",
        "query",
        "instruction",
        "model",
        "tokenizer",
        "top_k",
        "num_return_sequences",
    }


    overlap = len(
        params
        &
        useful_inputs
    )


    if overlap == 0:

        continue


    try:

        source = inspect.getsource(
            obj
        )

    except Exception:

        source = ""


    score = (
        overlap
        +
        sum(
            token in source
            for token in [
                ".generate(",
                "num_return_sequences",
                "predicted_paths",
                "relation",
            ]
        )
    )


    recovered_callable_candidates.append(
        {
            "name":
                name,

            "object":
                obj,

            "signature":
                sig,

            "score":
                score,
        }
    )


recovered_callable_candidates = sorted(
    recovered_callable_candidates,
    key=lambda x:
        (
            -x[
                "score"
            ],
            x[
                "name"
            ],
        )
)


print(
    "\nRecovered callable candidates:"
)


for row in recovered_callable_candidates[
    :30
]:

    print(
        f"  {row['name']}"
        f"{row['signature']} "
        f"score={row['score']}"
    )


# ======================================================================
# 15. FIND EXISTING PLANNER MODEL/TOKENIZER OBJECTS
# ======================================================================

def find_model_like_16a():

    preferred = [
        "planner_model",
        "model",
        "rog_model",
        "planning_model",
    ]


    for name in preferred:

        if name not in globals():

            continue


        obj = globals()[
            name
        ]


        if hasattr(
            obj,
            "generate"
        ):

            return (
                name,
                obj
            )


    for name, obj in globals().items():

        if hasattr(
            obj,
            "generate"
        ):

            module = (
                type(
                    obj
                ).__module__
            )


            if (
                "transformers"
                in
                module
            ):

                return (
                    name,
                    obj
                )


    return (
        None,
        None
    )


def find_tokenizer_like_16a():

    preferred = [
        "planner_tokenizer",
        "tokenizer",
        "rog_tokenizer",
    ]


    for name in preferred:

        if name not in globals():

            continue


        obj = globals()[
            name
        ]


        if callable(
            obj
        ) and hasattr(
            obj,
            "decode"
        ):

            return (
                name,
                obj
            )


    for name, obj in globals().items():

        if (
            callable(
                obj
            )
            and
            hasattr(
                obj,
                "decode"
            )
        ):

            module = (
                type(
                    obj
                ).__module__
            )


            if (
                "transformers"
                in
                module
            ):

                return (
                    name,
                    obj
                )


    return (
        None,
        None
    )


(
    EXISTING_PLANNER_MODEL_NAME_16A,
    EXISTING_PLANNER_MODEL_16A
) = find_model_like_16a()


(
    EXISTING_PLANNER_TOKENIZER_NAME_16A,
    EXISTING_PLANNER_TOKENIZER_16A
) = find_tokenizer_like_16a()


print(
    "\nExisting planner model:",
    EXISTING_PLANNER_MODEL_NAME_16A
)

print(
    "Existing planner tokenizer:",
    EXISTING_PLANNER_TOKENIZER_NAME_16A
)


# ======================================================================
# 16. CHECK FOR FROZEN PLANNER CONFIG / MANIFEST ARTIFACTS
# ======================================================================

planner_manifest_candidates_16a = []


for root in [
    Path(
        "/kaggle/working"
    ),
]:

    if not root.exists():

        continue


    for path in root.rglob(
        "*.json"
    ):

        lower = str(
            path
        ).lower()


        if not any(
            token
            in
            lower
            for token in [
                "planner",
                "planning",
                "relation_plan",
            ]
        ):

            continue


        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                obj = json.load(
                    f
                )

        except Exception:

            continue


        text = json.dumps(
            obj,
            sort_keys=True
        )


        if (
            FROZEN_ROG_MODEL_ID
            in
            text
            or
            FROZEN_PLANNER_CONFIG_SHA_PREFIX
            in
            text
            or
            "top_k"
            in
            text.lower()
        ):

            planner_manifest_candidates_16a.append(
                (
                    path,
                    obj
                )
            )


print(
    "\nPlanner/config artifact candidates:"
)


for path, _ in (
    planner_manifest_candidates_16a[
        :20
    ]
):

    print(
        " ",
        path
    )


# ======================================================================
# 17. SOFTWARE-RECOVERY STATUS
# ======================================================================

RECOVERY_REPORT_16A = {
    "freeze_sha":
        FINAL_AFP_FREEZE_SHA256,

    "model_id":
        FROZEN_ROG_MODEL_ID,

    "top_k":
        FROZEN_ROG_TOP_K,

    "instruction_recovered":
        FROZEN_ROG_INSTRUCTION
        is not None,

    "instruction_sha_prefix":
        FROZEN_INSTRUCTION_SHA256,

    "planning_cell_indices":
        [
            row[
                "cell_idx"
            ]
            for row in planning_cell_hits
        ],

    "recovered_planner_functions":
        [
            {
                "name":
                    row[
                        "name"
                    ],

                "signature":
                    str(
                        row[
                            "signature"
                        ]
                    ),

                "score":
                    int(
                        row[
                            "score"
                        ]
                    ),
            }

            for row in
            recovered_callable_candidates[
                :30
            ]
        ],

    "existing_model_name":
        EXISTING_PLANNER_MODEL_NAME_16A,

    "existing_tokenizer_name":
        EXISTING_PLANNER_TOKENIZER_NAME_16A,

    "webqsp_test_questions":
        len(
            webqsp_raw_test_16a
        ),

    "cwq_test_questions":
        len(
            cwq_raw_test_16a
        ),

    "webqsp_raw_test_path":
        (
            str(
                WEBQSP_RAW_TEST_PATH_16A
            )
            if
            WEBQSP_RAW_TEST_PATH_16A
            is not None
            else
            None
        ),

    "cwq_raw_test_path":
        (
            str(
                CWQ_RAW_TEST_PATH_16A
            )
            if
            CWQ_RAW_TEST_PATH_16A
            is not None
            else
            None
        ),
}


RECOVERY_REPORT_PATH_16A = (
    PLANNING_OUT_DIR
    / "cell16a_planner_recovery_report.json"
)


with open(
    RECOVERY_REPORT_PATH_16A,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RECOVERY_REPORT_16A,
        f,
        indent=2,
        ensure_ascii=False
    )


# ======================================================================
# 18. DO NOT GENERATE UNTIL RECOVERY IS UNAMBIGUOUS
# ======================================================================
#
# This is an intentional gate.
#
# The output from this cell tells us EXACTLY:
#
#   - which persisted planning function exists
#   - its signature
#   - whether the exact instruction was recovered
#   - whether model/tokenizer survived
#   - where the raw TEST data lives
#
# We will then invoke THAT exact function.
#
# This is safer than guessing the RoG generation wrapper after freeze.
# ======================================================================

print(
    "\n"
    + "=" * 122
)

print(
    "CELL 16A RECOVERY REPORT"
)

print(
    "=" * 122
)


print(
    "\nExact frozen instruction recovered:",
    FROZEN_ROG_INSTRUCTION
    is not None
)


if (
    FROZEN_ROG_INSTRUCTION
    is not None
):

    print(
        "Instruction SHA256:",
        sha256_text_16a(
            FROZEN_ROG_INSTRUCTION
        )
    )


print(
    "\nRaw TEST data:"
)

print(
    " WebQSP questions:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    " CWQ questions:",
    len(
        cwq_raw_test_16a
    )
)


print(
    "\nRecovered exact planner-callable candidates:"
)


for row in recovered_callable_candidates[
    :15
]:

    print(
        " ",
        row[
            "name"
        ],
        row[
            "signature"
        ],
        "score=",
        row[
            "score"
        ]
    )


print(
    "\nExisting model:",
    EXISTING_PLANNER_MODEL_NAME_16A
)

print(
    "Existing tokenizer:",
    EXISTING_PLANNER_TOKENIZER_NAME_16A
)


print(
    "\nRecovery report saved:"
)

print(
    " ",
    RECOVERY_REPORT_PATH_16A
)


print(
    "\nSTATUS:"
)

print(
    "  TEST planning generation has NOT started."
)

print(
    "  TEST gold has NOT been used by the planner."
)

print(
    "  AFP freeze remains unchanged."
)


print(
    "\nNEXT ACTION:"
)

print(
    "Send me this Cell 16A output."
)

print(
    "I will map the recovered exact planner callable "
    "to the TEST rows and give you the short materialization "
    "cell, then you rerun Cell 16 unchanged."
)

Cell 16A freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen planner identity:
 model: rmanluo/RoG
 Top-K: 3
 instruction SHA prefix: e3687b4a5081c22c
 planner config SHA prefix: 2c36bd

Persisted notebook loaded: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb

Candidate persisted planning cells:
  cell 139: score=4
  cell 21: score=2
  cell 27: score=2
  cell 47: score=2
  cell 49: score=2
  cell 63: score=2
  cell 65: score=2
  cell 83: score=2
  cell 87: score=2
  cell 105: score=2

Frozen instruction literal matches: 0

Recovered planner-related function candidates:
  cell 139: run_planning_checkpointed (score=3)
  cell 87: run_retrieval (score=2)

Surviving planner-like globals:
  select_rog(candidate_count)
  extract_plan_container(row)
  plan_is_empty(plan)
  audit_plan_rows(rows, dataset_name)
  discover_plan_files(dataset_name)
  restore_validation_plan_rows(dataset_name, existing_global_names)
  locate_r

AssertionError: WebQSP raw TEST rows were not found.

In [36]:
# ======================================================================
# PURPOSE
# -------
# Fix Cell 16A failure:
#
#   WebQSP raw TEST rows were not found
#
# We now load the OFFICIAL RoG Hugging Face datasets directly:
#
#   rmanluo/RoG-webqsp
#   rmanluo/RoG-cwq
#
# Then:
#   1. verify validation identity against frozen validation planning rows
#   2. expose exact TEST rows
#   3. recover run_planning_checkpointed from persisted notebook cell 139
#   4. inspect exact signature/source dependencies
#
# THIS CELL DOES NOT GENERATE TEST PLANS YET.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset


# ======================================================================
# 1. HARD FREEZE GATE
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing frozen prerequisites:\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AR = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AR
)


print("Cell 16A-R freeze gate: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 2. OFFICIAL FROZEN DATASET IDENTITIES
# ======================================================================

OFFICIAL_ROG_DATASETS_16AR = {
    "webqsp":
        "rmanluo/RoG-webqsp",

    "cwq":
        "rmanluo/RoG-cwq",
}


EXPECTED_SPLIT_COUNTS_16AR = {
    "webqsp": {
        "train": 2826,
        "validation": 246,
        "test": 1628,
    },

    "cwq": {
        "train": 27639,
        "validation": 3519,
        "test": 3531,
    },
}


print(
    "\nFrozen official data sources:"
)

for dataset_name, repo in (
    OFFICIAL_ROG_DATASETS_16AR.items()
):

    print(
        f"  {dataset_name}: {repo}"
    )


# ======================================================================
# 3. LOAD OFFICIAL VALIDATION + TEST SPLITS
# ======================================================================
#
# Validation is loaded deliberately so that we can prove that the
# official HF source is the SAME source used by our frozen development
# pipeline before opening TEST results.
#
# No TEST labels are used for any model selection.
# ======================================================================

def load_official_split_16ar(
    dataset_name,
    split
):

    repo = (
        OFFICIAL_ROG_DATASETS_16AR[
            dataset_name
        ]
    )


    print(
        f"\nLoading {repo} [{split}] ..."
    )


    ds = load_dataset(
        repo,
        split=split
    )


    expected = (
        EXPECTED_SPLIT_COUNTS_16AR[
            dataset_name
        ][
            split
        ]
    )


    assert len(
        ds
    ) == expected, (
        f"{dataset_name} {split} count mismatch: "
        f"{len(ds)} != {expected}"
    )


    required_fields = {
        "id",
        "question",
        "q_entity",
        "a_entity",
        "graph",
    }


    assert required_fields.issubset(
        set(
            ds.column_names
        )
    ), (
        f"{dataset_name} {split}: missing expected fields. "
        f"Found {ds.column_names}"
    )


    print(
        f"{dataset_name.upper()} {split}: "
        f"{len(ds)} rows"
    )


    return ds


webqsp_official_val_16ar = (
    load_official_split_16ar(
        "webqsp",
        "validation"
    )
)


cwq_official_val_16ar = (
    load_official_split_16ar(
        "cwq",
        "validation"
    )
)


webqsp_official_test_16ar = (
    load_official_split_16ar(
        "webqsp",
        "test"
    )
)


cwq_official_test_16ar = (
    load_official_split_16ar(
        "cwq",
        "test"
    )
)


# ======================================================================
# 4. NORMALIZATION HELPERS FOR SOURCE-IDENTITY GATE
# ======================================================================

def norm_sequence_16ar(
    value
):

    if value is None:

        return []


    if isinstance(
        value,
        np.ndarray
    ):

        value = value.tolist()


    if isinstance(
        value,
        tuple
    ):

        value = list(
            value
        )


    if not isinstance(
        value,
        list
    ):

        value = [
            value
        ]


    return [
        str(
            x
        )
        for x in value
    ]


def canonical_graph_16ar(
    graph
):

    if isinstance(
        graph,
        np.ndarray
    ):

        graph = graph.tolist()


    return [
        [
            str(
                x
            )
            for x in triplet
        ]
        for triplet in graph
    ]


def graph_sha_16ar(
    graph
):

    canonical = json.dumps(
        canonical_graph_16ar(
            graph
        ),
        ensure_ascii=False,
        separators=(",", ":")
    )


    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


# ======================================================================
# 5. VALIDATION DATA-SOURCE IDENTITY GATE
# ======================================================================
#
# This is important:
#
# We do NOT merely assume the HF repo is the same dataset source.
#
# We verify:
#   - all IDs
#   - exact questions
#   - q_entity
#   - a_entity
#
# and exact graph equality on deterministic samples.
# ======================================================================

def validate_source_identity_16ar(
    dataset_name,
    official_val,
    frozen_val_rows
):

    official_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in official_val
    }


    frozen_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in frozen_val_rows
    }


    assert set(
        official_by_id
    ) == set(
        frozen_by_id
    ), (
        f"{dataset_name}: validation ID sets differ "
        "between official HF source and frozen plans."
    )


    mismatch_question = 0
    mismatch_q_entity = 0
    mismatch_a_entity = 0


    sorted_ids = sorted(
        frozen_by_id.keys()
    )


    for qid in sorted_ids:

        hf = official_by_id[
            qid
        ]


        frozen = frozen_by_id[
            qid
        ]


        if str(
            hf[
                "question"
            ]
        ) != str(
            frozen[
                "question"
            ]
        ):

            mismatch_question += 1


        if norm_sequence_16ar(
            hf[
                "q_entity"
            ]
        ) != norm_sequence_16ar(
            frozen[
                "q_entity"
            ]
        ):

            mismatch_q_entity += 1


        if norm_sequence_16ar(
            hf[
                "a_entity"
            ]
        ) != norm_sequence_16ar(
            frozen[
                "a_entity"
            ]
        ):

            mismatch_a_entity += 1


    assert mismatch_question == 0
    assert mismatch_q_entity == 0
    assert mismatch_a_entity == 0


    # --------------------------------------------------------------
    # Deterministic graph equality sample.
    #
    # Full CWQ validation graph serialization is very large, so exact
    # graph equality is checked at deterministic spread-out positions.
    # --------------------------------------------------------------

    n = len(
        sorted_ids
    )


    sample_positions = sorted(
        set(
            [
                0,
                1,
                n // 10,
                n // 4,
                n // 2,
                (3 * n) // 4,
                (9 * n) // 10,
                n - 2,
                n - 1,
            ]
        )
    )


    graph_checks = 0


    for pos in sample_positions:

        qid = sorted_ids[
            pos
        ]


        hf_graph = official_by_id[
            qid
        ][
            "graph"
        ]


        frozen_graph = frozen_by_id[
            qid
        ][
            "graph"
        ]


        hf_sha = graph_sha_16ar(
            hf_graph
        )


        frozen_sha = graph_sha_16ar(
            frozen_graph
        )


        assert hf_sha == frozen_sha, (
            f"{dataset_name}: graph mismatch for "
            f"validation qid={qid}"
        )


        graph_checks += 1


    print(
        f"{dataset_name.upper()} official-source "
        "validation identity: PASSED"
    )

    print(
        "  IDs:",
        len(
            sorted_ids
        )
    )

    print(
        "  question/entity mismatches: 0"
    )

    print(
        "  exact graph samples:",
        graph_checks,
        "/",
        graph_checks
    )


validate_source_identity_16ar(
    "webqsp",
    webqsp_official_val_16ar,
    webqsp_val_plan_rows
)


validate_source_identity_16ar(
    "cwq",
    cwq_official_val_16ar,
    cwq_val_plan_rows
)


print(
    "\nOfficial RoG dataset-source identity gate: PASSED"
)


# ======================================================================
# 6. MATERIALIZE RAW TEST ROWS IN NOTEBOOK MEMORY
# ======================================================================

def dataset_to_rows_16ar(
    ds
):

    rows = []


    for row in ds:

        rows.append(
            {
                "id":
                    str(
                        row[
                            "id"
                        ]
                    ),

                "question":
                    str(
                        row[
                            "question"
                        ]
                    ),

                "answer":
                    norm_sequence_16ar(
                        row.get(
                            "answer",
                            []
                        )
                    ),

                "q_entity":
                    norm_sequence_16ar(
                        row[
                            "q_entity"
                        ]
                    ),

                "a_entity":
                    norm_sequence_16ar(
                        row[
                            "a_entity"
                        ]
                    ),

                "graph":
                    canonical_graph_16ar(
                        row[
                            "graph"
                        ]
                    ),

                "choices":
                    row.get(
                        "choices",
                        []
                    ),
            }
        )


    return rows


webqsp_raw_test_16ar = (
    dataset_to_rows_16ar(
        webqsp_official_test_16ar
    )
)


cwq_raw_test_16ar = (
    dataset_to_rows_16ar(
        cwq_official_test_16ar
    )
)


assert len(
    webqsp_raw_test_16ar
) == 1628


assert len(
    cwq_raw_test_16ar
) == 3531


# Expose standard names for later planner/materialization cells.
webqsp_raw_test_16a = (
    webqsp_raw_test_16ar
)

cwq_raw_test_16a = (
    cwq_raw_test_16ar
)


print(
    "\nRaw TEST rows materialized:"
)

print(
    "  WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    "  CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


# ======================================================================
# 7. TEST / VALIDATION DISJOINTNESS
# ======================================================================

def ids_16ar(
    rows
):

    return {
        str(
            row[
                "id"
            ]
        )
        for row in rows
    }


assert not (
    ids_16ar(
        webqsp_raw_test_16a
    )
    &
    ids_16ar(
        webqsp_val_plan_rows
    )
)


assert not (
    ids_16ar(
        cwq_raw_test_16a
    )
    &
    ids_16ar(
        cwq_val_plan_rows
    )
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 8. LOAD PERSISTED NOTEBOOK
# ======================================================================

NOTEBOOK_PATH_16AR = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)


assert NOTEBOOK_PATH_16AR.exists()


with open(
    NOTEBOOK_PATH_16AR,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16ar = (
        json.load(
            f
        )
    )


# ======================================================================
# 9. EXTRACT CELL 139 EXACT SOURCE
# ======================================================================

PLANNER_CELL_INDEX_16AR = 139


assert (
    PLANNER_CELL_INDEX_16AR
    <
    len(
        persisted_nb_16ar[
            "cells"
        ]
    )
)


planner_cell_139_16ar = (
    persisted_nb_16ar[
        "cells"
    ][
        PLANNER_CELL_INDEX_16AR
    ]
)


assert (
    planner_cell_139_16ar.get(
        "cell_type"
    )
    ==
    "code"
)


planner_cell_source_16ar = "".join(
    planner_cell_139_16ar.get(
        "source",
        []
    )
)


assert (
    "run_planning_checkpointed"
    in
    planner_cell_source_16ar
), (
    "Cell 139 no longer contains "
    "run_planning_checkpointed."
)


print(
    "\nPersisted planner cell 139 recovered: PASSED"
)


# ======================================================================
# 10. EXTRACT EXACT run_planning_checkpointed SOURCE
# ======================================================================

tree_16ar = ast.parse(
    planner_cell_source_16ar
)


run_planning_source_16ar = None


for node in tree_16ar.body:

    if (
        isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        )
        and
        node.name
        ==
        "run_planning_checkpointed"
    ):

        run_planning_source_16ar = (
            ast.get_source_segment(
                planner_cell_source_16ar,
                node
            )
        )

        break


assert run_planning_source_16ar is not None


print(
    "Exact run_planning_checkpointed "
    "function source recovered: PASSED"
)


# ======================================================================
# 11. RECOVER ALL SAFE DEFINITIONS FROM CELL 139
# ======================================================================
#
# Only definitions/imports/literal assignments are executed.
# Top-level planner loops are NOT executed.
# ======================================================================

planner_ns_16ar = dict(
    globals()
)


safe_nodes_16ar = []


for node in tree_16ar.body:

    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef,
            ast.Import,
            ast.ImportFrom,
        )
    ):

        safe_nodes_16ar.append(
            node
        )


    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign,
        )
    ):

        value_node = node.value


        if value_node is None:

            continue


        try:

            ast.literal_eval(
                value_node
            )

        except Exception:

            continue


        safe_nodes_16ar.append(
            node
        )


safe_module_16ar = ast.Module(
    body=
        safe_nodes_16ar,

    type_ignores=[]
)


ast.fix_missing_locations(
    safe_module_16ar
)


exec(
    compile(
        safe_module_16ar,
        filename="<persisted_planner_cell_139>",
        mode="exec"
    ),
    planner_ns_16ar
)


assert (
    "run_planning_checkpointed"
    in
    planner_ns_16ar
)


RUN_PLANNING_CHECKPOINTED_16AR = (
    planner_ns_16ar[
        "run_planning_checkpointed"
    ]
)


# ======================================================================
# 12. EXACT SIGNATURE
# ======================================================================

RUN_PLANNING_SIGNATURE_16AR = (
    inspect.signature(
        RUN_PLANNING_CHECKPOINTED_16AR
    )
)


print(
    "\nExact run_planning_checkpointed signature:"
)

print(
    " ",
    RUN_PLANNING_SIGNATURE_16AR
)


# ======================================================================
# 13. SOURCE DEPENDENCY SCAN
# ======================================================================

function_tree_16ar = ast.parse(
    run_planning_source_16ar
)


called_names_16ar = set()


for node in ast.walk(
    function_tree_16ar
):

    if isinstance(
        node,
        ast.Call
    ):

        if isinstance(
            node.func,
            ast.Name
        ):

            called_names_16ar.add(
                node.func.id
            )


defined_or_builtin_16ar = {
    "len",
    "str",
    "int",
    "float",
    "bool",
    "list",
    "dict",
    "set",
    "tuple",
    "enumerate",
    "range",
    "print",
    "open",
    "min",
    "max",
    "sum",
    "sorted",
}


external_called_names_16ar = sorted(
    name
    for name in called_names_16ar

    if (
        name
        not in
        defined_or_builtin_16ar
    )
)


print(
    "\nDirect function dependencies referenced by "
    "run_planning_checkpointed:"
)


for name in external_called_names_16ar:

    status = (
        "AVAILABLE"
        if
        name
        in
        planner_ns_16ar
        else
        "MISSING"
    )

    print(
        f"  {name:<40} {status}"
    )


# ======================================================================
# 14. PRINT EXACT FUNCTION SOURCE
# ======================================================================
#
# This is deliberately shown now because it is the final point before
# TEST planner generation.
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "EXACT PERSISTED run_planning_checkpointed SOURCE"
)

print(
    "=" * 120
)


print(
    run_planning_source_16ar
)


# ======================================================================
# 15. SAVE RAW TEST SNAPSHOT + RECOVERY REPORT
# ======================================================================

RECOVERY_DIR_16AR = Path(
    "/kaggle/working/"
    "step2_rq1_test/"
    "planner_recovery"
)


RECOVERY_DIR_16AR.mkdir(
    parents=True,
    exist_ok=True
)


def sha_json_rows_16ar(
    rows
):

    h = hashlib.sha256()


    for row in rows:

        # Avoid one giant in-memory string.
        text = json.dumps(
            row,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False
        )


        h.update(
            text.encode(
                "utf-8"
            )
        )

        h.update(
            b"\n"
        )


    return h.hexdigest()


WEBQSP_RAW_TEST_SHA_16AR = (
    sha_json_rows_16ar(
        webqsp_raw_test_16a
    )
)


CWQ_RAW_TEST_SHA_16AR = (
    sha_json_rows_16ar(
        cwq_raw_test_16a
    )
)


CELL16AR_REPORT = {
    "freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "official_dataset_sources": {
        "webqsp":
            "rmanluo/RoG-webqsp",

        "cwq":
            "rmanluo/RoG-cwq",
    },

    "counts": {
        "webqsp_test":
            len(
                webqsp_raw_test_16a
            ),

        "cwq_test":
            len(
                cwq_raw_test_16a
            ),
    },

    "raw_test_sha256": {
        "webqsp":
            WEBQSP_RAW_TEST_SHA_16AR,

        "cwq":
            CWQ_RAW_TEST_SHA_16AR,
    },

    "validation_source_identity":
        True,

    "validation_test_disjoint":
        True,

    "planner_cell_index":
        139,

    "planner_function":
        "run_planning_checkpointed",

    "planner_signature":
        str(
            RUN_PLANNING_SIGNATURE_16AR
        ),

    "direct_dependencies":
        external_called_names_16ar,

    "test_planning_started":
        False,

    "test_used_for_tuning":
        False,
}


CELL16AR_REPORT_PATH = (
    RECOVERY_DIR_16AR
    / "cell16ar_exact_dataset_and_planner_recovery.json"
)


with open(
    CELL16AR_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL16AR_REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AR_COMPLETE = True


# ======================================================================
# 16. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "=== CELL 16A-R: RAW TEST + EXACT PLANNER RECOVERY COMPLETE ==="
)

print(
    "=" * 124
)


print(
    "\nOfficial source identity:"
)

print(
    "  WebQSP validation match: PASSED"
)

print(
    "  CWQ validation match:    PASSED"
)


print(
    "\nRaw TEST:"
)

print(
    "  WebQSP:",
    len(
        webqsp_raw_test_16a
    )
)

print(
    "  CWQ:   ",
    len(
        cwq_raw_test_16a
    )
)


print(
    "\nPlanner:"
)

print(
    "  persisted cell: 139"
)

print(
    "  callable: run_planning_checkpointed"
)

print(
    "  signature:",
    RUN_PLANNING_SIGNATURE_16AR
)


print(
    "\nTEST planner generation started: NO"
)

print(
    "AFP freeze changed: NO"
)


print(
    "\nReport:"
)

print(
    " ",
    CELL16AR_REPORT_PATH
)


print(
    "\nNEXT:"
)

print(
    "Use the exact signature/source printed above "
    "to materialize frozen TEST relation plans."
)

Cell 16A-R freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen official data sources:
  webqsp: rmanluo/RoG-webqsp
  cwq: rmanluo/RoG-cwq

Loading rmanluo/RoG-webqsp [validation] ...


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

CWQ validation: 3519 rows

Loading rmanluo/RoG-webqsp [test] ...
WEBQSP test: 1628 rows

Loading rmanluo/RoG-cwq [test] ...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

CWQ test: 3531 rows
WEBQSP official-source validation identity: PASSED
  IDs: 246
  question/entity mismatches: 0
  exact graph samples: 9 / 9
CWQ official-source validation identity: PASSED
  IDs: 3519
  question/entity mismatches: 0
  exact graph samples: 9 / 9

Official RoG dataset-source identity gate: PASSED

Raw TEST rows materialized:
  WebQSP: 1628
  CWQ:    3531
Validation / TEST disjointness: PASSED

Persisted planner cell 139 recovered: PASSED
Exact run_planning_checkpointed function source recovered: PASSED

Exact run_planning_checkpointed signature:
  (dataset_split, ckpt_path, desc)

Direct function dependencies referenced by run_planning_checkpointed:
  generate_seq                             MISSING
  load_checkpoint                          MISSING
  parse_prediction                         MISSING
  tqdm                                     AVAILABLE

EXACT PERSISTED run_planning_checkpointed SOURCE
def run_planning_checkpointed(dataset_split, ckpt_path, desc):
    do

In [37]:
# ======================================================================
# CELL 16A-MATERIALIZE
# EXACT FROZEN RoG TEST RELATION-PLAN MATERIALIZATION
# ======================================================================
#
# RUN AS A NEW CELL AFTER CELL 16A-R.
#
# DO NOT:
#   - restart kernel
#   - replace Cell 16
#   - rerun development cells
#   - modify AFP
#   - tune using TEST
#
# THIS CELL:
#   1. recovers exact persisted planner dependencies
#   2. recovers exact INSTRUCTION / N_BEAM / prompter
#   3. recovers or restores exact rmanluo/RoG model + tokenizer
#   4. verifies planner software/configuration identity
#   5. generates TEST plans using the EXACT recovered wrapper:
#
#          run_planning_checkpointed(
#              dataset_split,
#              ckpt_path,
#              desc
#          )
#
#   6. saves checkpointed:
#
#      /kaggle/working/step2_rq1_test/
#          planning_webqsp_test.jsonl
#
#      /kaggle/working/step2_rq1_test/
#          planning_cwq_test.jsonl
#
#   7. exposes:
#
#      webqsp_test_plan_rows
#      cwq_test_plan_rows
#
# After this cell completes successfully:
#
#      RERUN CELL 16 UNCHANGED.
#
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import builtins
import hashlib
import inspect
import json
import math
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch

from tqdm.auto import tqdm


# ======================================================================
# 1. HARD FREEZE + CELL 16A-R GATES
# ======================================================================

required = [
    "CELL16AR_COMPLETE",

    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "persisted_nb_16ar",

    "run_planning_source_16ar",
    "planner_cell_source_16ar",

    "RUN_PLANNING_SIGNATURE_16AR",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing Cell 16A-R / frozen prerequisites:\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AM = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AM
), (
    "AFP freeze SHA mismatch. STOP."
)


print(
    "Cell 16A-MATERIALIZE freeze gate: PASSED"
)

print(
    "Freeze SHA:",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 2. FROZEN PLANNER FACTS
# ======================================================================

FROZEN_MODEL_ID_16AM = (
    "rmanluo/RoG"
)

FROZEN_TOP_K_16AM = 3

FROZEN_INSTRUCTION_SHA_PREFIX_16AM = (
    "e3687b4a5081c22c"
)

FROZEN_PLANNER_CONFIG_SHA_PREFIX_16AM = (
    "2c36bd"
)


assert (
    "do_sample=True"
    in
    run_planning_source_16ar
)

assert (
    "max_new_tokens=100"
    in
    run_planning_source_16ar
)

assert (
    "num_beam=N_BEAM"
    in
    run_planning_source_16ar
)


print(
    "\nFrozen wrapper constants:"
)

print(
    "  model:",
    FROZEN_MODEL_ID_16AM
)

print(
    "  beam:",
    FROZEN_TOP_K_16AM
)

print(
    "  do_sample: True"
)

print(
    "  max_new_tokens: 100"
)


# ======================================================================
# 3. OUTPUT PATHS
# ======================================================================

PLANNING_OUT_DIR_16AM = Path(
    "/kaggle/working/step2_rq1_test"
)

PLANNING_OUT_DIR_16AM.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR_16AM
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    PLANNING_OUT_DIR_16AM
    / "planning_cwq_test.jsonl"
)


MATERIALIZE_MANIFEST_PATH_16AM = (
    PLANNING_OUT_DIR_16AM
    / "cell16a_test_plan_materialization_manifest.json"
)


# ======================================================================
# 4. SHA HELPERS
# ======================================================================

def sha256_text_16am(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def sha256_file_16am(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================
# 5. AST HELPERS
# ======================================================================

PLANNER_MAX_CELL_16AM = 139


def target_names_16am(
    target
):

    names = []


    if isinstance(
        target,
        ast.Name
    ):

        names.append(
            target.id
        )


    elif isinstance(
        target,
        (
            ast.Tuple,
            ast.List,
        )
    ):

        for element in target.elts:

            names.extend(
                target_names_16am(
                    element
                )
            )


    return names


def assignment_target_names_16am(
    node
):

    names = []


    if isinstance(
        node,
        ast.Assign
    ):

        for target in node.targets:

            names.extend(
                target_names_16am(
                    target
                )
            )


    elif isinstance(
        node,
        ast.AnnAssign
    ):

        names.extend(
            target_names_16am(
                node.target
            )
        )


    return names


# ======================================================================
# 6. BUILD INDEX OF TOP-LEVEL NOTEBOOK DEFINITIONS
# ======================================================================

NOTEBOOK_INDEX_16AM = {
    "functions":
        {},

    "classes":
        {},

    "assignments":
        {},

    "imports":
        [],

    "seed_calls":
        [],
}


for cell_idx in range(
    min(
        PLANNER_MAX_CELL_16AM + 1,
        len(
            persisted_nb_16ar[
                "cells"
            ]
        )
    )
):

    cell = (
        persisted_nb_16ar[
            "cells"
        ][
            cell_idx
        ]
    )


    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        segment = (
            ast.get_source_segment(
                source,
                node
            )
        )


        if not segment:

            continue


        # --------------------------------------------------------------
        # Imports
        # --------------------------------------------------------------

        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
            )
        ):

            NOTEBOOK_INDEX_16AM[
                "imports"
            ].append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Functions
        # --------------------------------------------------------------

        elif isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            NOTEBOOK_INDEX_16AM[
                "functions"
            ].setdefault(
                node.name,
                []
            ).append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Classes
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.ClassDef
        ):

            NOTEBOOK_INDEX_16AM[
                "classes"
            ].setdefault(
                node.name,
                []
            ).append(
                {
                    "cell_idx":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        # --------------------------------------------------------------
        # Assignments
        # --------------------------------------------------------------

        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            names = (
                assignment_target_names_16am(
                    node
                )
            )


            for name in names:

                NOTEBOOK_INDEX_16AM[
                    "assignments"
                ].setdefault(
                    name,
                    []
                ).append(
                    {
                        "cell_idx":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


        # --------------------------------------------------------------
        # Explicit seed calls
        # --------------------------------------------------------------

        elif isinstance(
            node,
            ast.Expr
        ):

            lower = (
                segment.lower()
            )


            if any(
                token
                in
                lower
                for token
                in [
                    "set_seed(",
                    "manual_seed(",
                    "random.seed(",
                    "np.random.seed(",
                    "numpy.random.seed(",
                ]
            ):

                NOTEBOOK_INDEX_16AM[
                    "seed_calls"
                ].append(
                    {
                        "cell_idx":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


print(
    "\nPersisted notebook AST index: READY"
)


# ======================================================================
# 7. CREATE ISOLATED EXACT PLANNER NAMESPACE
# ======================================================================

planner_ns_16am = {
    "__builtins__":
        builtins.__dict__,

    "ast":
        ast,

    "hashlib":
        hashlib,

    "inspect":
        inspect,

    "json":
        json,

    "math":
        math,

    "os":
        os,

    "random":
        random,

    "re":
        re,

    "time":
        time,

    "Path":
        Path,

    "np":
        np,

    "numpy":
        np,

    "torch":
        torch,

    "tqdm":
        tqdm,
}


# ======================================================================
# 8. EXECUTE PERSISTED IMPORTS
# ======================================================================
#
# Import failures are recorded rather than silently treated as success.
# Many irrelevant notebook imports may legitimately fail; required
# planner imports will be checked later.
# ======================================================================

IMPORT_FAILURES_16AM = []


for row in (
    NOTEBOOK_INDEX_16AM[
        "imports"
    ]
):

    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except Exception as exc:

        IMPORT_FAILURES_16AM.append(
            {
                "cell_idx":
                    row[
                        "cell_idx"
                    ],

                "source":
                    row[
                        "source"
                    ],

                "error":
                    (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
            }
        )


print(
    "Persisted imports attempted."
)

print(
    "Non-critical import failures:",
    len(
        IMPORT_FAILURES_16AM
    )
)


# ======================================================================
# 9. EXECUTE ALL PERSISTED CLASS DEFINITIONS
# ======================================================================

for class_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "classes"
    ].items()
):

    # Latest persisted definition before planner cell.
    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    try:

        exec(
            (
                "from __future__ import annotations\n"
                +
                row[
                    "source"
                ]
            ),
            planner_ns_16am
        )

    except Exception:

        # Not every notebook class belongs to planning.
        pass


# ======================================================================
# 10. EXECUTE ALL PERSISTED FUNCTION DEFINITIONS
# ======================================================================
#
# Definitions have no planning-loop side effect.
# ======================================================================

for function_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "functions"
    ].items()
):

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    try:

        exec(
            (
                "from __future__ import annotations\n"
                +
                row[
                    "source"
                ]
            ),
            planner_ns_16am
        )

    except Exception:

        pass


# ======================================================================
# 11. EXECUTE SAFE LITERAL ASSIGNMENTS
# ======================================================================

for target_name, rows in (
    NOTEBOOK_INDEX_16AM[
        "assignments"
    ].items()
):

    for row in sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    ):

        source = row[
            "source"
        ]


        try:

            tree = ast.parse(
                source
            )


            node = tree.body[
                0
            ]


            value_node = node.value


            ast.literal_eval(
                value_node
            )


            exec(
                source,
                planner_ns_16am
            )


        except Exception:

            pass


# ======================================================================
# 12. FORCE EXACT REQUIRED FUNCTION DEFINITIONS
# ======================================================================

REQUIRED_FUNCTIONS_16AM = [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
    "run_planning_checkpointed",
]


def recover_latest_function_16am(
    name
):

    rows = (
        NOTEBOOK_INDEX_16AM[
            "functions"
        ].get(
            name,
            []
        )
    )


    assert rows, (
        f"Could not recover persisted function: {name}"
    )


    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ]
    )[
        -1
    ]


    exec(
        (
            "from __future__ import annotations\n"
            +
            row[
                "source"
            ]
        ),
        planner_ns_16am
    )


    assert callable(
        planner_ns_16am[
            name
        ]
    )


    return row


RECOVERED_FUNCTION_ROWS_16AM = {}


for name in REQUIRED_FUNCTIONS_16AM:

    RECOVERED_FUNCTION_ROWS_16AM[
        name
    ] = recover_latest_function_16am(
        name
    )


print(
    "\nExact planner functions recovered:"
)


for name in REQUIRED_FUNCTIONS_16AM:

    row = (
        RECOVERED_FUNCTION_ROWS_16AM[
            name
        ]
    )


    print(
        f"  {name:<28} "
        f"cell={row['cell_idx']} "
        f"SHA={sha256_text_16am(row['source'])[:16]}"
    )


# ======================================================================
# 13. GENERIC ASSIGNMENT RECOVERY
# ======================================================================

def missing_name_from_error_16am(
    exc
):

    match = re.search(
        r"name '([^']+)' is not defined",
        str(
            exc
        )
    )


    if not match:

        return None


    return match.group(
        1
    )


RECOVERY_STACK_16AM = set()


def recover_assignment_target_16am(
    target_name,
    cutoff_cell=PLANNER_MAX_CELL_16AM,
    preferred_filter=None
):

    if target_name in (
        RECOVERY_STACK_16AM
    ):

        raise RuntimeError(
            f"Circular assignment recovery: "
            f"{target_name}"
        )


    rows = [
        row
        for row
        in NOTEBOOK_INDEX_16AM[
            "assignments"
        ].get(
            target_name,
            []
        )
        if row[
            "cell_idx"
        ]
        <=
        cutoff_cell
    ]


    if preferred_filter is not None:

        preferred = [
            row
            for row
            in rows
            if preferred_filter(
                row
            )
        ]


        if preferred:

            rows = preferred


    assert rows, (
        f"No persisted assignment found for "
        f"{target_name}"
    )


    # Latest matching assignment first.
    rows = sorted(
        rows,
        key=lambda x:
            x[
                "cell_idx"
            ],
        reverse=True
    )


    last_error = None


    RECOVERY_STACK_16AM.add(
        target_name
    )


    try:

        for row in rows:

            source = row[
                "source"
            ]


            for attempt in range(
                12
            ):

                try:

                    exec(
                        source,
                        planner_ns_16am
                    )


                    if target_name in (
                        planner_ns_16am
                    ):

                        return row


                    break


                except NameError as exc:

                    missing_name = (
                        missing_name_from_error_16am(
                            exc
                        )
                    )


                    if not missing_name:

                        last_error = exc
                        break


                    # Function / class definition?
                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "functions"
                        ]
                    ):

                        recover_latest_function_16am(
                            missing_name
                        )

                        continue


                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "classes"
                        ]
                    ):

                        class_row = sorted(
                            NOTEBOOK_INDEX_16AM[
                                "classes"
                            ][
                                missing_name
                            ],
                            key=lambda x:
                                x[
                                    "cell_idx"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            (
                                "from __future__ "
                                "import annotations\n"
                                +
                                class_row[
                                    "source"
                                ]
                            ),
                            planner_ns_16am
                        )

                        continue


                    # Recover exact persisted assignment.
                    if missing_name in (
                        NOTEBOOK_INDEX_16AM[
                            "assignments"
                        ]
                    ):

                        recover_assignment_target_16am(
                            missing_name,
                            cutoff_cell=
                                row[
                                    "cell_idx"
                                ]
                        )

                        continue


                    last_error = exc
                    break


                except Exception as exc:

                    last_error = exc
                    break


    finally:

        RECOVERY_STACK_16AM.remove(
            target_name
        )


    raise AssertionError(
        f"Failed to recover assignment for "
        f"{target_name}.\n"
        f"Last error: "
        f"{type(last_error).__name__ if last_error else 'unknown'}: "
        f"{last_error}"
    )


# ======================================================================
# 14. RECOVER EXACT INSTRUCTION
# ======================================================================

instruction_rows = (
    NOTEBOOK_INDEX_16AM[
        "assignments"
    ].get(
        "INSTRUCTION",
        []
    )
)


assert instruction_rows, (
    "No persisted INSTRUCTION assignment found."
)


INSTRUCTION_RECOVERY_ROW_16AM = None


for row in sorted(
    instruction_rows,
    key=lambda x:
        x[
            "cell_idx"
        ],
    reverse=True
):

    # Work in current exact namespace.
    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except NameError:

        try:

            recover_assignment_target_16am(
                "INSTRUCTION",
                cutoff_cell=
                    row[
                        "cell_idx"
                ]
            )

        except Exception:

            continue


    except Exception:

        continue


    if (
        "INSTRUCTION"
        not in
        planner_ns_16am
    ):

        continue


    value = planner_ns_16am[
        "INSTRUCTION"
    ]


    if not isinstance(
        value,
        str
    ):

        continue


    digest = sha256_text_16am(
        value
    )


    if digest.startswith(
        FROZEN_INSTRUCTION_SHA_PREFIX_16AM
    ):

        INSTRUCTION_RECOVERY_ROW_16AM = row

        break


assert (
    INSTRUCTION_RECOVERY_ROW_16AM
    is not None
), (
    "Exact frozen INSTRUCTION could not be recovered "
    "with expected SHA prefix."
)


INSTRUCTION_16AM = (
    planner_ns_16am[
        "INSTRUCTION"
    ]
)


INSTRUCTION_SHA_16AM = (
    sha256_text_16am(
        INSTRUCTION_16AM
    )
)


print(
    "\nExact frozen instruction: PASSED"
)

print(
    "  cell:",
    INSTRUCTION_RECOVERY_ROW_16AM[
        "cell_idx"
    ]
)

print(
    "  SHA256:",
    INSTRUCTION_SHA_16AM
)


# ======================================================================
# 15. RECOVER EXACT N_BEAM
# ======================================================================

N_BEAM_RECOVERY_ROW_16AM = (
    recover_assignment_target_16am(
        "N_BEAM"
    )
)


N_BEAM_16AM = int(
    planner_ns_16am[
        "N_BEAM"
    ]
)


assert (
    N_BEAM_16AM
    ==
    FROZEN_TOP_K_16AM
), (
    f"N_BEAM mismatch: "
    f"{N_BEAM_16AM} != {FROZEN_TOP_K_16AM}"
)


print(
    "\nExact N_BEAM: PASSED"
)

print(
    "  N_BEAM =",
    N_BEAM_16AM
)


# ======================================================================
# 16. RECOVER EXACT PROMPTER
# ======================================================================

PROMPTER_RECOVERY_ROW_16AM = (
    recover_assignment_target_16am(
        "prompter"
    )
)


prompter_16am = (
    planner_ns_16am[
        "prompter"
    ]
)


assert hasattr(
    prompter_16am,
    "format"
), (
    "Recovered prompter lacks format()."
)


# Verify prompt creation on frozen validation text.
validation_prompt_probe_16am = (
    prompter_16am.format(
        instruction=
            INSTRUCTION_16AM,

        message=
            str(
                webqsp_val_plan_rows[
                    0
                ][
                    "question"
                ]
            )
    )
)


assert isinstance(
    validation_prompt_probe_16am,
    str
)

assert len(
    validation_prompt_probe_16am
) > 0


PROMPT_PROBE_SHA_16AM = (
    sha256_text_16am(
        validation_prompt_probe_16am
    )
)


print(
    "\nExact prompter recovered: PASSED"
)

print(
    "  assignment cell:",
    PROMPTER_RECOVERY_ROW_16AM[
        "cell_idx"
    ]
)

print(
    "  validation prompt probe SHA:",
    PROMPT_PROBE_SHA_16AM
)


# ======================================================================
# 17. MODEL-ID EVIDENCE HELPER
# ======================================================================

def assignment_references_model_id_16am(
    row
):

    source = row[
        "source"
    ]


    if (
        FROZEN_MODEL_ID_16AM.lower()
        in
        source.lower()
    ):

        return True


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        return False


    names = {
        node.id
        for node in ast.walk(
            tree
        )
        if isinstance(
            node,
            ast.Name
        )
    }


    for name in names:

        if name not in (
            planner_ns_16am
        ):

            continue


        value = planner_ns_16am[
            name
        ]


        if (
            isinstance(
                value,
                str
            )
            and
            FROZEN_MODEL_ID_16AM.lower()
            in
            value.lower()
        ):

            return True


    return False


# ======================================================================
# 18. RUNTIME MODEL/TOKENIZER IDENTITY
# ======================================================================

def runtime_identity_strings_16am(
    obj
):

    values = []


    for attr in [
        "name_or_path",
        "_name_or_path",
    ]:

        value = getattr(
            obj,
            attr,
            None
        )


        if isinstance(
            value,
            str
        ):

            values.append(
                value
            )


    config = getattr(
        obj,
        "config",
        None
    )


    if config is not None:

        for attr in [
            "name_or_path",
            "_name_or_path",
        ]:

            value = getattr(
                config,
                attr,
                None
            )


            if isinstance(
                value,
                str
            ):

                values.append(
                    value
                )


    return list(
        dict.fromkeys(
            values
        )
    )


def runtime_matches_model_id_16am(
    obj
):

    return any(
        FROZEN_MODEL_ID_16AM.lower()
        in
        value.lower()

        for value in
        runtime_identity_strings_16am(
            obj
        )
    )


# ======================================================================
# 19. USE SURVIVING EXACT MODEL/TOKENIZER IF AVAILABLE
# ======================================================================

SURVIVING_MODEL_16AM = None
SURVIVING_TOKENIZER_16AM = None


# Preferred exact global names.
for name in [
    "model",
    "planner_model",
    "rog_model",
    "planning_model",
]:

    if name not in globals():

        continue


    obj = globals()[
        name
    ]


    if (
        hasattr(
            obj,
            "generate"
        )
        and
        runtime_matches_model_id_16am(
            obj
        )
    ):

        SURVIVING_MODEL_16AM = obj

        print(
            "\nUsing surviving exact planner model:",
            name
        )

        break


for name in [
    "tokenizer",
    "planner_tokenizer",
    "rog_tokenizer",
]:

    if name not in globals():

        continue


    obj = globals()[
        name
    ]


    if (
        callable(
            obj
        )
        and
        hasattr(
            obj,
            "decode"
        )
        and
        runtime_matches_model_id_16am(
            obj
        )
    ):

        SURVIVING_TOKENIZER_16AM = obj

        print(
            "Using surviving exact planner tokenizer:",
            name
        )

        break


# ======================================================================
# 20. RECOVER EXACT MODEL ASSIGNMENT IF NEEDED
# ======================================================================

MODEL_RECOVERY_ROW_16AM = None


if SURVIVING_MODEL_16AM is None:

    model_rows = (
        NOTEBOOK_INDEX_16AM[
            "assignments"
        ].get(
            "model",
            []
        )
    )


    model_evidence_rows = [
        row
        for row
        in model_rows
        if assignment_references_model_id_16am(
            row
        )
    ]


    assert model_evidence_rows, (
        "No unambiguous persisted planner-model assignment "
        "referencing rmanluo/RoG was found.\n"
        "STOP rather than loading a guessed model implementation."
    )


    MODEL_RECOVERY_ROW_16AM = (
        sorted(
            model_evidence_rows,
            key=lambda x:
                x[
                    "cell_idx"
                ],
            reverse=True
        )[
            0
        ]
    )


    # Exact source may assign model + tokenizer together.
    source = (
        MODEL_RECOVERY_ROW_16AM[
            "source"
        ]
    )


    print(
        "\nExecuting exact persisted model assignment "
        f"from cell "
        f"{MODEL_RECOVERY_ROW_16AM['cell_idx']}..."
    )


    # Recover missing assignment dependencies recursively.
    for attempt in range(
        20
    ):

        try:

            exec(
                source,
                planner_ns_16am
            )

            break


        except NameError as exc:

            missing_name = (
                missing_name_from_error_16am(
                    exc
                )
            )


            assert missing_name is not None


            if missing_name in (
                NOTEBOOK_INDEX_16AM[
                    "assignments"
                ]
            ):

                recover_assignment_target_16am(
                    missing_name,
                    cutoff_cell=
                        MODEL_RECOVERY_ROW_16AM[
                            "cell_idx"
                        ]
                )

                continue


            if missing_name in (
                NOTEBOOK_INDEX_16AM[
                    "functions"
                ]
            ):

                recover_latest_function_16am(
                    missing_name
                )

                continue


            raise


    assert (
        "model"
        in
        planner_ns_16am
    )


    SURVIVING_MODEL_16AM = (
        planner_ns_16am[
            "model"
        ]
    )


# ======================================================================
# 21. RECOVER EXACT TOKENIZER ASSIGNMENT IF NEEDED
# ======================================================================

TOKENIZER_RECOVERY_ROW_16AM = None


if SURVIVING_TOKENIZER_16AM is None:

    # It may already have been created by model tuple assignment.
    if (
        "tokenizer"
        in
        planner_ns_16am
        and
        hasattr(
            planner_ns_16am[
                "tokenizer"
            ],
            "decode"
        )
    ):

        SURVIVING_TOKENIZER_16AM = (
            planner_ns_16am[
                "tokenizer"
            ]
        )


    else:

        tokenizer_rows = (
            NOTEBOOK_INDEX_16AM[
                "assignments"
            ].get(
                "tokenizer",
                []
            )
        )


        tokenizer_evidence_rows = [
            row
            for row
            in tokenizer_rows
            if assignment_references_model_id_16am(
                row
            )
        ]


        assert tokenizer_evidence_rows, (
            "No unambiguous persisted tokenizer assignment "
            "referencing rmanluo/RoG was found."
        )


        TOKENIZER_RECOVERY_ROW_16AM = (
            sorted(
                tokenizer_evidence_rows,
                key=lambda x:
                    x[
                        "cell_idx"
                    ],
                reverse=True
            )[
                0
            ]
        )


        source = (
            TOKENIZER_RECOVERY_ROW_16AM[
                "source"
            ]
        )


        print(
            "Executing exact persisted tokenizer assignment "
            f"from cell "
            f"{TOKENIZER_RECOVERY_ROW_16AM['cell_idx']}..."
        )


        for attempt in range(
            20
        ):

            try:

                exec(
                    source,
                    planner_ns_16am
                )

                break


            except NameError as exc:

                missing_name = (
                    missing_name_from_error_16am(
                        exc
                    )
                )


                assert missing_name is not None


                if missing_name in (
                    NOTEBOOK_INDEX_16AM[
                        "assignments"
                    ]
                ):

                    recover_assignment_target_16am(
                        missing_name,
                        cutoff_cell=
                            TOKENIZER_RECOVERY_ROW_16AM[
                                "cell_idx"
                            ]
                    )

                    continue


                if missing_name in (
                    NOTEBOOK_INDEX_16AM[
                        "functions"
                    ]
                ):

                    recover_latest_function_16am(
                        missing_name
                    )

                    continue


                raise


        assert (
            "tokenizer"
            in
            planner_ns_16am
        )


        SURVIVING_TOKENIZER_16AM = (
            planner_ns_16am[
                "tokenizer"
            ]
        )


# ======================================================================
# 22. HARD MODEL / TOKENIZER SOFTWARE GATES
# ======================================================================

assert SURVIVING_MODEL_16AM is not None
assert SURVIVING_TOKENIZER_16AM is not None

assert hasattr(
    SURVIVING_MODEL_16AM,
    "generate"
)

assert hasattr(
    SURVIVING_TOKENIZER_16AM,
    "decode"
)


model_runtime_ids_16am = (
    runtime_identity_strings_16am(
        SURVIVING_MODEL_16AM
    )
)


tokenizer_runtime_ids_16am = (
    runtime_identity_strings_16am(
        SURVIVING_TOKENIZER_16AM
    )
)


model_assignment_evidence_16am = (
    MODEL_RECOVERY_ROW_16AM
    is not None
    and
    assignment_references_model_id_16am(
        MODEL_RECOVERY_ROW_16AM
    )
)


tokenizer_assignment_evidence_16am = (
    TOKENIZER_RECOVERY_ROW_16AM
    is not None
    and
    assignment_references_model_id_16am(
        TOKENIZER_RECOVERY_ROW_16AM
    )
)


model_identity_verified_16am = (
    runtime_matches_model_id_16am(
        SURVIVING_MODEL_16AM
    )
    or
    model_assignment_evidence_16am
)


tokenizer_identity_verified_16am = (
    runtime_matches_model_id_16am(
        SURVIVING_TOKENIZER_16AM
    )
    or
    tokenizer_assignment_evidence_16am
    or
    runtime_matches_model_id_16am(
        SURVIVING_MODEL_16AM
    )
)


assert model_identity_verified_16am, (
    "Planner model identity could not be tied "
    "to rmanluo/RoG."
)


assert tokenizer_identity_verified_16am, (
    "Planner tokenizer identity could not be tied "
    "to rmanluo/RoG."
)


print(
    "\nPlanner model/tokenizer identity: PASSED"
)

print(
    "  model runtime IDs:",
    model_runtime_ids_16am
)

print(
    "  tokenizer runtime IDs:",
    tokenizer_runtime_ids_16am
)


# ======================================================================
# 23. INJECT EXACT REQUIRED GLOBALS INTO RECOVERED WRAPPER NAMESPACE
# ======================================================================

planner_ns_16am[
    "model"
] = SURVIVING_MODEL_16AM

planner_ns_16am[
    "tokenizer"
] = SURVIVING_TOKENIZER_16AM

planner_ns_16am[
    "prompter"
] = prompter_16am

planner_ns_16am[
    "INSTRUCTION"
] = INSTRUCTION_16AM

planner_ns_16am[
    "N_BEAM"
] = N_BEAM_16AM

planner_ns_16am[
    "time"
] = time

planner_ns_16am[
    "json"
] = json

planner_ns_16am[
    "tqdm"
] = tqdm


# Re-execute exact required functions LAST so their __globals__
# point to the final exact planner namespace.
for name in REQUIRED_FUNCTIONS_16AM:

    row = (
        RECOVERED_FUNCTION_ROWS_16AM[
            name
        ]
    )


    exec(
        (
            "from __future__ import annotations\n"
            +
            row[
                "source"
            ]
        ),
        planner_ns_16am
    )


GENERATE_SEQ_16AM = (
    planner_ns_16am[
        "generate_seq"
    ]
)


LOAD_CHECKPOINT_16AM = (
    planner_ns_16am[
        "load_checkpoint"
    ]
)


PARSE_PREDICTION_16AM = (
    planner_ns_16am[
        "parse_prediction"
    ]
)


RUN_PLANNING_16AM = (
    planner_ns_16am[
        "run_planning_checkpointed"
    ]
)


print(
    "\nExact planner namespace injection: PASSED"
)


# ======================================================================
# 24. SIGNATURE + SOURCE SOFTWARE GATES
# ======================================================================

assert str(
    inspect.signature(
        RUN_PLANNING_16AM
    )
) == (
    "(dataset_split, ckpt_path, desc)"
)


assert str(
    inspect.signature(
        RUN_PLANNING_16AM
    )
) == str(
    RUN_PLANNING_SIGNATURE_16AR
)


print(
    "run_planning_checkpointed signature fidelity: PASSED"
)


# ======================================================================
# 25. DIRECT DEPENDENCY AVAILABILITY GATE
# ======================================================================

for name in [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
    "prompter",
    "INSTRUCTION",
    "model",
    "tokenizer",
    "N_BEAM",
    "time",
]:

    assert name in (
        RUN_PLANNING_16AM.__globals__
    ), (
        f"Missing planner global dependency: {name}"
    )


print(
    "Planner direct dependencies: PASSED"
)


# ======================================================================
# 26. GENERATE_SEQ CONFIGURATION SOURCE AUDIT
# ======================================================================

GENERATE_SEQ_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "generate_seq"
    ][
        "source"
    ]
)


PARSE_PREDICTION_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "parse_prediction"
    ][
        "source"
    ]
)


LOAD_CHECKPOINT_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "load_checkpoint"
    ][
        "source"
    ]
)


RUN_PLANNING_SOURCE_16AM = (
    RECOVERED_FUNCTION_ROWS_16AM[
        "run_planning_checkpointed"
    ][
        "source"
    ]
)


FUNCTION_SHA_16AM = {
    "generate_seq":
        sha256_text_16am(
            GENERATE_SEQ_SOURCE_16AM
        ),

    "parse_prediction":
        sha256_text_16am(
            PARSE_PREDICTION_SOURCE_16AM
        ),

    "load_checkpoint":
        sha256_text_16am(
            LOAD_CHECKPOINT_SOURCE_16AM
        ),

    "run_planning_checkpointed":
        sha256_text_16am(
            RUN_PLANNING_SOURCE_16AM
        ),
}


print(
    "\nExact planner function SHA256:"
)


for name, digest in (
    FUNCTION_SHA_16AM.items()
):

    print(
        f"  {name:<28} {digest}"
    )


# ======================================================================
# 27. STOCHASTIC-SEED AUDIT
# ======================================================================
#
# Wrapper explicitly uses do_sample=True.
#
# We therefore inspect seed calls in the recovered planning setup.
#
# IMPORTANT:
# We DO NOT invent a seed if the original notebook did not explicitly
# establish one in the planning setup.
# ======================================================================

setup_cells_16am = [
    INSTRUCTION_RECOVERY_ROW_16AM[
        "cell_idx"
    ],

    N_BEAM_RECOVERY_ROW_16AM[
        "cell_idx"
    ],

    PROMPTER_RECOVERY_ROW_16AM[
        "cell_idx"
    ],
]


if MODEL_RECOVERY_ROW_16AM is not None:

    setup_cells_16am.append(
        MODEL_RECOVERY_ROW_16AM[
            "cell_idx"
        ]
    )


if TOKENIZER_RECOVERY_ROW_16AM is not None:

    setup_cells_16am.append(
        TOKENIZER_RECOVERY_ROW_16AM[
            "cell_idx"
        ]
    )


PLANNER_SETUP_START_CELL_16AM = min(
    setup_cells_16am
)


PLANNER_SEED_CALLS_16AM = [
    row
    for row
    in NOTEBOOK_INDEX_16AM[
        "seed_calls"
    ]
    if (
        row[
            "cell_idx"
        ]
        >=
        PLANNER_SETUP_START_CELL_16AM
        and
        row[
            "cell_idx"
        ]
        <=
        PLANNER_MAX_CELL_16AM
    )
]


print(
    "\nRecovered explicit seed calls "
    "inside planner setup window:"
)


if PLANNER_SEED_CALLS_16AM:

    for row in (
        PLANNER_SEED_CALLS_16AM
    ):

        print(
            f"  cell {row['cell_idx']}: "
            f"{row['source']}"
        )


else:

    print(
        "  NONE"
    )


# Execute exactly recovered planner-window seed calls in notebook order.
for row in sorted(
    PLANNER_SEED_CALLS_16AM,
    key=lambda x:
        x[
            "cell_idx"
        ]
):

    try:

        exec(
            row[
                "source"
            ],
            planner_ns_16am
        )

    except Exception as exc:

        raise AssertionError(
            "Recovered explicit planner seed call "
            "could not be reproduced:\n"
            f"{row['source']}\n"
            f"{type(exc).__name__}: {exc}"
        )


if PLANNER_SEED_CALLS_16AM:

    print(
        "Exact recovered planner seed setup: APPLIED"
    )


else:

    print(
        "\nIMPORTANT REPRODUCIBILITY NOTE:"
    )

    print(
        "The persisted planning setup contains no explicit "
        "seed call in the recovered planner setup window."
    )

    print(
        "No new seed will be invented."
    )

    print(
        "The materialized TEST plan files themselves will "
        "be frozen by SHA256 after generation."
    )


# ======================================================================
# 28. FINAL PRE-GENERATION SAFETY GATES
# ======================================================================

assert len(
    webqsp_raw_test_16a
) == 1628


assert len(
    cwq_raw_test_16a
) == 3531


assert len(
    webqsp_official_test_16ar
) == 1628


assert len(
    cwq_official_test_16ar
) == 3531


# Exact wrapper uses only sample["question"] for planner input.
wrapper_compact_16am = (
    re.sub(
        r"\s+",
        "",
        RUN_PLANNING_SOURCE_16AM
    )
)


assert (
    'message=sample["question"]'
    in
    wrapper_compact_16am
), (
    "Could not verify that planner prompt uses "
    "question-only input."
)


# Ensure gold is NOT used in planner prompt/generation expression.
prompt_generation_prefix_16am = (
    RUN_PLANNING_SOURCE_16AM.split(
        "rec =",
        1
    )[
        0
    ]
)


assert (
    'sample["a_entity"]'
    not in
    prompt_generation_prefix_16am
)


assert (
    'sample["answer"]'
    not in
    prompt_generation_prefix_16am
)


print(
    "\nPRE-GENERATION SAFETY GATES: PASSED"
)

print(
    "  TEST gold used in planner prompt: NO"
)

print(
    "  TEST gold used in generation:     NO"
)

print(
    "  AFP changed:                      NO"
)

print(
    "  hyperparameter tuning:            NO"
)


# ======================================================================
# 29. CHECK EXISTING PARTIAL CHECKPOINTS
# ======================================================================

def checkpoint_count_16am(
    path
):

    if not path.exists():

        return 0


    count = 0


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                count += 1


    return count


web_existing_16am = (
    checkpoint_count_16am(
        WEBQSP_TEST_PLAN_PATH
    )
)


cwq_existing_16am = (
    checkpoint_count_16am(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nCheckpoint status before generation:"
)

print(
    "  WebQSP:",
    web_existing_16am,
    "/ 1628"
)

print(
    "  CWQ:   ",
    cwq_existing_16am,
    "/ 3531"
)


# ======================================================================
# 30. MATERIALIZE WEBQSP TEST PLANS
# ======================================================================
#
# Exact persisted wrapper.
#
# Checkpointed / resume-safe.
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "MATERIALIZING FROZEN WEBQSP TEST RELATION PLANS"
)

print(
    "=" * 124
)


webqsp_materialize_start_16am = (
    time.time()
)


webqsp_test_plan_rows = (
    RUN_PLANNING_16AM(
        dataset_split=
            webqsp_official_test_16ar,

        ckpt_path=
            str(
                WEBQSP_TEST_PLAN_PATH
            ),

        desc=
            "WebQSP frozen TEST planning"
    )
)


webqsp_materialize_elapsed_16am = (
    time.time()
    -
    webqsp_materialize_start_16am
)


print(
    "\nWebQSP planning elapsed:",
    f"{webqsp_materialize_elapsed_16am/60:.2f} min"
)


# ======================================================================
# 31. MATERIALIZE CWQ TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 124
)

print(
    "MATERIALIZING FROZEN CWQ TEST RELATION PLANS"
)

print(
    "=" * 124
)


cwq_materialize_start_16am = (
    time.time()
)


cwq_test_plan_rows = (
    RUN_PLANNING_16AM(
        dataset_split=
            cwq_official_test_16ar,

        ckpt_path=
            str(
                CWQ_TEST_PLAN_PATH
            ),

        desc=
            "CWQ frozen TEST planning"
    )
)


cwq_materialize_elapsed_16am = (
    time.time()
    -
    cwq_materialize_start_16am
)


print(
    "\nCWQ planning elapsed:",
    f"{cwq_materialize_elapsed_16am/60:.2f} min"
)


# ======================================================================
# 32. STRICT MATERIALIZED-PLAN AUDIT
# ======================================================================

REQUIRED_OUTPUT_FIELDS_16AM = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
    "planning_time_sec",
}


def audit_materialized_plans_16am(
    dataset_name,
    rows,
    raw_rows,
    expected_count
):

    assert len(
        rows
    ) == expected_count, (
        f"{dataset_name}: output row count "
        f"{len(rows)} != {expected_count}"
    )


    ids = [
        str(
            row[
                "id"
            ]
        )
        for row in rows
    ]


    assert len(
        ids
    ) == len(
        set(
            ids
        )
    ), (
        f"{dataset_name}: duplicate planning IDs."
    )


    raw_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row
        for row in raw_rows
    }


    assert set(
        ids
    ) == set(
        raw_by_id.keys()
    ), (
        f"{dataset_name}: materialized ID set "
        "does not equal raw TEST ID set."
    )


    empty_plan_records = 0

    total_predicted_paths = 0

    max_predicted_paths = 0


    for row in rows:

        assert (
            REQUIRED_OUTPUT_FIELDS_16AM.issubset(
                row.keys()
            )
        ), (
            f"{dataset_name}: missing planning fields "
            f"for id={row.get('id')}"
        )


        qid = str(
            row[
                "id"
            ]
        )


        raw = raw_by_id[
            qid
        ]


        assert str(
            row[
                "question"
            ]
        ) == str(
            raw[
                "question"
            ]
        )


        assert list(
            row[
                "q_entity"
            ]
        ) == list(
            raw[
                "q_entity"
            ]
        )


        assert list(
            row[
                "a_entity"
            ]
        ) == list(
            raw[
                "a_entity"
            ]
        )


        paths = row[
            "predicted_paths"
        ]


        assert isinstance(
            paths,
            list
        ), (
            f"{dataset_name}: predicted_paths "
            f"is not list for {qid}"
        )


        total_predicted_paths += len(
            paths
        )


        max_predicted_paths = max(
            max_predicted_paths,
            len(
                paths
            )
        )


        if len(
            paths
        ) == 0:

            empty_plan_records += 1


        # Frozen Top-K planning.
        assert len(
            paths
        ) <= FROZEN_TOP_K_16AM, (
            f"{dataset_name}: >Top-K plans "
            f"for id={qid}: {len(paths)}"
        )


        planning_time = float(
            row[
                "planning_time_sec"
            ]
        )


        assert (
            planning_time
            >=
            0.0
        )


    summary = {
        "questions":
            int(
                len(
                    rows
                )
            ),

        "empty_plan_questions":
            int(
                empty_plan_records
            ),

        "total_predicted_paths":
            int(
                total_predicted_paths
            ),

        "mean_predicted_paths_per_question":
            float(
                total_predicted_paths
                /
                len(
                    rows
                )
            ),

        "max_predicted_paths_per_question":
            int(
                max_predicted_paths
            ),
    }


    print(
        f"\n{dataset_name.upper()} materialized-plan audit: PASSED"
    )


    print(
        "  questions:",
        summary[
            "questions"
        ]
    )


    print(
        "  total plans:",
        summary[
            "total_predicted_paths"
        ]
    )


    print(
        "  mean plans/question:",
        f"{summary['mean_predicted_paths_per_question']:.6f}"
    )


    print(
        "  max plans/question:",
        summary[
            "max_predicted_paths_per_question"
        ]
    )


    print(
        "  empty-plan questions:",
        summary[
            "empty_plan_questions"
        ]
    )


    return summary


WEBQSP_TEST_PLAN_AUDIT_16AM = (
    audit_materialized_plans_16am(
        dataset_name=
            "webqsp",

        rows=
            webqsp_test_plan_rows,

        raw_rows=
            webqsp_raw_test_16a,

        expected_count=
            1628
    )
)


CWQ_TEST_PLAN_AUDIT_16AM = (
    audit_materialized_plans_16am(
        dataset_name=
            "cwq",

        rows=
            cwq_test_plan_rows,

        raw_rows=
            cwq_raw_test_16a,

        expected_count=
            3531
    )
)


# ======================================================================
# 33. VERIFY FILE-LEVEL CHECKPOINT CONTENT
# ======================================================================

assert WEBQSP_TEST_PLAN_PATH.exists()
assert CWQ_TEST_PLAN_PATH.exists()


assert (
    checkpoint_count_16am(
        WEBQSP_TEST_PLAN_PATH
    )
    ==
    1628
)


assert (
    checkpoint_count_16am(
        CWQ_TEST_PLAN_PATH
    )
    ==
    3531
)


WEBQSP_TEST_PLAN_SHA = (
    sha256_file_16am(
        WEBQSP_TEST_PLAN_PATH
    )
)


CWQ_TEST_PLAN_SHA = (
    sha256_file_16am(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nFrozen TEST planning artifact SHA256:"
)

print(
    "  WebQSP:",
    WEBQSP_TEST_PLAN_SHA
)

print(
    "  CWQ:   ",
    CWQ_TEST_PLAN_SHA
)


# ======================================================================
# 34. VALIDATION/TEST DISJOINTNESS AGAIN AFTER MATERIALIZATION
# ======================================================================

web_val_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    webqsp_val_plan_rows
}


cwq_val_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    cwq_val_plan_rows
}


web_test_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    webqsp_test_plan_rows
}


cwq_test_ids_16am = {
    str(
        row[
            "id"
        ]
    )
    for row in
    cwq_test_plan_rows
}


assert not (
    web_val_ids_16am
    &
    web_test_ids_16am
)


assert not (
    cwq_val_ids_16am
    &
    cwq_test_ids_16am
)


print(
    "\nFinal validation / TEST disjointness: PASSED"
)


# ======================================================================
# 35. PLANNER CONFIG FINGERPRINT
# ======================================================================
#
# This does NOT replace the earlier frozen planner config hash.
# It records the exact implementation used to generate TEST artifacts.
# ======================================================================

TEST_PLANNER_IMPLEMENTATION_PAYLOAD_16AM = {
    "model_id":
        FROZEN_MODEL_ID_16AM,

    "instruction_sha256":
        INSTRUCTION_SHA_16AM,

    "N_BEAM":
        N_BEAM_16AM,

    "do_sample":
        True,

    "max_new_tokens":
        100,

    "prompter_assignment_cell":
        int(
            PROMPTER_RECOVERY_ROW_16AM[
                "cell_idx"
            ]
        ),

    "function_sha256":
        FUNCTION_SHA_16AM,

    "seed_calls":
        [
            {
                "cell_idx":
                    int(
                        row[
                            "cell_idx"
                        ]
                    ),

                "source":
                    row[
                        "source"
                    ],
            }

            for row in
            PLANNER_SEED_CALLS_16AM
        ],
}


TEST_PLANNER_IMPLEMENTATION_JSON_16AM = (
    json.dumps(
        TEST_PLANNER_IMPLEMENTATION_PAYLOAD_16AM,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False
    )
)


TEST_PLANNER_IMPLEMENTATION_SHA_16AM = (
    sha256_text_16am(
        TEST_PLANNER_IMPLEMENTATION_JSON_16AM
    )
)


print(
    "\nTEST planner implementation fingerprint:"
)

print(
    " ",
    TEST_PLANNER_IMPLEMENTATION_SHA_16AM
)


# ======================================================================
# 36. SAVE MATERIALIZATION MANIFEST
# ======================================================================

CELL16A_MATERIALIZE_MANIFEST = {
    "cell":
        "CELL16A_MATERIALIZE_EXACT_FROZEN_ROG_TEST_PLANS",

    "afp_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "development_planner_identity": {
        "model_id":
            FROZEN_MODEL_ID_16AM,

        "top_k":
            FROZEN_TOP_K_16AM,

        "instruction_sha256":
            INSTRUCTION_SHA_16AM,

        "instruction_expected_sha_prefix":
            FROZEN_INSTRUCTION_SHA_PREFIX_16AM,

        "prior_planner_config_sha_prefix":
            FROZEN_PLANNER_CONFIG_SHA_PREFIX_16AM,
    },

    "generation_configuration": {
        "N_BEAM":
            N_BEAM_16AM,

        "do_sample":
            True,

        "max_new_tokens":
            100,

        "generation_order": [
            "webqsp",
            "cwq",
        ],

        "explicit_recovered_seed_calls":
            [
                {
                    "cell_idx":
                        int(
                            row[
                                "cell_idx"
                            ]
                        ),

                    "source":
                        row[
                            "source"
                        ],
                }

                for row in
                PLANNER_SEED_CALLS_16AM
            ],

        "new_seed_invented":
            False,
    },

    "software_identity": {
        "planner_function_signature":
            str(
                inspect.signature(
                    RUN_PLANNING_16AM
                )
            ),

        "function_sha256":
            FUNCTION_SHA_16AM,

        "test_planner_implementation_sha256":
            TEST_PLANNER_IMPLEMENTATION_SHA_16AM,

        "model_runtime_identity":
            model_runtime_ids_16am,

        "tokenizer_runtime_identity":
            tokenizer_runtime_ids_16am,

        "model_identity_verified":
            bool(
                model_identity_verified_16am
            ),

        "tokenizer_identity_verified":
            bool(
                tokenizer_identity_verified_16am
            ),

        "prompt_probe_sha256":
            PROMPT_PROBE_SHA_16AM,
    },

    "test_input": {
        "webqsp_questions":
            1628,

        "cwq_questions":
            3531,

        "official_source_validation_identity_gate":
            True,

        "validation_test_disjoint":
            True,
    },

    "output": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TEST_PLAN_PATH
                ),

            "sha256":
                WEBQSP_TEST_PLAN_SHA,

            **WEBQSP_TEST_PLAN_AUDIT_16AM,
        },

        "cwq": {
            "path":
                str(
                    CWQ_TEST_PLAN_PATH
                ),

            "sha256":
                CWQ_TEST_PLAN_SHA,

            **CWQ_TEST_PLAN_AUDIT_16AM,
        },
    },

    "leakage": {
        "test_gold_used_in_planner_prompt":
            False,

        "test_gold_used_in_generation":
            False,

        "test_used_for_parameter_selection":
            False,

        "test_used_for_hyperparameter_tuning":
            False,

        "test_triggered_method_change":
            False,
    },

    "afp_changed_after_freeze":
        False,

    "next":
        "rerun_Cell16_unchanged",
}


with open(
    MATERIALIZE_MANIFEST_PATH_16AM,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CELL16A_MATERIALIZE_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16A_MATERIALIZE_COMPLETE = True

FROZEN_TEST_PLANS_MATERIALIZED = True


# ======================================================================
# 37. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 128
)

print(
    "=== CELL 16A-MATERIALIZE: "
    "FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ==="
)

print(
    "=" * 128
)


print(
    "\nPlanner identity:"
)

print(
    "  model:",
    FROZEN_MODEL_ID_16AM
)

print(
    "  instruction SHA:",
    INSTRUCTION_SHA_16AM
)

print(
    "  N_BEAM:",
    N_BEAM_16AM
)

print(
    "  do_sample:",
    True
)

print(
    "  max_new_tokens:",
    100
)


print(
    "\nTEST plans:"
)

print(
    "  WebQSP:",
    len(
        webqsp_test_plan_rows
    ),
    "questions"
)

print(
    "    file:",
    WEBQSP_TEST_PLAN_PATH
)

print(
    "    SHA:",
    WEBQSP_TEST_PLAN_SHA
)


print(
    "  CWQ:",
    len(
        cwq_test_plan_rows
    ),
    "questions"
)

print(
    "    file:",
    CWQ_TEST_PLAN_PATH
)

print(
    "    SHA:",
    CWQ_TEST_PLAN_SHA
)


print(
    "\nLeakage / freeze status:"
)

print(
    "  TEST gold in planner prompt: NO"
)

print(
    "  TEST tuning:                 NO"
)

print(
    "  TEST-triggered changes:      NO"
)

print(
    "  AFP changed after freeze:    NO"
)


print(
    "\nMaterialization manifest:"
)

print(
    " ",
    MATERIALIZE_MANIFEST_PATH_16AM
)


print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16 UNCHANGED."
)

print(
    "Do NOT move directly to Cell 17."
)

Cell 16A-MATERIALIZE freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen wrapper constants:
  model: rmanluo/RoG
  beam: 3
  do_sample: True
  max_new_tokens: 100

Persisted notebook AST index: READY
Persisted imports attempted.
Non-critical import failures: 9


AssertionError: Could not recover persisted function: generate_seq

In [38]:
# ======================================================================
# CELL 16A-D
# EXACT PLANNER DEPENDENCY ORIGIN RECOVERY
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT restart kernel.
# DO NOT rerun Cell 16 yet.
#
# PURPOSE
# -------
# Determine EXACTLY where these persisted planner dependencies came from:
#
#   generate_seq
#   load_checkpoint
#   parse_prediction
#
# They may be:
#   - imported with `from module import ...`
#   - imported with `from module import *`
#   - assigned/aliased
#   - defined in another local .py module
#   - defined in notebook code not represented as top-level FunctionDef
#
# NO TEST generation occurs here.
# ======================================================================

import ast
import importlib
import importlib.util
import inspect
import json
import os
import re
import sys
from pathlib import Path


# ======================================================================
# 1. HARD GATE
# ======================================================================

required = [
    "CELL16AR_COMPLETE",
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",
    "persisted_nb_16ar",
]

missing = [
    x for x in required
    if x not in globals()
]

assert not missing, (
    "Missing prerequisites:\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("Cell 16A-D freeze gate: PASSED")


# ======================================================================
# 2. TARGET SYMBOLS
# ======================================================================

TARGETS_16AD = [
    "generate_seq",
    "load_checkpoint",
    "parse_prediction",
]


# ======================================================================
# 3. COLLECT NOTEBOOK OCCURRENCES
# ======================================================================

occurrences_16ad = {
    name: []
    for name in TARGETS_16AD
}

sys_path_statements_16ad = []


for cell_idx, cell in enumerate(
    persisted_nb_16ar["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    # Only relevant pre-planner history.
    if cell_idx > 139:
        continue

    # --------------------------------------------------------------
    # Recover sys.path mutations because imported helper modules may
    # depend on them.
    # --------------------------------------------------------------

    for line in source.splitlines():

        stripped = line.strip()

        if (
            stripped.startswith("sys.path.append(")
            or
            stripped.startswith("sys.path.insert(")
        ):
            sys_path_statements_16ad.append(
                {
                    "cell_idx": cell_idx,
                    "source": stripped,
                }
            )

    # --------------------------------------------------------------
    # Any textual occurrence of target symbols.
    # --------------------------------------------------------------

    for name in TARGETS_16AD:

        if name not in source:
            continue

        info = {
            "cell_idx": cell_idx,
            "full_source": source,
            "definitions": [],
            "imports": [],
            "assignments": [],
        }

        try:
            tree = ast.parse(source)

        except Exception:
            occurrences_16ad[name].append(info)
            continue

        for node in tree.body:

            segment = ast.get_source_segment(
                source,
                node
            )

            if not segment:
                continue

            # Direct function definition.
            if (
                isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    )
                )
                and
                node.name == name
            ):
                info["definitions"].append(
                    segment
                )

            # from module import symbol
            elif isinstance(
                node,
                ast.ImportFrom
            ):

                imported_names = [
                    alias.name
                    for alias in node.names
                ]

                aliases = [
                    alias.asname
                    for alias in node.names
                ]

                if (
                    name in imported_names
                    or
                    name in aliases
                    or
                    "*" in imported_names
                ):
                    info["imports"].append(
                        segment
                    )

            # import module
            elif isinstance(
                node,
                ast.Import
            ):

                if name in segment:
                    info["imports"].append(
                        segment
                    )

            # Assignment / alias.
            elif isinstance(
                node,
                (
                    ast.Assign,
                    ast.AnnAssign,
                )
            ):

                if name in segment:
                    info["assignments"].append(
                        segment
                    )

        occurrences_16ad[name].append(info)


# ======================================================================
# 4. PRINT NOTEBOOK ORIGIN EVIDENCE
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "NOTEBOOK SYMBOL ORIGIN EVIDENCE"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    print(
        f"\n### {name}"
    )

    rows = occurrences_16ad[name]

    print(
        "occurrence cells:",
        [
            row["cell_idx"]
            for row in rows
        ]
    )

    for row in rows:

        meaningful = (
            row["definitions"]
            or
            row["imports"]
            or
            row["assignments"]
        )

        if not meaningful:
            continue

        print(
            f"\n--- cell {row['cell_idx']} ---"
        )

        for src in row["definitions"]:
            print(
                "[DEFINITION]"
            )
            print(src)

        for src in row["imports"]:
            print(
                "[IMPORT]"
            )
            print(src)

        for src in row["assignments"]:
            print(
                "[ASSIGNMENT]"
            )
            print(src)


# ======================================================================
# 5. REPLAY ONLY EXACT sys.path MUTATIONS
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "RECOVERED sys.path MUTATIONS"
)

print(
    "=" * 120
)


for row in sys_path_statements_16ad:

    print(
        f"cell {row['cell_idx']}: "
        f"{row['source']}"
    )


for row in sorted(
    sys_path_statements_16ad,
    key=lambda x: x["cell_idx"]
):

    try:

        exec(
            row["source"],
            {
                "sys": sys,
                "os": os,
                "Path": Path,
            }
        )

    except Exception as exc:

        print(
            "Could not replay:",
            row["source"]
        )

        print(
            type(exc).__name__,
            exc
        )


# ======================================================================
# 6. ATTEMPT EXACT NOTEBOOK IMPORTS
# ======================================================================

resolved_16ad = {}

import_attempts_16ad = []


for name in TARGETS_16AD:

    # --------------------------------------------------------------
    # First: maybe function already exists in live global namespace.
    # --------------------------------------------------------------

    live = globals().get(
        name
    )

    if callable(live):

        resolved_16ad[name] = {
            "object": live,
            "origin":
                "surviving_notebook_global",
            "source":
                (
                    inspect.getsource(live)
                    if inspect.isfunction(live)
                    else None
                ),
        }

        continue

    # --------------------------------------------------------------
    # Replay exact persisted import statements that mention the target.
    # --------------------------------------------------------------

    candidate_imports = []

    for row in occurrences_16ad[name]:

        candidate_imports.extend(
            row["imports"]
        )

    candidate_imports = list(
        dict.fromkeys(
            candidate_imports
        )
    )

    for statement in candidate_imports:

        local_ns = {}

        try:

            exec(
                statement,
                globals(),
                local_ns
            )

            obj = (
                local_ns.get(name)
                or
                globals().get(name)
            )

            # Star imports may place it in locals.
            if callable(obj):

                resolved_16ad[name] = {
                    "object": obj,
                    "origin":
                        f"exact_import: {statement}",
                    "source":
                        (
                            inspect.getsource(obj)
                            if inspect.isfunction(obj)
                            else None
                        ),
                }

                import_attempts_16ad.append(
                    {
                        "symbol": name,
                        "statement": statement,
                        "status": "RESOLVED",
                    }
                )

                break

            import_attempts_16ad.append(
                {
                    "symbol": name,
                    "statement": statement,
                    "status":
                        "import succeeded but symbol absent",
                }
            )

        except Exception as exc:

            import_attempts_16ad.append(
                {
                    "symbol": name,
                    "statement": statement,
                    "status":
                        (
                            f"{type(exc).__name__}: "
                            f"{exc}"
                        ),
                }
            )


# ======================================================================
# 7. SEARCH LOCAL PYTHON SOURCES
# ======================================================================
#
# Only searches code-sized roots, not dataset parquet content.
# ======================================================================

SEARCH_ROOTS_16AD = [
    Path.cwd(),
    Path(
        "/kaggle/working"
    ),
    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap"
    ),
]


file_hits_16ad = {
    name: []
    for name in TARGETS_16AD
}


seen_files_16ad = set()


for root in SEARCH_ROOTS_16AD:

    if not root.exists():
        continue

    try:
        py_files = list(
            root.rglob("*.py")
        )

    except Exception:
        continue

    for path in py_files:

        path_key = str(
            path.resolve()
        )

        if path_key in seen_files_16ad:
            continue

        seen_files_16ad.add(
            path_key
        )

        # Avoid giant accidental source files.
        try:

            if path.stat().st_size > 5_000_000:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception:
            continue

        for name in TARGETS_16AD:

            patterns = [
                rf"\bdef\s+{re.escape(name)}\s*\(",
                rf"\b{name}\s*=",
                rf"\bimport\s+{re.escape(name)}\b",
                rf"\bimport\s+.*\b{re.escape(name)}\b",
            ]

            if any(
                re.search(pattern, text)
                for pattern in patterns
            ):

                file_hits_16ad[name].append(
                    path
                )


# ======================================================================
# 8. PRINT FILE SEARCH
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "LOCAL PYTHON SOURCE HITS"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    print(
        f"\n{name}:"
    )

    if not file_hits_16ad[name]:

        print(
            "  NONE"
        )

    else:

        for path in file_hits_16ad[name]:

            print(
                " ",
                path
            )


# ======================================================================
# 9. INSPECT IMPORT ATTEMPTS
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "EXACT IMPORT ATTEMPTS"
)

print(
    "=" * 120
)


if not import_attempts_16ad:

    print(
        "No direct import statements found."
    )


for row in import_attempts_16ad:

    print(
        f"\n{row['symbol']}"
    )

    print(
        " statement:",
        row["statement"]
    )

    print(
        " status:",
        row["status"]
    )


# ======================================================================
# 10. RESOLVED FUNCTION REPORT
# ======================================================================

print(
    "\n"
    + "=" * 120
)

print(
    "RESOLVED EXACT PLANNER DEPENDENCIES"
)

print(
    "=" * 120
)


for name in TARGETS_16AD:

    if name not in resolved_16ad:

        print(
            f"\n{name}: NOT YET RESOLVED"
        )

        continue

    item = resolved_16ad[name]

    obj = item["object"]

    print(
        f"\n{name}: RESOLVED"
    )

    print(
        " origin:",
        item["origin"]
    )

    try:

        print(
            " module:",
            obj.__module__
        )

    except Exception:
        pass

    try:

        print(
            " signature:",
            inspect.signature(
                obj
            )
        )

    except Exception:
        pass

    try:

        src = inspect.getsource(
            obj
        )

        print(
            "\nSOURCE:"
        )

        print(src)

    except Exception as exc:

        print(
            " source unavailable:",
            type(exc).__name__,
            exc
        )


# ======================================================================
# 11. SAVE DIAGNOSTIC REPORT
# ======================================================================

OUT_DIR_16AD = Path(
    "/kaggle/working/"
    "step2_rq1_test/"
    "planner_recovery"
)

OUT_DIR_16AD.mkdir(
    parents=True,
    exist_ok=True
)


REPORT_PATH_16AD = (
    OUT_DIR_16AD
    / "cell16ad_dependency_origin_report.json"
)


report_16ad = {
    "freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "targets":
        TARGETS_16AD,

    "resolved": {
        name:
            {
                "resolved":
                    name in resolved_16ad,

                "origin":
                    (
                        resolved_16ad[
                            name
                        ][
                            "origin"
                        ]
                        if name in resolved_16ad
                        else None
                    ),

                "module":
                    (
                        getattr(
                            resolved_16ad[
                                name
                            ][
                                "object"
                            ],
                            "__module__",
                            None
                        )
                        if name in resolved_16ad
                        else None
                    ),
            }

        for name in TARGETS_16AD
    },

    "notebook_occurrence_cells": {
        name:
            [
                row["cell_idx"]
                for row in
                occurrences_16ad[name]
            ]

        for name in TARGETS_16AD
    },

    "file_hits": {
        name:
            [
                str(path)
                for path in
                file_hits_16ad[name]
            ]

        for name in TARGETS_16AD
    },

    "import_attempts":
        import_attempts_16ad,

    "test_planning_started":
        False,

    "afp_changed":
        False,
}


with open(
    REPORT_PATH_16AD,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report_16ad,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AD_COMPLETE = True

RESOLVED_PLANNER_DEPENDENCIES_16AD = (
    resolved_16ad
)


print(
    "\n"
    + "=" * 120
)

print(
    "=== CELL 16A-D COMPLETE ==="
)

print(
    "=" * 120
)

print(
    "\nTEST planning started: NO"
)

print(
    "AFP changed: NO"
)

print(
    "\nResolved:",
    {
        name:
            name in resolved_16ad
        for name in TARGETS_16AD
    }
)

print(
    "\nReport:"
)

print(
    " ",
    REPORT_PATH_16AD
)

Cell 16A-D freeze gate: PASSED

NOTEBOOK SYMBOL ORIGIN EVIDENCE

### generate_seq
occurrence cells: [19, 21, 47, 83, 105, 139]

--- cell 19 ---
[IMPORT]
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

### load_checkpoint
occurrence cells: [47, 57, 105, 115, 139]

--- cell 47 ---
[DEFINITION]
def load_checkpoint(path):
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done[rec["id"]] = rec
    return done
[ASSIGNMENT]
planning_done = load_checkpoint(PLANNING_CKPT)

--- cell 57 ---
[ASSIGNMENT]
reasoning_done = load_checkpoint(REASON_CKPT)

--- cell 105 ---
[ASSIGNMENT]
cwq_planning_done = load_checkpoint(CWQ_PLANNING_CKPT)

### parse_prediction
occurrence cells: [19, 21, 41, 47, 83, 99, 105, 139]

--- cell 19 ---
[IMPORT]
from qa_prediction.gen_rule_path import generate_seq, parse_prediction, INSTRUCTION

R

In [4]:
# ======================================================================
# CELL 16A-M3
# EXACT FROZEN RoG TEST PLAN MATERIALIZATION
# DIRECT-SOURCE RECOVERY — NO utils PACKAGE IMPORT
# ======================================================================
#
# RUN AS A NEW CELL.
#
# DO NOT:
#   - rerun M2/M2R
#   - restart kernel
#   - run Cell 16 yet
#   - install/upgrade RoG dependencies
#   - change AFP
#
# WHY THIS VERSION
# ----------------
# RoG's gen_rule_path.py imports the full "utils" package.
# That indirectly imports graph_utils -> walker.
#
# But our frozen planning wrapper uses only:
#
#   generate_seq
#   parse_prediction
#   INSTRUCTION
#
# Those symbols do NOT need graph-walker.
#
# Therefore this cell extracts their EXACT source definitions directly
# from the SHA-verified local gen_rule_path.py instead of importing the
# whole module.
#
# It also extracts the exact InstructFormater implementation directly
# from src/utils/utils.py without importing src/utils/__init__.py.
#
# NO source code is modified.
# ======================================================================


# ======================================================================
# 0. IMPORTS
# ======================================================================

import ast
import hashlib
import inspect
import json
import math
import os
import random
import re
import sys
import time
import types
from pathlib import Path

import numpy as np
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

try:
    from peft import (
        AutoPeftModelForCausalLM,
    )
except Exception:
    AutoPeftModelForCausalLM = None


# ======================================================================
# 1. HARD GATES
# ======================================================================

required = [
    "CELL16AR_COMPLETE",
    "CELL16AD_COMPLETE",

    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "persisted_nb_16ar",
    "run_planning_source_16ar",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",
]

missing = [
    x
    for x in required
    if x not in globals()
]

assert not missing, (
    "Missing prerequisite object(s):\n  "
    + "\n  ".join(missing)
)

assert CELL16AR_COMPLETE is True
assert CELL16AD_COMPLETE is True
assert AFP_DEVELOPMENT_FROZEN is True


EXPECTED_FREEZE_SHA_16AM3 = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    EXPECTED_FREEZE_SHA_16AM3
)


print(
    "Cell 16A-M3 freeze gate: PASSED"
)

print(
    "Freeze SHA:",
    FINAL_AFP_FREEZE_SHA256
)


# ======================================================================
# 2. FROZEN PLANNER FACTS
# ======================================================================

FROZEN_MODEL_ID_16AM3 = (
    "rmanluo/RoG"
)

FROZEN_N_BEAM_16AM3 = 3

FROZEN_INSTRUCTION_SHA_PREFIX_16AM3 = (
    "e3687b4a5081c22c"
)


assert (
    "do_sample=True"
    in
    run_planning_source_16ar
)

assert (
    "max_new_tokens=100"
    in
    run_planning_source_16ar
)

assert (
    "num_beam=N_BEAM"
    in
    run_planning_source_16ar
)


print(
    "\nFrozen planner:"
)

print(
    " model:",
    FROZEN_MODEL_ID_16AM3
)

print(
    " N_BEAM:",
    FROZEN_N_BEAM_16AM3
)

print(
    " do_sample:",
    True
)

print(
    " max_new_tokens:",
    100
)


# ======================================================================
# 3. CUDA GATE
# ======================================================================
#
# Exact persisted generate_seq uses:
#
#     .to("cuda")
#
# Therefore CPU substitution would change the implementation.
# ======================================================================

assert torch.cuda.is_available(), (
    "CUDA is unavailable.\n"
    "Exact recovered generate_seq hardcodes .to('cuda').\n"
    "Enable a Kaggle GPU before materializing TEST plans."
)


print(
    "\nCUDA gate: PASSED"
)

print(
    " GPU:",
    torch.cuda.get_device_name(0)
)


# ======================================================================
# 4. SHA HELPERS
# ======================================================================

def sha256_file_16am3(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_text_16am3(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


# ======================================================================
# 5. EXACT SOURCE FILES
# ======================================================================

ROG_REPO_16AM3 = Path(
    "/kaggle/working/"
    "reasoning-on-graphs"
)


GEN_RULE_PATH_16AM3 = (
    ROG_REPO_16AM3
    / "src"
    / "qa_prediction"
    / "gen_rule_path.py"
)


UTILS_UTILS_PATH_16AM3 = (
    ROG_REPO_16AM3
    / "src"
    / "utils"
    / "utils.py"
)


assert GEN_RULE_PATH_16AM3.exists()
assert UTILS_UTILS_PATH_16AM3.exists()


EXPECTED_GEN_RULE_SHA_16AM3 = (
    "e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af"
)


ACTUAL_GEN_RULE_SHA_16AM3 = (
    sha256_file_16am3(
        GEN_RULE_PATH_16AM3
    )
)


assert (
    ACTUAL_GEN_RULE_SHA_16AM3
    ==
    EXPECTED_GEN_RULE_SHA_16AM3
)


print(
    "\nExact gen_rule_path.py SHA gate: PASSED"
)

print(
    " ",
    ACTUAL_GEN_RULE_SHA_16AM3
)


# ======================================================================
# 6. READ EXACT gen_rule_path.py AS TEXT
# ======================================================================
#
# IMPORTANT:
# We DO NOT import the module.
#
# Therefore:
#
#   utils
#   graph_utils
#   walker
#
# are never imported.
# ======================================================================

GEN_RULE_SOURCE_16AM3 = (
    GEN_RULE_PATH_16AM3.read_text(
        encoding="utf-8"
    )
)


GEN_RULE_AST_16AM3 = ast.parse(
    GEN_RULE_SOURCE_16AM3
)


# ======================================================================
# 7. EXTRACT EXACT CONSTANTS + FUNCTIONS
# ======================================================================

INSTRUCTION_NODE_16AM3 = None
PATH_RE_NODE_16AM3 = None

GENERATE_SEQ_NODE_16AM3 = None
PARSE_PREDICTION_NODE_16AM3 = None


for node in GEN_RULE_AST_16AM3.body:

    # --------------------------------------------------------------
    # Constants
    # --------------------------------------------------------------

    if isinstance(
        node,
        ast.Assign
    ):

        target_names = [
            target.id
            for target in node.targets
            if isinstance(
                target,
                ast.Name
            )
        ]


        if "INSTRUCTION" in target_names:

            INSTRUCTION_NODE_16AM3 = node


        if "PATH_RE" in target_names:

            PATH_RE_NODE_16AM3 = node


    # --------------------------------------------------------------
    # Functions
    # --------------------------------------------------------------

    elif isinstance(
        node,
        ast.FunctionDef
    ):

        if node.name == "generate_seq":

            GENERATE_SEQ_NODE_16AM3 = node


        elif node.name == "parse_prediction":

            PARSE_PREDICTION_NODE_16AM3 = node


assert INSTRUCTION_NODE_16AM3 is not None
assert PATH_RE_NODE_16AM3 is not None
assert GENERATE_SEQ_NODE_16AM3 is not None
assert PARSE_PREDICTION_NODE_16AM3 is not None


INSTRUCTION_16AM3 = ast.literal_eval(
    INSTRUCTION_NODE_16AM3.value
)


PATH_RE_16AM3 = ast.literal_eval(
    PATH_RE_NODE_16AM3.value
)


GENERATE_SEQ_SOURCE_16AM3 = (
    ast.get_source_segment(
        GEN_RULE_SOURCE_16AM3,
        GENERATE_SEQ_NODE_16AM3
    )
)


PARSE_PREDICTION_SOURCE_16AM3 = (
    ast.get_source_segment(
        GEN_RULE_SOURCE_16AM3,
        PARSE_PREDICTION_NODE_16AM3
    )
)


assert GENERATE_SEQ_SOURCE_16AM3
assert PARSE_PREDICTION_SOURCE_16AM3


# ======================================================================
# 8. INSTRUCTION FIDELITY
# ======================================================================

INSTRUCTION_SHA_16AM3 = (
    sha256_text_16am3(
        INSTRUCTION_16AM3
    )
)


assert INSTRUCTION_SHA_16AM3.startswith(
    FROZEN_INSTRUCTION_SHA_PREFIX_16AM3
), (
    "Exact local INSTRUCTION does not match "
    "the development-frozen SHA."
)


print(
    "\nInstruction fidelity: PASSED"
)

print(
    " SHA:",
    INSTRUCTION_SHA_16AM3
)


# ======================================================================
# 9. EXECUTE ONLY EXACT REQUIRED PLANNER HELPERS
# ======================================================================

helper_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "torch":
        torch,

    "re":
        re,

    "PATH_RE":
        PATH_RE_16AM3,
}


exec(
    GENERATE_SEQ_SOURCE_16AM3,
    helper_ns_16am3
)


exec(
    PARSE_PREDICTION_SOURCE_16AM3,
    helper_ns_16am3
)


generate_seq_16am3 = (
    helper_ns_16am3[
        "generate_seq"
    ]
)


parse_prediction_16am3 = (
    helper_ns_16am3[
        "parse_prediction"
    ]
)


assert callable(
    generate_seq_16am3
)

assert callable(
    parse_prediction_16am3
)


print(
    "\nExact planner helpers reconstructed: PASSED"
)

print(
    " generate_seq:",
    inspect.signature(
        generate_seq_16am3
    )
)

print(
    " parse_prediction:",
    inspect.signature(
        parse_prediction_16am3
    )
)


print(
    " generate_seq SHA:",
    sha256_text_16am3(
        GENERATE_SEQ_SOURCE_16AM3
    )
)

print(
    " parse_prediction SHA:",
    sha256_text_16am3(
        PARSE_PREDICTION_SOURCE_16AM3
    )
)


# ======================================================================
# 10. EXTRACT EXACT InstructFormater WITHOUT IMPORTING utils PACKAGE
# ======================================================================

UTILS_SOURCE_16AM3 = (
    UTILS_UTILS_PATH_16AM3.read_text(
        encoding="utf-8"
    )
)


UTILS_AST_16AM3 = ast.parse(
    UTILS_SOURCE_16AM3
)


READ_PROMPT_SOURCE_16AM3 = None
INSTRUCT_FORMATTER_SOURCE_16AM3 = None


for node in UTILS_AST_16AM3.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "read_prompt"
    ):

        READ_PROMPT_SOURCE_16AM3 = (
            ast.get_source_segment(
                UTILS_SOURCE_16AM3,
                node
            )
        )


    elif (
        isinstance(
            node,
            ast.ClassDef
        )
        and
        node.name
        ==
        "InstructFormater"
    ):

        INSTRUCT_FORMATTER_SOURCE_16AM3 = (
            ast.get_source_segment(
                UTILS_SOURCE_16AM3,
                node
            )
        )


assert READ_PROMPT_SOURCE_16AM3
assert INSTRUCT_FORMATTER_SOURCE_16AM3


formatter_ns_16am3 = {
    "__builtins__":
        __builtins__,
}


exec(
    READ_PROMPT_SOURCE_16AM3,
    formatter_ns_16am3
)


exec(
    INSTRUCT_FORMATTER_SOURCE_16AM3,
    formatter_ns_16am3
)


InstructFormater_16AM3 = (
    formatter_ns_16am3[
        "InstructFormater"
    ]
)


read_prompt_16am3 = (
    formatter_ns_16am3[
        "read_prompt"
    ]
)


# Mimic exact notebook "utils.InstructFormater" reference,
# without importing utils/__init__.py.
utils_exact_16am3 = (
    types.SimpleNamespace(
        InstructFormater=
            InstructFormater_16AM3
    )
)


print(
    "\nExact InstructFormater source: PASSED"
)

print(
    " source:",
    UTILS_UTILS_PATH_16AM3
)


# ======================================================================
# 11. INDEX INITIAL PLANNER-SETUP NOTEBOOK CELLS
# ======================================================================
#
# Cell 19 is the confirmed exact import:
#
# from qa_prediction.gen_rule_path import
#     generate_seq, parse_prediction, INSTRUCTION
#
# We restrict planner model/prompt recovery to the initial planning
# setup region through cell 21, avoiding later reasoner assignments.
# ======================================================================

SETUP_MAX_CELL_16AM3 = 21


assignment_index_16am3 = {}

function_index_16am3 = {}

class_index_16am3 = {}

seed_statements_16am3 = []


def target_names_16am3(
    target
):

    if isinstance(
        target,
        ast.Name
    ):

        return [
            target.id
        ]


    if isinstance(
        target,
        (
            ast.Tuple,
            ast.List,
        )
    ):

        result = []


        for element in target.elts:

            result.extend(
                target_names_16am3(
                    element
                )
            )


        return result


    return []


for cell_idx in range(
    min(
        SETUP_MAX_CELL_16AM3 + 1,
        len(
            persisted_nb_16ar[
                "cells"
            ]
        )
    )
):

    cell = (
        persisted_nb_16ar[
            "cells"
        ][
            cell_idx
        ]
    )


    if cell.get(
        "cell_type"
    ) != "code":

        continue


    source = "".join(
        cell.get(
            "source",
            []
        )
    )


    try:

        tree = ast.parse(
            source
        )

    except Exception:

        continue


    for node in tree.body:

        segment = ast.get_source_segment(
            source,
            node
        )


        if not segment:

            continue


        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            function_index_16am3.setdefault(
                node.name,
                []
            ).append(
                {
                    "cell":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        elif isinstance(
            node,
            ast.ClassDef
        ):

            class_index_16am3.setdefault(
                node.name,
                []
            ).append(
                {
                    "cell":
                        cell_idx,

                    "source":
                        segment,
                }
            )


        elif isinstance(
            node,
            (
                ast.Assign,
                ast.AnnAssign,
            )
        ):

            if isinstance(
                node,
                ast.Assign
            ):

                targets = []


                for target in node.targets:

                    targets.extend(
                        target_names_16am3(
                            target
                        )
                    )


            else:

                targets = target_names_16am3(
                    node.target
                )


            for name in targets:

                assignment_index_16am3.setdefault(
                    name,
                    []
                ).append(
                    {
                        "cell":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


        elif isinstance(
            node,
            ast.Expr
        ):

            lower = segment.lower()


            if any(
                token in lower
                for token in [
                    "set_seed(",
                    "manual_seed(",
                    "random.seed(",
                    "np.random.seed(",
                    "numpy.random.seed(",
                ]
            ):

                seed_statements_16am3.append(
                    {
                        "cell":
                            cell_idx,

                        "source":
                            segment,
                    }
                )


print(
    "\nInitial planner-setup index: READY"
)


# ======================================================================
# 12. PRINT CRITICAL RECOVERED ASSIGNMENT LOCATIONS
# ======================================================================

print(
    "\nCritical setup assignment locations:"
)


for symbol in [
    "N_BEAM",
    "prompter",
    "model",
    "tokenizer",
]:

    rows = assignment_index_16am3.get(
        symbol,
        []
    )


    print(
        f" {symbol}:",
        [
            row[
                "cell"
            ]
            for row in rows
        ]
    )


# ======================================================================
# 13. EXACT RECOVERY NAMESPACE
# ======================================================================

planner_setup_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "os":
        os,

    "sys":
        sys,

    "json":
        json,

    "time":
        time,

    "random":
        random,

    "re":
        re,

    "np":
        np,

    "numpy":
        np,

    "torch":
        torch,

    "Path":
        Path,

    "tqdm":
        tqdm,

    "AutoTokenizer":
        AutoTokenizer,

    "AutoModelForCausalLM":
        AutoModelForCausalLM,

    "AutoPeftModelForCausalLM":
        AutoPeftModelForCausalLM,

    # Exact source-derived replacement for utils.InstructFormater.
    "utils":
        utils_exact_16am3,

    "InstructFormater":
        InstructFormater_16AM3,

    # Exact recovered planner helpers.
    "generate_seq":
        generate_seq_16am3,

    "parse_prediction":
        parse_prediction_16am3,

    "INSTRUCTION":
        INSTRUCTION_16AM3,

    # Exact repository root.
    "REPO_DIR":
        str(
            ROG_REPO_16AM3
        ),
}


# ======================================================================
# 14. EXECUTE DEFINITIONS FROM SETUP REGION
# ======================================================================

for rows in class_index_16am3.values():

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ]
    )[
        -1
    ]


    try:

        exec(
            row[
                "source"
            ],
            planner_setup_ns_16am3
        )

    except Exception:

        pass


for rows in function_index_16am3.values():

    row = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ]
    )[
        -1
    ]


    try:

        exec(
            row[
                "source"
            ],
            planner_setup_ns_16am3
        )

    except Exception:

        pass


# ======================================================================
# 15. RECURSIVE ASSIGNMENT RECOVERY
# ======================================================================

def extract_missing_name_16am3(
    exc
):

    match = re.search(
        r"name '([^']+)' is not defined",
        str(
            exc
        )
    )


    return (
        match.group(
            1
        )
        if match
        else None
    )


recovery_stack_16am3 = set()


def recover_symbol_16am3(
    symbol,
    max_cell=SETUP_MAX_CELL_16AM3
):

    if symbol in planner_setup_ns_16am3:

        return None


    if symbol in recovery_stack_16am3:

        raise RuntimeError(
            f"Circular recovery dependency: "
            f"{symbol}"
        )


    rows = [
        row
        for row in
        assignment_index_16am3.get(
            symbol,
            []
        )
        if row[
            "cell"
        ]
        <=
        max_cell
    ]


    assert rows, (
        f"No initial-planning assignment found "
        f"for required symbol '{symbol}'."
    )


    rows = sorted(
        rows,
        key=lambda x:
            x[
                "cell"
            ],
        reverse=True
    )


    recovery_stack_16am3.add(
        symbol
    )


    last_error = None


    original_cwd = os.getcwd()


    try:

        # Relative prompt/model paths in the original repository
        # were intended to resolve from the RoG project root.
        os.chdir(
            ROG_REPO_16AM3
        )


        for row in rows:

            for _ in range(
                20
            ):

                try:

                    exec(
                        row[
                            "source"
                        ],
                        planner_setup_ns_16am3
                    )


                    if symbol in planner_setup_ns_16am3:

                        return row


                    break


                except NameError as exc:

                    last_error = exc


                    missing = (
                        extract_missing_name_16am3(
                            exc
                        )
                    )


                    if (
                        missing
                        and
                        missing in
                        assignment_index_16am3
                    ):

                        recover_symbol_16am3(
                            missing,
                            max_cell=
                                row[
                                    "cell"
                                ]
                        )

                        continue


                    if (
                        missing
                        and
                        missing in
                        function_index_16am3
                    ):

                        frow = sorted(
                            function_index_16am3[
                                missing
                            ],
                            key=lambda x:
                                x[
                                    "cell"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            frow[
                                "source"
                            ],
                            planner_setup_ns_16am3
                        )

                        continue


                    if (
                        missing
                        and
                        missing in
                        class_index_16am3
                    ):

                        crow = sorted(
                            class_index_16am3[
                                missing
                            ],
                            key=lambda x:
                                x[
                                    "cell"
                                ]
                        )[
                            -1
                        ]


                        exec(
                            crow[
                                "source"
                            ],
                            planner_setup_ns_16am3
                        )

                        continue


                    break


                except Exception as exc:

                    last_error = exc

                    break


    finally:

        os.chdir(
            original_cwd
        )

        recovery_stack_16am3.remove(
            symbol
        )


    raise AssertionError(
        f"Could not recover '{symbol}'.\n"
        f"Last error: "
        f"{type(last_error).__name__ if last_error else 'unknown'}: "
        f"{last_error}"
    )


# ======================================================================
# 16. RECOVER N_BEAM
# ======================================================================

N_BEAM_ROW_16AM3 = (
    recover_symbol_16am3(
        "N_BEAM"
    )
)


N_BEAM_16AM3 = int(
    planner_setup_ns_16am3[
        "N_BEAM"
    ]
)


assert (
    N_BEAM_16AM3
    ==
    FROZEN_N_BEAM_16AM3
)


print(
    "\nN_BEAM recovery: PASSED"
)

print(
    " N_BEAM:",
    N_BEAM_16AM3
)


# ======================================================================
# 17. RECOVER PROMPTER
# ======================================================================

PROMPTER_ROW_16AM3 = (
    recover_symbol_16am3(
        "prompter"
    )
)


prompter_16am3 = (
    planner_setup_ns_16am3[
        "prompter"
    ]
)


assert hasattr(
    prompter_16am3,
    "format"
)


prompt_probe_16am3 = (
    prompter_16am3.format(
        instruction=
            INSTRUCTION_16AM3,

        message=
            str(
                webqsp_val_plan_rows[
                    0
                ][
                    "question"
                ]
            )
    )
)


assert isinstance(
    prompt_probe_16am3,
    str
)

assert len(
    prompt_probe_16am3
) > 0


PROMPT_PROBE_SHA_16AM3 = (
    sha256_text_16am3(
        prompt_probe_16am3
    )
)


print(
    "\nPrompter recovery: PASSED"
)

print(
    " assignment cell:",
    (
        PROMPTER_ROW_16AM3[
            "cell"
        ]
        if PROMPTER_ROW_16AM3
        else
        "already available"
    )
)

print(
    " prompt probe SHA:",
    PROMPT_PROBE_SHA_16AM3
)


# ======================================================================
# 18. MODEL IDENTITY HELPERS
# ======================================================================

def identity_strings_16am3(
    obj
):

    values = []


    for attr in [
        "name_or_path",
        "_name_or_path",
    ]:

        value = getattr(
            obj,
            attr,
            None
        )


        if isinstance(
            value,
            str
        ):

            values.append(
                value
            )


    config = getattr(
        obj,
        "config",
        None
    )


    if config is not None:

        for attr in [
            "name_or_path",
            "_name_or_path",
        ]:

            value = getattr(
                config,
                attr,
                None
            )


            if isinstance(
                value,
                str
            ):

                values.append(
                    value
                )


    return list(
        dict.fromkeys(
            values
        )
    )


def is_rog_runtime_16am3(
    obj
):

    return any(
        FROZEN_MODEL_ID_16AM3.lower()
        in value.lower()

        for value in
        identity_strings_16am3(
            obj
        )
    )


# ======================================================================
# 19. REUSE SURVIVING RoG MODEL IF PRESENT
# ======================================================================

model_16am3 = None
tokenizer_16am3 = None


for name, obj in list(
    globals().items()
):

    try:

        if (
            hasattr(
                obj,
                "generate"
            )
            and
            is_rog_runtime_16am3(
                obj
            )
        ):

            model_16am3 = obj

            print(
                "\nUsing surviving frozen RoG model:",
                name
            )

            break


    except Exception:

        pass


for name, obj in list(
    globals().items()
):

    try:

        if (
            callable(
                obj
            )
            and
            hasattr(
                obj,
                "decode"
            )
            and
            is_rog_runtime_16am3(
                obj
            )
        ):

            tokenizer_16am3 = obj

            print(
                "Using surviving frozen RoG tokenizer:",
                name
            )

            break


    except Exception:

        pass


# ======================================================================
# 20. OTHERWISE RECOVER EXACT INITIAL MODEL ASSIGNMENTS
# ======================================================================

if model_16am3 is None:

    # Ensure a stale unrelated "model" does not block recursive recovery.
    planner_setup_ns_16am3.pop(
        "model",
        None
    )


    MODEL_ROW_16AM3 = (
        recover_symbol_16am3(
            "model"
        )
    )


    model_16am3 = (
        planner_setup_ns_16am3[
            "model"
        ]
    )


else:

    MODEL_ROW_16AM3 = None


if tokenizer_16am3 is None:

    candidate = planner_setup_ns_16am3.get(
        "tokenizer"
    )


    if (
        candidate is not None
        and
        callable(
            candidate
        )
        and
        hasattr(
            candidate,
            "decode"
        )
    ):

        tokenizer_16am3 = candidate


    else:

        planner_setup_ns_16am3.pop(
            "tokenizer",
            None
        )


        TOKENIZER_ROW_16AM3 = (
            recover_symbol_16am3(
                "tokenizer"
            )
        )


        tokenizer_16am3 = (
            planner_setup_ns_16am3[
                "tokenizer"
            ]
        )


else:

    TOKENIZER_ROW_16AM3 = None


assert hasattr(
    model_16am3,
    "generate"
)


assert callable(
    tokenizer_16am3
)

assert hasattr(
    tokenizer_16am3,
    "decode"
)


MODEL_IDS_16AM3 = (
    identity_strings_16am3(
        model_16am3
    )
)


TOKENIZER_IDS_16AM3 = (
    identity_strings_16am3(
        tokenizer_16am3
    )
)


assert is_rog_runtime_16am3(
    model_16am3
), (
    "Recovered planning model is not rmanluo/RoG.\n"
    f"Runtime identity: {MODEL_IDS_16AM3}"
)


print(
    "\nPlanner model identity: PASSED"
)

print(
    " model IDs:",
    MODEL_IDS_16AM3
)

print(
    " tokenizer IDs:",
    TOKENIZER_IDS_16AM3
)


# ======================================================================
# 21. ENSURE MODEL IS USABLE FOR EXACT generate_seq
# ======================================================================

model_16am3.eval()


print(
    "Model eval mode: READY"
)


# ======================================================================
# 22. EXACT load_checkpoint FROM NOTEBOOK CELL 47
# ======================================================================

CELL47_SOURCE_16AM3 = "".join(
    persisted_nb_16ar[
        "cells"
    ][
        47
    ].get(
        "source",
        []
    )
)


CELL47_AST_16AM3 = ast.parse(
    CELL47_SOURCE_16AM3
)


LOAD_CHECKPOINT_SOURCE_16AM3 = None


for node in CELL47_AST_16AM3.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "load_checkpoint"
    ):

        LOAD_CHECKPOINT_SOURCE_16AM3 = (
            ast.get_source_segment(
                CELL47_SOURCE_16AM3,
                node
            )
        )

        break


assert LOAD_CHECKPOINT_SOURCE_16AM3


runtime_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "os":
        os,

    "json":
        json,
}


exec(
    LOAD_CHECKPOINT_SOURCE_16AM3,
    runtime_ns_16am3
)


load_checkpoint_16am3 = (
    runtime_ns_16am3[
        "load_checkpoint"
    ]
)


print(
    "\nload_checkpoint recovery: PASSED"
)

print(
    " SHA:",
    sha256_text_16am3(
        LOAD_CHECKPOINT_SOURCE_16AM3
    )
)


# ======================================================================
# 23. EXACT PLANNING WRAPPER FROM CELL 139
# ======================================================================

planning_ns_16am3 = {
    "__builtins__":
        __builtins__,

    "load_checkpoint":
        load_checkpoint_16am3,

    "generate_seq":
        generate_seq_16am3,

    "parse_prediction":
        parse_prediction_16am3,

    "INSTRUCTION":
        INSTRUCTION_16AM3,

    "prompter":
        prompter_16am3,

    "model":
        model_16am3,

    "tokenizer":
        tokenizer_16am3,

    "N_BEAM":
        N_BEAM_16AM3,

    "time":
        time,

    "json":
        json,

    "tqdm":
        tqdm,
}


exec(
    run_planning_source_16ar,
    planning_ns_16am3
)


run_planning_checkpointed_16am3 = (
    planning_ns_16am3[
        "run_planning_checkpointed"
    ]
)


assert str(
    inspect.signature(
        run_planning_checkpointed_16am3
    )
) == (
    "(dataset_split, ckpt_path, desc)"
)


print(
    "\nExact Cell-139 planner wrapper: PASSED"
)

print(
    " SHA:",
    sha256_text_16am3(
        run_planning_source_16ar
    )
)


# ======================================================================
# 24. GOLD-LEAKAGE SOURCE GATE
# ======================================================================

generation_section_16am3 = (
    run_planning_source_16ar
    .split(
        "rec =",
        1
    )[
        0
    ]
)


assert (
    'sample["a_entity"]'
    not in
    generation_section_16am3
)


assert (
    'sample["answer"]'
    not in
    generation_section_16am3
)


assert (
    'sample["question"]'
    in
    generation_section_16am3
)


print(
    "\nLeakage source gate: PASSED"
)

print(
    " TEST gold in prompt/generation: NO"
)


# ======================================================================
# 25. SEED AUDIT
# ======================================================================
#
# do_sample=True, therefore seed behavior matters.
#
# Apply only seed statements explicitly present in the original
# initial-planning setup. Never invent seed 42 here.
# ======================================================================

print(
    "\nExplicit original initial-planning seed statements:"
)


if seed_statements_16am3:

    for row in seed_statements_16am3:

        print(
            f" cell {row['cell']}: "
            f"{row['source']}"
        )


else:

    print(
        " NONE"
    )


for row in sorted(
    seed_statements_16am3,
    key=lambda x:
        x[
            "cell"
        ]
):

    exec(
        row[
            "source"
        ],
        planner_setup_ns_16am3
    )


if not seed_statements_16am3:

    print(
        "No new random seed invented."
    )


# ======================================================================
# 26. OUTPUT PATHS
# ======================================================================

TEST_PLANNING_DIR_16AM3 = Path(
    "/kaggle/working/"
    "step2_rq1_test"
)


TEST_PLANNING_DIR_16AM3.mkdir(
    parents=True,
    exist_ok=True
)


WEBQSP_TEST_PLAN_PATH = (
    TEST_PLANNING_DIR_16AM3
    / "planning_webqsp_test.jsonl"
)


CWQ_TEST_PLAN_PATH = (
    TEST_PLANNING_DIR_16AM3
    / "planning_cwq_test.jsonl"
)


MANIFEST_PATH_16AM3 = (
    TEST_PLANNING_DIR_16AM3
    / "cell16a_m3_materialization_manifest.json"
)


# ======================================================================
# 27. CHECKPOINT COUNTER
# ======================================================================

def checkpoint_count_16am3(
    path
):

    if not path.exists():

        return 0


    count = 0


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if line.strip():

                count += 1


    return count


print(
    "\nExisting TEST planning checkpoints:"
)

print(
    " WebQSP:",
    checkpoint_count_16am3(
        WEBQSP_TEST_PLAN_PATH
    ),
    "/ 1628"
)

print(
    " CWQ:",
    checkpoint_count_16am3(
        CWQ_TEST_PLAN_PATH
    ),
    "/ 3531"
)


# ======================================================================
# 28. INPUT COUNT / DISJOINTNESS GATE
# ======================================================================

assert len(
    webqsp_official_test_16ar
) == 1628


assert len(
    cwq_official_test_16ar
) == 3531


web_val_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    webqsp_val_plan_rows
}


cwq_val_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    cwq_val_plan_rows
}


web_test_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    webqsp_raw_test_16a
}


cwq_test_ids_16am3 = {
    str(
        x[
            "id"
        ]
    )
    for x in
    cwq_raw_test_16a
}


assert not (
    web_val_ids_16am3
    &
    web_test_ids_16am3
)


assert not (
    cwq_val_ids_16am3
    &
    cwq_test_ids_16am3
)


print(
    "\nTEST dataset gate: PASSED"
)

print(
    " WebQSP: 1628"
)

print(
    " CWQ:    3531"
)

print(
    " validation/test overlap: 0"
)


# ======================================================================
# 29. FINAL PRE-GENERATION AUDIT
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "FINAL PRE-GENERATION FROZEN PLANNER AUDIT"
)

print(
    "=" * 126
)


print(
    "gen_rule_path source SHA:      PASSED"
)

print(
    "generate_seq exact source:     PASSED"
)

print(
    "parse_prediction exact source: PASSED"
)

print(
    "INSTRUCTION frozen SHA:        PASSED"
)

print(
    "InstructFormater exact source: PASSED"
)

print(
    "N_BEAM = 3:                    PASSED"
)

print(
    "rmanluo/RoG model identity:    PASSED"
)

print(
    "Cell-47 checkpoint loader:     PASSED"
)

print(
    "Cell-139 planning wrapper:     PASSED"
)

print(
    "TEST gold in generation:       NO"
)

print(
    "AFP changed:                   NO"
)

print(
    "TEST tuning:                   NO"
)


# ======================================================================
# 30. MATERIALIZE WEBQSP TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "MATERIALIZING WEBQSP FROZEN TEST PLANS"
)

print(
    "=" * 126
)


web_start_16am3 = time.time()


webqsp_test_plan_rows = (
    run_planning_checkpointed_16am3(
        dataset_split=
            webqsp_official_test_16ar,

        ckpt_path=
            str(
                WEBQSP_TEST_PLAN_PATH
            ),

        desc=
            "WebQSP frozen TEST planning"
    )
)


WEBQSP_PLANNING_SECONDS_16AM3 = (
    time.time()
    -
    web_start_16am3
)


print(
    "\nWebQSP planning elapsed:",
    f"{WEBQSP_PLANNING_SECONDS_16AM3/60:.2f} min"
)


# ======================================================================
# 31. MATERIALIZE CWQ TEST PLANS
# ======================================================================

print(
    "\n"
    + "=" * 126
)

print(
    "MATERIALIZING CWQ FROZEN TEST PLANS"
)

print(
    "=" * 126
)


cwq_start_16am3 = time.time()


cwq_test_plan_rows = (
    run_planning_checkpointed_16am3(
        dataset_split=
            cwq_official_test_16ar,

        ckpt_path=
            str(
                CWQ_TEST_PLAN_PATH
            ),

        desc=
            "CWQ frozen TEST planning"
    )
)


CWQ_PLANNING_SECONDS_16AM3 = (
    time.time()
    -
    cwq_start_16am3
)


print(
    "\nCWQ planning elapsed:",
    f"{CWQ_PLANNING_SECONDS_16AM3/60:.2f} min"
)


# ======================================================================
# 32. STRICT OUTPUT AUDIT
# ======================================================================

REQUIRED_OUTPUT_FIELDS_16AM3 = {
    "id",
    "question",
    "q_entity",
    "a_entity",
    "graph",
    "predicted_paths",
    "planning_time_sec",
}


def audit_plans_16am3(
    dataset_name,
    rows,
    raw_rows,
    expected_n
):

    assert len(
        rows
    ) == expected_n


    raw_by_id = {
        str(
            row[
                "id"
            ]
        ):
            row

        for row in raw_rows
    }


    output_ids = [
        str(
            row[
                "id"
            ]
        )

        for row in rows
    ]


    assert len(
        output_ids
    ) == len(
        set(
            output_ids
        )
    )


    assert set(
        output_ids
    ) == set(
        raw_by_id.keys()
    )


    total_plans = 0
    empty_questions = 0

    lengths = []


    for row in rows:

        assert REQUIRED_OUTPUT_FIELDS_16AM3.issubset(
            row.keys()
        )


        qid = str(
            row[
                "id"
            ]
        )


        raw = raw_by_id[
            qid
        ]


        assert str(
            row[
                "question"
            ]
        ) == str(
            raw[
                "question"
            ]
        )


        assert list(
            row[
                "q_entity"
            ]
        ) == list(
            raw[
                "q_entity"
            ]
        )


        assert list(
            row[
                "a_entity"
            ]
        ) == list(
            raw[
                "a_entity"
            ]
        )


        paths = row[
            "predicted_paths"
        ]


        assert isinstance(
            paths,
            list
        )


        assert len(
            paths
        ) <= 3


        total_plans += len(
            paths
        )


        if len(
            paths
        ) == 0:

            empty_questions += 1


        for plan in paths:

            assert isinstance(
                plan,
                list
            )

            lengths.append(
                len(
                    plan
                )
            )


        assert float(
            row[
                "planning_time_sec"
            ]
        ) >= 0.0


    result = {
        "questions":
            int(
                len(
                    rows
                )
            ),

        "total_predicted_plans":
            int(
                total_plans
            ),

        "mean_plans_per_question":
            float(
                total_plans
                /
                len(
                    rows
                )
            ),

        "empty_plan_questions":
            int(
                empty_questions
            ),

        "min_plan_length":
            (
                int(
                    min(
                        lengths
                    )
                )
                if lengths
                else None
            ),

        "max_plan_length":
            (
                int(
                    max(
                        lengths
                    )
                )
                if lengths
                else None
            ),

        "mean_plan_length":
            (
                float(
                    np.mean(
                        lengths
                    )
                )
                if lengths
                else None
            ),
    }


    print(
        f"\n{dataset_name.upper()} audit: PASSED"
    )

    print(
        " questions:",
        result[
            "questions"
        ]
    )

    print(
        " predicted plans:",
        result[
            "total_predicted_plans"
        ]
    )

    print(
        " mean plans/question:",
        f"{result['mean_plans_per_question']:.6f}"
    )

    print(
        " empty-plan questions:",
        result[
            "empty_plan_questions"
        ]
    )

    print(
        " plan-length min/max:",
        result[
            "min_plan_length"
        ],
        "/",
        result[
            "max_plan_length"
        ]
    )


    return result


WEBQSP_PLAN_AUDIT_16AM3 = (
    audit_plans_16am3(
        "webqsp",
        webqsp_test_plan_rows,
        webqsp_raw_test_16a,
        1628
    )
)


CWQ_PLAN_AUDIT_16AM3 = (
    audit_plans_16am3(
        "cwq",
        cwq_test_plan_rows,
        cwq_raw_test_16a,
        3531
    )
)


# ======================================================================
# 33. FILE COMPLETENESS + SHA
# ======================================================================

assert (
    checkpoint_count_16am3(
        WEBQSP_TEST_PLAN_PATH
    )
    ==
    1628
)


assert (
    checkpoint_count_16am3(
        CWQ_TEST_PLAN_PATH
    )
    ==
    3531
)


WEBQSP_TEST_PLAN_SHA = (
    sha256_file_16am3(
        WEBQSP_TEST_PLAN_PATH
    )
)


CWQ_TEST_PLAN_SHA = (
    sha256_file_16am3(
        CWQ_TEST_PLAN_PATH
    )
)


print(
    "\nFrozen TEST planning SHA256:"
)

print(
    " WebQSP:",
    WEBQSP_TEST_PLAN_SHA
)

print(
    " CWQ:",
    CWQ_TEST_PLAN_SHA
)


# ======================================================================
# 34. IMPLEMENTATION FINGERPRINT
# ======================================================================

PLANNER_IMPLEMENTATION_16AM3 = {
    "model_id":
        FROZEN_MODEL_ID_16AM3,

    "gen_rule_path_file_sha256":
        ACTUAL_GEN_RULE_SHA_16AM3,

    "instruction_sha256":
        INSTRUCTION_SHA_16AM3,

    "generate_seq_source_sha256":
        sha256_text_16am3(
            GENERATE_SEQ_SOURCE_16AM3
        ),

    "parse_prediction_source_sha256":
        sha256_text_16am3(
            PARSE_PREDICTION_SOURCE_16AM3
        ),

    "InstructFormater_source_sha256":
        sha256_text_16am3(
            INSTRUCT_FORMATTER_SOURCE_16AM3
        ),

    "load_checkpoint_source_sha256":
        sha256_text_16am3(
            LOAD_CHECKPOINT_SOURCE_16AM3
        ),

    "run_planning_checkpointed_source_sha256":
        sha256_text_16am3(
            run_planning_source_16ar
        ),

    "N_BEAM":
        N_BEAM_16AM3,

    "do_sample":
        True,

    "max_new_tokens":
        100,

    "model_runtime_identity":
        MODEL_IDS_16AM3,

    "tokenizer_runtime_identity":
        TOKENIZER_IDS_16AM3,

    "prompt_probe_sha256":
        PROMPT_PROBE_SHA_16AM3,

    "seed_statements":
        seed_statements_16am3,

    "invented_new_seed":
        False,

    "graph_walker_required_for_planning_helpers":
        False,

    "full_utils_package_imported":
        False,
}


PLANNER_IMPLEMENTATION_SHA_16AM3 = (
    sha256_text_16am3(
        json.dumps(
            PLANNER_IMPLEMENTATION_16AM3,
            sort_keys=True,
            separators=(
                ",",
                ":"
            ),
            ensure_ascii=False
        )
    )
)


print(
    "\nPlanner implementation fingerprint:"
)

print(
    " ",
    PLANNER_IMPLEMENTATION_SHA_16AM3
)


# ======================================================================
# 35. SAVE MATERIALIZATION MANIFEST
# ======================================================================

MATERIALIZATION_MANIFEST_16AM3 = {
    "stage":
        "frozen_test_relation_plan_materialization",

    "afp_freeze_sha256":
        FINAL_AFP_FREEZE_SHA256,

    "planner":
        {
            **PLANNER_IMPLEMENTATION_16AM3,

            "implementation_sha256":
                PLANNER_IMPLEMENTATION_SHA_16AM3,
        },

    "origin_evidence": {
        "generate_seq":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "parse_prediction":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "INSTRUCTION":
            (
                "exact AST extraction from SHA-verified "
                "src/qa_prediction/gen_rule_path.py"
            ),

        "InstructFormater":
            (
                "exact AST extraction from "
                "src/utils/utils.py; "
                "utils package __init__ not imported"
            ),

        "load_checkpoint":
            "exact notebook cell 47 source",

        "run_planning_checkpointed":
            "exact notebook cell 139 source",
    },

    "outputs": {
        "webqsp": {
            "path":
                str(
                    WEBQSP_TEST_PLAN_PATH
                ),

            "sha256":
                WEBQSP_TEST_PLAN_SHA,

            **WEBQSP_PLAN_AUDIT_16AM3,
        },

        "cwq": {
            "path":
                str(
                    CWQ_TEST_PLAN_PATH
                ),

            "sha256":
                CWQ_TEST_PLAN_SHA,

            **CWQ_PLAN_AUDIT_16AM3,
        },
    },

    "leakage": {
        "test_gold_in_prompt":
            False,

        "test_gold_in_generation":
            False,

        "test_used_for_tuning":
            False,

        "test_used_for_model_selection":
            False,

        "test_triggered_method_change":
            False,
    },

    "afp_changed_after_freeze":
        False,

    "next":
        "rerun_Cell16_unchanged",
}


with open(
    MANIFEST_PATH_16AM3,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MATERIALIZATION_MANIFEST_16AM3,
        f,
        indent=2,
        ensure_ascii=False
    )


CELL16AM3_COMPLETE = True

FROZEN_TEST_PLANS_MATERIALIZED = True


# ======================================================================
# 36. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 130
)

print(
    "=== CELL 16A-M3: "
    "FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ==="
)

print(
    "=" * 130
)


print(
    "\nRecovery strategy:"
)

print(
    " gen_rule_path module imported: NO"
)

print(
    " utils package imported:        NO"
)

print(
    " graph-walker required:         NO"
)

print(
    " exact source extracted:        YES"
)


print(
    "\nFrozen planner:"
)

print(
    " model:",
    FROZEN_MODEL_ID_16AM3
)

print(
    " N_BEAM:",
    N_BEAM_16AM3
)

print(
    " instruction SHA:",
    INSTRUCTION_SHA_16AM3
)


print(
    "\nTEST planning files:"
)

print(
    " WebQSP:"
)

print(
    "  questions:",
    len(
        webqsp_test_plan_rows
    )
)

print(
    "  file:",
    WEBQSP_TEST_PLAN_PATH
)

print(
    "  SHA:",
    WEBQSP_TEST_PLAN_SHA
)


print(
    " CWQ:"
)

print(
    "  questions:",
    len(
        cwq_test_plan_rows
    )
)

print(
    "  file:",
    CWQ_TEST_PLAN_PATH
)

print(
    "  SHA:",
    CWQ_TEST_PLAN_SHA
)


print(
    "\nScientific integrity:"
)

print(
    " TEST gold in generation: NO"
)

print(
    " TEST tuning:             NO"
)

print(
    " AFP changed:             NO"
)

print(
    " new seed invented:       NO"
)


print(
    "\nManifest:"
)

print(
    " ",
    MANIFEST_PATH_16AM3
)


print(
    "\nNEXT ACTION:"
)

print(
    "RERUN CELL 16 UNCHANGED."
)

print(
    "Then Cell 17, then Cell 18."
)

Cell 16A-M3 freeze gate: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116

Frozen planner:
 model: rmanluo/RoG
 N_BEAM: 3
 do_sample: True
 max_new_tokens: 100

CUDA gate: PASSED
 GPU: Tesla T4

Exact gen_rule_path.py SHA gate: PASSED
  e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af

Instruction fidelity: PASSED
 SHA: e3687b4a5081c22cecf474857676a2d10a7132d667507eac5bebc79263e5096d

Exact planner helpers reconstructed: PASSED
 generate_seq: (model, input_text, tokenizer, num_beam=3, do_sample=False, max_new_tokens=100)
 parse_prediction: (prediction)
 generate_seq SHA: 23deb37187b09722b84af94c3c7b146b37d2d65a1755bc31c149533c57aa09d9
 parse_prediction SHA: 6fc2edaf54a0a0388cb4ccc0cb68f3cc7cbc3b6c3657c5d13f0e7d38dd9ee4ad

Exact InstructFormater source: PASSED
 source: /kaggle/working/reasoning-on-graphs/src/utils/utils.py

Initial planner-setup index: READY

Critical setup assignment locations:
 N_BEAM: [9]
 prompter: [19]
 model: [13]

config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...


Planner model identity: PASSED
 model IDs: ['rmanluo/RoG']
 tokenizer IDs: ['rmanluo/RoG']
Model eval mode: READY

load_checkpoint recovery: PASSED
 SHA: a50d1a3b3f62b72c6745467a0ba96dc8b9313183865671d6ad1b7448d22e889c

Exact Cell-139 planner wrapper: PASSED
 SHA: 8dc630d5e3cb8c448e7858c03f66b9143af1599e7f8bb196f81439f917bc2bdd

Leakage source gate: PASSED
 TEST gold in prompt/generation: NO

Explicit original initial-planning seed statements:
 NONE
No new random seed invented.

Existing TEST planning checkpoints:
 WebQSP: 0 / 1628
 CWQ: 0 / 3531

TEST dataset gate: PASSED
 WebQSP: 1628
 CWQ:    3531
 validation/test overlap: 0

FINAL PRE-GENERATION FROZEN PLANNER AUDIT
gen_rule_path source SHA:      PASSED
generate_seq exact source:     PASSED
parse_prediction exact source: PASSED
INSTRUCTION frozen SHA:        PASSED
InstructFormater exact source: PASSED
N_BEAM = 3:                    PASSED
rmanluo/RoG model identity:    PASSED
Cell-47 checkpoint loader:     PASSED
Cell-139 plannin

WebQSP frozen TEST planning:   0%|          | 0/1628 [00:00<?, ?it/s]


WebQSP planning elapsed: 54.98 min

MATERIALIZING CWQ FROZEN TEST PLANS
Resuming: 0 / 3531 already planned (CWQ frozen TEST planning).


CWQ frozen TEST planning:   0%|          | 0/3531 [00:00<?, ?it/s]


CWQ planning elapsed: 139.98 min

WEBQSP audit: PASSED
 questions: 1628
 predicted plans: 4880
 mean plans/question: 2.997543
 empty-plan questions: 0
 plan-length min/max: 1 / 4

CWQ audit: PASSED
 questions: 3531
 predicted plans: 10537
 mean plans/question: 2.984140
 empty-plan questions: 10
 plan-length min/max: 1 / 5

Frozen TEST planning SHA256:
 WebQSP: ef5a647ee8d5952043792b8c2caa0e6a0b8d49cd3fbd0f1ff3aef3524a79d790
 CWQ: 95529560838f68b9c582bab3fe5b2357b76e401291dbed75b5b49ec72a4e3e53

Planner implementation fingerprint:
  8d5537a822cf4e8e28d586bd804f8d02c47d87592e2ecea2b12fe768ea2be058

=== CELL 16A-M3: FROZEN RoG TEST PLAN MATERIALIZATION COMPLETE ===

Recovery strategy:
 gen_rule_path module imported: NO
 utils package imported:        NO
 graph-walker required:         NO
 exact source extracted:        YES

Frozen planner:
 model: rmanluo/RoG
 N_BEAM: 3
 instruction SHA: e3687b4a5081c22cecf474857676a2d10a7132d667507eac5bebc79263e5096d

TEST planning files:
 WebQSP:
  que

In [42]:
# ======================================================================
# CPU → GPU SAFETY BACKUP
# RUN THIS BEFORE CHANGING KAGGLE ACCELERATOR
# ======================================================================

from pathlib import Path
import zipfile
import hashlib
import os

BACKUP_ZIP = Path(
    "/kaggle/working/AdaPruner_CPU_to_GPU_backup.zip"
)

critical_paths = [
    # --------------------------------------------------------------
    # Final frozen AFP development boundary
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "14_selector_ablations_and_final_freeze"
    ),

    # --------------------------------------------------------------
    # Selected scorer checkpoints
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "07_final_scorer"
    ),

    # --------------------------------------------------------------
    # Frozen selector / baseline definitions
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "08_adaptive_selector"
    ),

    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "09_controlled_baselines"
    ),

    # --------------------------------------------------------------
    # Exact validation traversal/fidelity artifacts
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "10_validation_traversal"
    ),

    # --------------------------------------------------------------
    # Validation feature/scorer evidence
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step3_rq2_dev_v1/"
        "13_feature_ablations"
    ),

    # --------------------------------------------------------------
    # Frozen validation relation plans
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step2_rq1_dev/"
        "planning_webqsp_validation.jsonl"
    ),

    Path(
        "/kaggle/working/step2_rq1_dev/"
        "planning_cwq_validation.jsonl"
    ),

    # --------------------------------------------------------------
    # Any test-recovery material created so far
    # --------------------------------------------------------------
    Path(
        "/kaggle/working/step2_rq1_test"
    ),
]


def add_path_to_zip(zf, path):
    if not path.exists():
        print("NOT FOUND:", path)
        return

    if path.is_file():
        arcname = str(path).lstrip("/")
        zf.write(path, arcname)
        print("ADDED FILE:", path)
        return

    for file_path in path.rglob("*"):
        if file_path.is_file():
            arcname = str(file_path).lstrip("/")
            zf.write(file_path, arcname)

    print("ADDED DIR :", path)


with zipfile.ZipFile(
    BACKUP_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:

    for path in critical_paths:
        add_path_to_zip(zf, path)


# SHA256
h = hashlib.sha256()

with open(BACKUP_ZIP, "rb") as f:
    while True:
        chunk = f.read(1024 * 1024)

        if not chunk:
            break

        h.update(chunk)


backup_sha = h.hexdigest()
backup_mb = BACKUP_ZIP.stat().st_size / (1024 ** 2)


print("\n" + "=" * 90)
print("BACKUP COMPLETE")
print("=" * 90)

print("File:", BACKUP_ZIP)
print(f"Size: {backup_mb:.2f} MB")
print("SHA256:", backup_sha)

print("\nIMPORTANT:")
print("Download this ZIP to your computer BEFORE switching CPU → GPU.")

ADDED DIR : /kaggle/working/step3_rq2_dev_v1/14_selector_ablations_and_final_freeze
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/07_final_scorer
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/08_adaptive_selector
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/09_controlled_baselines
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/10_validation_traversal
ADDED DIR : /kaggle/working/step3_rq2_dev_v1/13_feature_ablations
ADDED FILE: /kaggle/working/step2_rq1_dev/planning_webqsp_validation.jsonl
ADDED FILE: /kaggle/working/step2_rq1_dev/planning_cwq_validation.jsonl
ADDED DIR : /kaggle/working/step2_rq1_test

BACKUP COMPLETE
File: /kaggle/working/AdaPruner_CPU_to_GPU_backup.zip
Size: 189.68 MB
SHA256: 3422b3f86f2e9aef00e8b00feb9cf3be6850da1705e8d997aca4601a0c47bf2b

IMPORTANT:
Download this ZIP to your computer BEFORE switching CPU → GPU.


In [43]:
# ============================================================
# VERIFY CPU→GPU BACKUP BEFORE SWITCHING SESSION
# ============================================================

from pathlib import Path
import zipfile

backup = Path(
    "/kaggle/working/AdaPruner_CPU_to_GPU_backup.zip"
)

assert backup.exists(), "Backup ZIP is missing."

required_fragments = [
    # Final freeze
    "final_afp_development_freeze.json",
    "final_afp_development_freeze.sha256",

    # Selected scorers
    "webqsp_afp_scorer_selected.pt",
    "cwq_afp_scorer_selected.pt",

    # Validation planning
    "planning_webqsp_validation.jsonl",
    "planning_cwq_validation.jsonl",

    # Controlled baseline / selector
    "controlled_baseline_definition_manifest.json",

    # Final selector/freeze development artifacts
    "webqsp_selector_component_ablation.csv",
    "cwq_selector_component_ablation.csv",
]

with zipfile.ZipFile(backup, "r") as zf:
    names = zf.namelist()

print("Backup contains", len(names), "files.")

missing = []

for fragment in required_fragments:
    matches = [
        name for name in names
        if name.endswith(fragment)
    ]

    if matches:
        print("FOUND:", fragment)
    else:
        print("MISSING:", fragment)
        missing.append(fragment)

assert not missing, (
    "\nCritical experiment files missing from backup:\n"
    + "\n".join(missing)
)

print("\n" + "=" * 80)
print("CRITICAL EXPERIMENT BACKUP VERIFIED")
print("=" * 80)
print("It is safe to keep this ZIP as the frozen-development backup.")

Backup contains 41 files.
FOUND: final_afp_development_freeze.json
FOUND: final_afp_development_freeze.sha256
FOUND: webqsp_afp_scorer_selected.pt
FOUND: cwq_afp_scorer_selected.pt
FOUND: planning_webqsp_validation.jsonl
FOUND: planning_cwq_validation.jsonl
FOUND: controlled_baseline_definition_manifest.json
FOUND: webqsp_selector_component_ablation.csv
FOUND: cwq_selector_component_ablation.csv

CRITICAL EXPERIMENT BACKUP VERIFIED
It is safe to keep this ZIP as the frozen-development backup.


In [1]:
# ======================================================================
# GPU RESTART RECOVERY CELL
# Reconstruct frozen state needed for Cell 16A-M3 and Cell 16
# ======================================================================
#
# RUN AS A NEW CELL IN THE NEW GPU SESSION.
#
# This does NOT:
#   - retrain AFP
#   - retune anything
#   - change the frozen configuration
#   - generate TEST plans
#
# It restores/reconstructs:
#   - frozen AFP boundary
#   - validation planning rows
#   - persisted notebook
#   - exact planner wrapper source
#   - official RoG validation/test datasets
#   - raw TEST rows
#   - exact traversal helpers needed by Cell 16
#   - local RoG repository copy needed by M3
#
# AFTER THIS PASSES:
#   run Cell 16A-M3 unchanged.
# ======================================================================

import ast
import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch
import networkx as nx

from datasets import load_dataset


# ======================================================================
# 1. GPU CHECK
# ======================================================================

assert torch.cuda.is_available(), (
    "GPU is still unavailable. "
    "Check Kaggle Accelerator settings."
)

print("GPU recovery session: READY")
print("GPU:", torch.cuda.get_device_name(0))


# ======================================================================
# 2. FIND / RESTORE BACKUP IF NECESSARY
# ======================================================================

WORK = Path("/kaggle/working")

DEV_ROOT = (
    WORK
    / "step3_rq2_dev_v1"
)

FREEZE_DIR = (
    DEV_ROOT
    / "14_selector_ablations_and_final_freeze"
)

FREEZE_JSON = (
    FREEZE_DIR
    / "final_afp_development_freeze.json"
)

FREEZE_SHA_FILE = (
    FREEZE_DIR
    / "final_afp_development_freeze.sha256"
)


def find_backup_zip():
    candidates = []

    search_roots = [
        Path("/kaggle/working"),
        Path("/kaggle/input"),
    ]

    names = [
        "AdaPruner_CPU_to_GPU_backup.zip",
        "AdaPruner_FULL_experiment_backup.zip",
    ]

    for root in search_roots:
        if not root.exists():
            continue

        for name in names:
            candidates.extend(
                root.rglob(name)
            )

    return candidates


# If files survived the restart, no extraction is necessary.
if FREEZE_JSON.exists() and FREEZE_SHA_FILE.exists():

    print("\nExisting experiment files survived GPU restart.")

else:

    backup_candidates = find_backup_zip()

    assert backup_candidates, (
        "\nExperiment files are not currently under /kaggle/working "
        "and I cannot find your backup ZIP in this GPU session.\n\n"
        "Upload your downloaded "
        "AdaPruner_CPU_to_GPU_backup.zip "
        "to Kaggle, then rerun THIS recovery cell.\n\n"
        "Do NOT rerun development experiments."
    )

    # Prefer full backup if available.
    backup_candidates = sorted(
        backup_candidates,
        key=lambda p:
            (
                "FULL" not in p.name,
                str(p)
            )
    )

    BACKUP_USED = backup_candidates[0]

    print("\nRestoring backup:")
    print(" ", BACKUP_USED)

    with zipfile.ZipFile(
        BACKUP_USED,
        "r"
    ) as zf:

        # ZIP paths were stored like:
        # kaggle/working/step3...
        #
        # Extract to temporary folder, then copy back to /
        tmp_restore = (
            WORK
            / "_adapruner_gpu_restore"
        )

        if tmp_restore.exists():
            shutil.rmtree(tmp_restore)

        tmp_restore.mkdir(
            parents=True,
            exist_ok=True
        )

        zf.extractall(
            tmp_restore
        )


    restored_work = (
        tmp_restore
        / "kaggle"
        / "working"
    )

    assert restored_work.exists(), (
        "Backup structure was not recognized."
    )


    for item in restored_work.iterdir():

        destination = (
            WORK
            / item.name
        )

        if item.is_dir():

            shutil.copytree(
                item,
                destination,
                dirs_exist_ok=True
            )

        else:

            shutil.copy2(
                item,
                destination
            )


    shutil.rmtree(
        tmp_restore
    )

    print("Backup restoration: COMPLETE")


# ======================================================================
# 3. VERIFY DEVELOPMENT FREEZE
# ======================================================================

EXPECTED_FREEZE_SHA = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

assert FREEZE_JSON.exists()
assert FREEZE_SHA_FILE.exists()


with open(
    FREEZE_SHA_FILE,
    "r",
    encoding="utf-8"
) as f:

    freeze_sha_text = (
        f.read()
        .strip()
    )


assert freeze_sha_text == EXPECTED_FREEZE_SHA, (
    "Frozen-development SHA mismatch."
)


with open(
    FREEZE_JSON,
    "r",
    encoding="utf-8"
) as f:

    FINAL_AFP_CONFIG_FROZEN = json.load(f)


FINAL_AFP_FREEZE_SHA256 = (
    EXPECTED_FREEZE_SHA
)

AFP_DEVELOPMENT_FROZEN = True


print("\nAFP frozen boundary: PASSED")
print("Freeze SHA:", FINAL_AFP_FREEZE_SHA256)


# ======================================================================
# 4. SELECTED SCORER CHECKPOINT IDENTITY
# ======================================================================

SCORER_DIR = (
    DEV_ROOT
    / "07_final_scorer"
)

WEB_SCORER = (
    SCORER_DIR
    / "webqsp_afp_scorer_selected.pt"
)

CWQ_SCORER = (
    SCORER_DIR
    / "cwq_afp_scorer_selected.pt"
)


assert WEB_SCORER.exists()
assert CWQ_SCORER.exists()


def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


webqsp_ckpt_sha = file_sha256(
    WEB_SCORER
)

cwq_ckpt_sha = file_sha256(
    CWQ_SCORER
)


assert (
    webqsp_ckpt_sha
    ==
    "bee65146403d565661b5105c41831af3e81d54ed5fba1b2a99bcb37382420d9f"
)

assert (
    cwq_ckpt_sha
    ==
    "91a531c057bb02d0b311ef8b78bf02248a63cd66e57fe8959ef86e7297d9d0a1"
)


print("Selected scorer checkpoints: PASSED")


# ======================================================================
# 5. RESTORE FROZEN VALIDATION PLAN ROWS
# ======================================================================

RQ1_DEV = (
    WORK
    / "step2_rq1_dev"
)

WEB_VAL_PATH = (
    RQ1_DEV
    / "planning_webqsp_validation.jsonl"
)

CWQ_VAL_PATH = (
    RQ1_DEV
    / "planning_cwq_validation.jsonl"
)


assert WEB_VAL_PATH.exists()
assert CWQ_VAL_PATH.exists()


def read_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


webqsp_val_plan_rows = (
    read_jsonl(
        WEB_VAL_PATH
    )
)

cwq_val_plan_rows = (
    read_jsonl(
        CWQ_VAL_PATH
    )
)


assert len(
    webqsp_val_plan_rows
) == 246

assert len(
    cwq_val_plan_rows
) == 3519


print(
    "\nFrozen validation planning rows:"
)

print(
    " WebQSP:",
    len(webqsp_val_plan_rows)
)

print(
    " CWQ:",
    len(cwq_val_plan_rows)
)


# ======================================================================
# 6. LOAD PERSISTED NOTEBOOK
# ======================================================================

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists(), (
    "Persisted source notebook not found."
)


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    persisted_nb_16ar = (
        json.load(f)
    )


print(
    "\nPersisted notebook recovered:",
    NOTEBOOK_PATH
)


# ======================================================================
# 7. RECOVER EXACT CELL-139 PLANNER WRAPPER SOURCE
# ======================================================================

CELL139_SOURCE = "".join(
    persisted_nb_16ar[
        "cells"
    ][139].get(
        "source",
        []
    )
)


tree139 = ast.parse(
    CELL139_SOURCE
)


run_planning_source_16ar = None


for node in tree139.body:

    if (
        isinstance(
            node,
            ast.FunctionDef
        )
        and
        node.name
        ==
        "run_planning_checkpointed"
    ):

        run_planning_source_16ar = (
            ast.get_source_segment(
                CELL139_SOURCE,
                node
            )
        )

        break


assert run_planning_source_16ar is not None


print(
    "Exact Cell-139 planning wrapper: RECOVERED"
)


# ======================================================================
# 8. RESTORE EXACT RoG REPOSITORY TO /kaggle/working
# ======================================================================
#
# M3 expects:
#
# /kaggle/working/reasoning-on-graphs
#
# The persisted notebook input contains the same repository copy.
# ======================================================================

INPUT_ROG_REPO = Path(
    "/kaggle/input/notebooks/"
    "mdsadmansamikhan/rog-ap/"
    "reasoning-on-graphs"
)

WORKING_ROG_REPO = Path(
    "/kaggle/working/"
    "reasoning-on-graphs"
)


assert INPUT_ROG_REPO.exists()


if not WORKING_ROG_REPO.exists():

    print(
        "\nRestoring exact RoG repository "
        "from persisted notebook input..."
    )

    shutil.copytree(
        INPUT_ROG_REPO,
        WORKING_ROG_REPO
    )


GEN_RULE = (
    WORKING_ROG_REPO
    / "src"
    / "qa_prediction"
    / "gen_rule_path.py"
)


assert GEN_RULE.exists()


assert (
    file_sha256(
        GEN_RULE
    )
    ==
    "e50b596db05bbfa853d20319717560c1d31f652e61574779d2f01228cc3e21af"
)


print(
    "Exact RoG repository SHA gate: PASSED"
)


# ======================================================================
# 9. LOAD OFFICIAL RoG VALIDATION + TEST SPLITS
# ======================================================================

print(
    "\nLoading official RoG datasets..."
)


webqsp_official_val_16ar = (
    load_dataset(
        "rmanluo/RoG-webqsp",
        split="validation"
    )
)

cwq_official_val_16ar = (
    load_dataset(
        "rmanluo/RoG-cwq",
        split="validation"
    )
)

webqsp_official_test_16ar = (
    load_dataset(
        "rmanluo/RoG-webqsp",
        split="test"
    )
)

cwq_official_test_16ar = (
    load_dataset(
        "rmanluo/RoG-cwq",
        split="test"
    )
)


assert len(
    webqsp_official_val_16ar
) == 246

assert len(
    cwq_official_val_16ar
) == 3519

assert len(
    webqsp_official_test_16ar
) == 1628

assert len(
    cwq_official_test_16ar
) == 3531


print(
    " WebQSP val/test:",
    246,
    "/",
    1628
)

print(
    " CWQ val/test:",
    3519,
    "/",
    3531
)


# ======================================================================
# 10. DATA NORMALIZATION
# ======================================================================

def norm_seq_gpu_recovery(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    if not isinstance(
        value,
        list
    ):
        value = [value]

    return [
        str(x)
        for x in value
    ]


def canonical_graph_gpu_recovery(graph):

    if isinstance(
        graph,
        np.ndarray
    ):
        graph = graph.tolist()

    return [
        [
            str(x)
            for x in triple
        ]
        for triple in graph
    ]


def official_to_rows_gpu_recovery(ds):

    rows = []

    for row in ds:

        rows.append(
            {
                "id":
                    str(row["id"]),

                "question":
                    str(row["question"]),

                "answer":
                    norm_seq_gpu_recovery(
                        row.get(
                            "answer",
                            []
                        )
                    ),

                "q_entity":
                    norm_seq_gpu_recovery(
                        row["q_entity"]
                    ),

                "a_entity":
                    norm_seq_gpu_recovery(
                        row["a_entity"]
                    ),

                "graph":
                    canonical_graph_gpu_recovery(
                        row["graph"]
                    ),

                "choices":
                    row.get(
                        "choices",
                        []
                    ),
            }
        )

    return rows


webqsp_raw_test_16a = (
    official_to_rows_gpu_recovery(
        webqsp_official_test_16ar
    )
)

cwq_raw_test_16a = (
    official_to_rows_gpu_recovery(
        cwq_official_test_16ar
    )
)


assert len(
    webqsp_raw_test_16a
) == 1628

assert len(
    cwq_raw_test_16a
) == 3531


# ======================================================================
# 11. VALIDATION SOURCE-IDENTITY GATE
# ======================================================================

def verify_val_identity_gpu_recovery(
    official_ds,
    frozen_rows,
    dataset_name
):

    official = {
        str(row["id"]):
            row
        for row in official_ds
    }

    frozen = {
        str(row["id"]):
            row
        for row in frozen_rows
    }


    assert set(
        official
    ) == set(
        frozen
    )


    for qid in frozen:

        assert str(
            official[qid]["question"]
        ) == str(
            frozen[qid]["question"]
        )

        assert norm_seq_gpu_recovery(
            official[qid]["q_entity"]
        ) == norm_seq_gpu_recovery(
            frozen[qid]["q_entity"]
        )

        assert norm_seq_gpu_recovery(
            official[qid]["a_entity"]
        ) == norm_seq_gpu_recovery(
            frozen[qid]["a_entity"]
        )


    print(
        dataset_name,
        "validation source identity: PASSED"
    )


verify_val_identity_gpu_recovery(
    webqsp_official_val_16ar,
    webqsp_val_plan_rows,
    "WebQSP"
)

verify_val_identity_gpu_recovery(
    cwq_official_val_16ar,
    cwq_val_plan_rows,
    "CWQ"
)


# ======================================================================
# 12. VALIDATION / TEST DISJOINTNESS
# ======================================================================

assert not (
    {
        str(x["id"])
        for x in webqsp_val_plan_rows
    }
    &
    {
        str(x["id"])
        for x in webqsp_raw_test_16a
    }
)

assert not (
    {
        str(x["id"])
        for x in cwq_val_plan_rows
    }
    &
    {
        str(x["id"])
        for x in cwq_raw_test_16a
    }
)


print(
    "Validation / TEST disjointness: PASSED"
)


# ======================================================================
# 13. RECOVER EXACT CELL-16 TRAVERSAL HELPERS
# ======================================================================

HELPERS_TO_RECOVER = [
    "build_exact_rog_adjacency",
    "as_entity_list",
    "normalize_relation_plans",
]


def recover_latest_function_from_notebook(
    function_name
):

    hits = []


    for cell_idx, cell in enumerate(
        persisted_nb_16ar[
            "cells"
        ]
    ):

        if cell.get(
            "cell_type"
        ) != "code":
            continue


        source = "".join(
            cell.get(
                "source",
                []
            )
        )


        if function_name not in source:
            continue


        try:
            tree = ast.parse(
                source
            )

        except Exception:
            continue


        for node in tree.body:

            if (
                isinstance(
                    node,
                    ast.FunctionDef
                )
                and
                node.name
                ==
                function_name
            ):

                segment = (
                    ast.get_source_segment(
                        source,
                        node
                    )
                )


                if segment:

                    hits.append(
                        (
                            cell_idx,
                            segment
                        )
                    )


    assert hits, (
        f"Could not recover function: "
        f"{function_name}"
    )


    # latest persisted definition
    cell_idx, source = sorted(
        hits,
        key=lambda x:
            x[0]
    )[-1]


    ns = {
        "np": np,
        "nx": nx,
        "json": json,
        "re": __import__("re"),
    }


    exec(
        source,
        ns
    )


    print(
        f"Recovered {function_name} "
        f"from cell {cell_idx}"
    )


    return ns[
        function_name
    ]


build_exact_rog_adjacency = (
    recover_latest_function_from_notebook(
        "build_exact_rog_adjacency"
    )
)

as_entity_list = (
    recover_latest_function_from_notebook(
        "as_entity_list"
    )
)

normalize_relation_plans = (
    recover_latest_function_from_notebook(
        "normalize_relation_plans"
    )
)


print(
    "Cell-16 traversal helpers: READY"
)


# ======================================================================
# 14. RECONSTRUCT SUCCESS FLAGS FROM VERIFIED STATE
# ======================================================================
#
# These flags do not pretend RAM survived.
# They indicate that their required scientific gates were reconstructed
# successfully in this GPU session.
# ======================================================================

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True


# ======================================================================
# 15. FINAL REPORT
# ======================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GPU RESTART RECOVERY COMPLETE"
)

print(
    "=" * 100
)

print(
    "\nFrozen development:"
)

print(
    " AFP freeze SHA: VERIFIED"
)

print(
    " WebQSP scorer:   VERIFIED"
)

print(
    " CWQ scorer:      VERIFIED"
)


print(
    "\nFrozen validation planning:"
)

print(
    " WebQSP:",
    len(webqsp_val_plan_rows)
)

print(
    " CWQ:",
    len(cwq_val_plan_rows)
)


print(
    "\nFinal TEST inputs:"
)

print(
    " WebQSP:",
    len(webqsp_raw_test_16a)
)

print(
    " CWQ:",
    len(cwq_raw_test_16a)
)


print(
    "\nGPU:"
)

print(
    " ",
    torch.cuda.get_device_name(0)
)


print(
    "\nAFP retrained: NO"
)

print(
    "AFP retuned:   NO"
)

print(
    "TEST plans generated: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

GPU recovery session: READY
GPU: Tesla T4

Existing experiment files survived GPU restart.

AFP frozen boundary: PASSED
Freeze SHA: bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116
Selected scorer checkpoints: PASSED

Frozen validation planning rows:
 WebQSP: 246
 CWQ: 3519

Persisted notebook recovered: /kaggle/input/notebooks/mdsadmansamikhan/rog-ap/__notebook__.ipynb
Exact Cell-139 planning wrapper: RECOVERED
Exact RoG repository SHA gate: PASSED

Loading official RoG datasets...


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/3531 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

 WebQSP val/test: 246 / 1628
 CWQ val/test: 3519 / 3531
WebQSP validation source identity: PASSED
CWQ validation source identity: PASSED
Validation / TEST disjointness: PASSED


AssertionError: Could not recover function: build_exact_rog_adjacency

In [2]:
# ======================================================================
# GPU RESTART RECOVERY PATCH
# Exact RoG traversal-helper reconstruction + validation fidelity gate
# ======================================================================
#
# RUN AS A NEW CELL after the GPU recovery cell stopped.
#
# DO NOT:
#   - rerun the full GPU recovery cell
#   - retrain/tune anything
#   - run Cell 16 yet
#
# WHY THIS PATCH EXISTS
# ---------------------
# build_exact_rog_adjacency was a later verified experimental helper
# and is not present in the old persisted notebook snapshot.
#
# We reconstruct it from RoG's EXACT build_graph source and then require
# reproduction of the previously frozen validation traversal totals.
#
# If BOTH WebQSP and CWQ totals match exactly, recovery is behaviorally
# equivalent to the pre-restart traversal implementation.
# ======================================================================

import ast
import json
from pathlib import Path

import numpy as np
import networkx as nx


# ======================================================================
# 1. HARD GATES
# ======================================================================

required = [
    "AFP_DEVELOPMENT_FROZEN",
    "FINAL_AFP_FREEZE_SHA256",

    "persisted_nb_16ar",

    "webqsp_val_plan_rows",
    "cwq_val_plan_rows",

    "webqsp_official_test_16ar",
    "cwq_official_test_16ar",

    "webqsp_raw_test_16a",
    "cwq_raw_test_16a",

    "run_planning_source_16ar",
]

missing = [
    name
    for name in required
    if name not in globals()
]

assert not missing, (
    "Missing GPU-recovery prerequisite(s):\n  "
    + "\n  ".join(missing)
)

assert AFP_DEVELOPMENT_FROZEN is True

assert (
    FINAL_AFP_FREEZE_SHA256
    ==
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)

print("GPU recovery patch freeze gate: PASSED")


# ======================================================================
# 2. RECOVER EXACT RoG build_graph SOURCE
# ======================================================================

GRAPH_UTILS_CANDIDATES = [
    Path(
        "/kaggle/working/reasoning-on-graphs/"
        "src/utils/graph_utils.py"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/rog-ap/"
        "reasoning-on-graphs/src/utils/graph_utils.py"
    ),
]


GRAPH_UTILS_PATH = next(
    (
        path
        for path in GRAPH_UTILS_CANDIDATES
        if path.exists()
    ),
    None
)

assert GRAPH_UTILS_PATH is not None, (
    "Exact RoG graph_utils.py not found."
)


graph_utils_source = GRAPH_UTILS_PATH.read_text(
    encoding="utf-8"
)

graph_utils_tree = ast.parse(
    graph_utils_source
)


exact_build_graph_source = None


for node in graph_utils_tree.body:

    if (
        isinstance(node, ast.FunctionDef)
        and
        node.name == "build_graph"
    ):

        exact_build_graph_source = (
            ast.get_source_segment(
                graph_utils_source,
                node
            )
        )

        break


assert exact_build_graph_source is not None


build_graph_ns = {
    "nx": nx
}


exec(
    exact_build_graph_source,
    build_graph_ns
)


_exact_rog_build_graph = (
    build_graph_ns[
        "build_graph"
    ]
)


print("\nExact RoG build_graph recovered:")
print(exact_build_graph_source)


# ======================================================================
# 3. EXACT ADJACENCY ADAPTER
# ======================================================================
#
# RoG build_graph uses an UNDIRECTED simple nx.Graph:
#
#   G.add_edge(h, t, relation=r.strip())
#
# Consequences intentionally preserved:
#
# - undirected traversal
# - later duplicate unordered pair overwrites edge relation
# - neighbor insertion order is preserved by NetworkX
# - exact relation equality
# ======================================================================

def build_exact_rog_adjacency(triples):

    G = _exact_rog_build_graph(
        triples
    )

    adjacency = {}

    for node in G.nodes:

        adjacency[node] = {}

        for neighbor, attrs in G[node].items():

            adjacency[node][neighbor] = (
                attrs["relation"]
            )

    return adjacency


# ======================================================================
# 4. ENTITY NORMALIZER
# ======================================================================

def as_entity_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    if isinstance(
        value,
        set
    ):
        value = list(value)

    if isinstance(
        value,
        list
    ):
        return value

    return [value]


# ======================================================================
# 5. RELATION-PLAN NORMALIZER
# ======================================================================

def normalize_relation_plans(value):

    if value is None:
        return []

    if isinstance(
        value,
        np.ndarray
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple
    ):
        value = list(value)

    # Defensive support for serialized containers.
    if isinstance(
        value,
        str
    ):

        text = value.strip()

        if not text:
            return []

        try:
            parsed = json.loads(
                text
            )

            return normalize_relation_plans(
                parsed
            )

        except Exception:
            return [[text]]

    if not isinstance(
        value,
        list
    ):
        return [[value]]

    if len(value) == 0:
        return []

    # Flat relation-token list means one plan.
    if all(
        isinstance(x, str)
        for x in value
    ):
        return [
            list(value)
        ]

    plans = []

    for plan in value:

        if plan is None:
            continue

        if isinstance(
            plan,
            np.ndarray
        ):
            plan = plan.tolist()

        if isinstance(
            plan,
            tuple
        ):
            plan = list(plan)

        if isinstance(
            plan,
            str
        ):
            plans.append(
                [plan]
            )

        elif isinstance(
            plan,
            list
        ):
            # Preserve empty predicted plans.
            plans.append(
                list(plan)
            )

        else:
            plans.append(
                [plan]
            )

    return plans


# ======================================================================
# 6. EXACT VALIDATION TRAVERSAL FIDELITY CHECK
# ======================================================================

def validate_exact_rog_recovery(
    dataset_name,
    planning_rows
):

    active_hop_rows = 0
    active_prefixes_total = 0
    edges_examined_total = 0
    candidate_branches_total = 0

    reachable_plans = 0
    reachable_questions = 0

    total_plans = 0


    for rec in planning_rows:

        adjacency = (
            build_exact_rog_adjacency(
                rec["graph"]
            )
        )

        topics = (
            as_entity_list(
                rec["q_entity"]
            )
        )

        answers = set(
            as_entity_list(
                rec["a_entity"]
            )
        )

        plans = (
            normalize_relation_plans(
                rec["predicted_paths"]
            )
        )

        question_reachable = False


        for plan in plans:

            total_plans += 1

            # Frozen RQ1 convention:
            # empty relation plan contributes no active traversal row.
            if len(plan) == 0:
                continue


            active = [
                (entity,)
                for entity in topics
            ]


            for relation in plan:

                if not active:
                    break


                active_hop_rows += 1

                active_prefixes_total += (
                    len(active)
                )


                # RoG examines every unique neighbor of every
                # active PATH PREFIX endpoint before relation filtering.
                edges_examined_total += sum(
                    len(
                        adjacency.get(
                            prefix[-1],
                            {}
                        )
                    )
                    for prefix in active
                )


                candidates = []


                for prefix in active:

                    endpoint = prefix[-1]

                    for (
                        neighbor,
                        edge_relation
                    ) in adjacency.get(
                        endpoint,
                        {}
                    ).items():

                        if (
                            edge_relation
                            ==
                            relation
                        ):

                            candidates.append(
                                prefix
                                +
                                (neighbor,)
                            )


                candidate_branches_total += (
                    len(candidates)
                )


                active = candidates


            plan_reachable = any(
                prefix[-1] in answers
                for prefix in active
            )


            if plan_reachable:

                reachable_plans += 1

                question_reachable = True


        if question_reachable:

            reachable_questions += 1


    return {
        "questions":
            len(planning_rows),

        "plans":
            total_plans,

        "active_hop_rows":
            active_hop_rows,

        "active_prefixes":
            active_prefixes_total,

        "edges_examined":
            edges_examined_total,

        "candidate_branches":
            candidate_branches_total,

        "reachable_plans":
            reachable_plans,

        "reachable_questions":
            reachable_questions,
    }


# ======================================================================
# 7. PREVIOUSLY VERIFIED FROZEN VALIDATION REFERENCES
# ======================================================================

EXPECTED_WEBQSP = {
    "questions": 246,
    "plans": 721,
    "active_hop_rows": 971,
    "active_prefixes": 2440,
    "edges_examined": 341526,
    "candidate_branches": 7983,
    "reachable_plans": 345,
    "reachable_questions": 205,
}


EXPECTED_CWQ = {
    "questions": 3519,
    "plans": 10529,
    "active_hop_rows": 16564,
    "active_prefixes": 57841,
    "edges_examined": 5257272,
    "candidate_branches": 247161,
    "reachable_plans": 3971,
    "reachable_questions": 2425,
}


print(
    "\nRunning WebQSP exact RoG validation fidelity gate..."
)

WEB_RECOVERED = (
    validate_exact_rog_recovery(
        "webqsp",
        webqsp_val_plan_rows
    )
)


print(
    WEB_RECOVERED
)


print(
    "\nRunning CWQ exact RoG validation fidelity gate..."
)

CWQ_RECOVERED = (
    validate_exact_rog_recovery(
        "cwq",
        cwq_val_plan_rows
    )
)


print(
    CWQ_RECOVERED
)


# ======================================================================
# 8. EXACT VALUE-BY-VALUE ASSERTIONS
# ======================================================================

for key, expected in EXPECTED_WEBQSP.items():

    actual = WEB_RECOVERED[
        key
    ]

    assert actual == expected, (
        f"WebQSP recovery mismatch for {key}: "
        f"{actual} != {expected}"
    )


for key, expected in EXPECTED_CWQ.items():

    actual = CWQ_RECOVERED[
        key
    ]

    assert actual == expected, (
        f"CWQ recovery mismatch for {key}: "
        f"{actual} != {expected}"
    )


print(
    "\n"
    + "=" * 100
)

print(
    "EXACT RoG VALIDATION FIDELITY: PASSED"
)

print(
    "=" * 100
)

print(
    "WebQSP:"
)

print(
    " active-hop rows: 971"
)

print(
    " active prefixes: 2440"
)

print(
    " edges:           341526"
)

print(
    " candidates:      7983"
)

print(
    " reachable plans: 345"
)

print(
    " reachable q:     205"
)


print(
    "\nCWQ:"
)

print(
    " active-hop rows: 16564"
)

print(
    " active prefixes: 57841"
)

print(
    " edges:           5257272"
)

print(
    " candidates:      247161"
)

print(
    " reachable plans: 3971"
)

print(
    " reachable q:     2425"
)


# ======================================================================
# 9. RECONSTRUCT GPU-RECOVERY COMPLETION FLAGS
# ======================================================================

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True

EXACT_ROG_HELPERS_RECOVERED = True


print(
    "\n"
    + "=" * 100
)

print(
    "GPU RESTART RECOVERY PATCH COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nScientific status:"
)

print(
    " AFP freeze: VERIFIED"
)

print(
    " exact RoG traversal: VERIFIED"
)

print(
    " validation totals reproduced exactly: YES"
)

print(
    " TEST plans generated: NO"
)

print(
    " TEST tuning: NO"
)

print(
    " development changed: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

GPU recovery patch freeze gate: PASSED

Exact RoG build_graph recovered:
def build_graph(graph: list) -> nx.Graph:
    G = nx.Graph()
    for triplet in graph:
        h, r, t = triplet
        G.add_edge(h, t, relation=r.strip())
    return G

Running WebQSP exact RoG validation fidelity gate...
{'questions': 246, 'plans': 721, 'active_hop_rows': 971, 'active_prefixes': 2440, 'edges_examined': 341526, 'candidate_branches': 7983, 'reachable_plans': 345, 'reachable_questions': 205}

Running CWQ exact RoG validation fidelity gate...
{'questions': 3519, 'plans': 10536, 'active_hop_rows': 16564, 'active_prefixes': 57841, 'edges_examined': 5257272, 'candidate_branches': 247161, 'reachable_plans': 3971, 'reachable_questions': 2425}


AssertionError: CWQ recovery mismatch for plans: 10536 != 10529

In [3]:
# ======================================================================
# GPU RECOVERY PATCH-FIX
# Correct CWQ total-plan vs non-empty-plan reference
# ======================================================================
#
# RUN AS A NEW CELL.
#
# Do NOT rerun the full GPU recovery.
# Do NOT rerun the previous large recovery patch.
#
# No TEST generation occurs here.
# ======================================================================


# ----------------------------------------------------------------------
# 1. Frozen validation references — corrected
# ----------------------------------------------------------------------

EXPECTED_WEBQSP_CORRECTED = {
    "questions": 246,
    "plans": 721,
    "active_hop_rows": 971,
    "active_prefixes": 2440,
    "edges_examined": 341526,
    "candidate_branches": 7983,
    "reachable_plans": 345,
    "reachable_questions": 205,
}


EXPECTED_CWQ_CORRECTED = {
    "questions": 3519,

    # IMPORTANT:
    # 10,536 = ALL predicted relation plans
    # 10,529 = NON-EMPTY predicted relation plans
    # 7      = empty relation plans
    "plans": 10536,

    "active_hop_rows": 16564,
    "active_prefixes": 57841,
    "edges_examined": 5257272,
    "candidate_branches": 247161,
    "reachable_plans": 3971,
    "reachable_questions": 2425,
}


# ----------------------------------------------------------------------
# 2. Verify already-computed traversal results
# ----------------------------------------------------------------------

assert "WEB_RECOVERED" in globals()
assert "CWQ_RECOVERED" in globals()


for key, expected in EXPECTED_WEBQSP_CORRECTED.items():

    actual = WEB_RECOVERED[key]

    assert actual == expected, (
        f"WebQSP mismatch for {key}: "
        f"{actual} != {expected}"
    )


for key, expected in EXPECTED_CWQ_CORRECTED.items():

    actual = CWQ_RECOVERED[key]

    assert actual == expected, (
        f"CWQ mismatch for {key}: "
        f"{actual} != {expected}"
    )


# ----------------------------------------------------------------------
# 3. Explicitly verify CWQ empty/non-empty plan distinction
# ----------------------------------------------------------------------

cwq_total_plans = 0
cwq_empty_plans = 0
cwq_nonempty_plans = 0


for rec in cwq_val_plan_rows:

    plans = normalize_relation_plans(
        rec["predicted_paths"]
    )

    for plan in plans:

        cwq_total_plans += 1

        if len(plan) == 0:
            cwq_empty_plans += 1
        else:
            cwq_nonempty_plans += 1


assert cwq_total_plans == 10536
assert cwq_empty_plans == 7
assert cwq_nonempty_plans == 10529


print("CWQ plan accounting: PASSED")
print("  total plans:    ", cwq_total_plans)
print("  non-empty plans:", cwq_nonempty_plans)
print("  empty plans:    ", cwq_empty_plans)


# ----------------------------------------------------------------------
# 4. WebQSP plan accounting
# ----------------------------------------------------------------------

web_total_plans = 0
web_empty_plans = 0
web_nonempty_plans = 0


for rec in webqsp_val_plan_rows:

    plans = normalize_relation_plans(
        rec["predicted_paths"]
    )

    for plan in plans:

        web_total_plans += 1

        if len(plan) == 0:
            web_empty_plans += 1
        else:
            web_nonempty_plans += 1


assert web_total_plans == 721


print("\nWebQSP plan accounting: PASSED")
print("  total plans:    ", web_total_plans)
print("  non-empty plans:", web_nonempty_plans)
print("  empty plans:    ", web_empty_plans)


# ----------------------------------------------------------------------
# 5. Mark reconstructed traversal state as verified
# ----------------------------------------------------------------------

CELL16AR_COMPLETE = True
CELL16AD_COMPLETE = True

GPU_RESTART_RECOVERY_COMPLETE = True
EXACT_ROG_HELPERS_RECOVERED = True
EXACT_ROG_VALIDATION_FIDELITY_PASSED = True


print(
    "\n"
    + "=" * 100
)

print(
    "EXACT RoG VALIDATION FIDELITY: PASSED"
)

print(
    "=" * 100
)


print("\nWebQSP")
print("  plans:             721")
print("  active-hop rows:   971")
print("  active prefixes:   2440")
print("  edges:              341526")
print("  candidates:         7983")
print("  reachable plans:    345")
print("  reachable questions:205")


print("\nCWQ")
print("  total plans:        10536")
print("  non-empty plans:    10529")
print("  empty plans:        7")
print("  active-hop rows:    16564")
print("  active prefixes:    57841")
print("  edges:               5257272")
print("  candidates:          247161")
print("  reachable plans:     3971")
print("  reachable questions: 2425")


print(
    "\nGPU RESTART RECOVERY: COMPLETE"
)

print(
    "Development changed: NO"
)

print(
    "TEST plans generated: NO"
)

print(
    "TEST tuning: NO"
)


print(
    "\nNEXT ACTION:"
)

print(
    "RUN CELL 16A-M3 UNCHANGED."
)

CWQ plan accounting: PASSED
  total plans:     10536
  non-empty plans: 10529
  empty plans:     7

WebQSP plan accounting: PASSED
  total plans:     721
  non-empty plans: 721
  empty plans:     0

EXACT RoG VALIDATION FIDELITY: PASSED

WebQSP
  plans:             721
  active-hop rows:   971
  active prefixes:   2440
  edges:              341526
  candidates:         7983
  reachable plans:    345
  reachable questions:205

CWQ
  total plans:        10536
  non-empty plans:    10529
  empty plans:        7
  active-hop rows:    16564
  active prefixes:    57841
  edges:               5257272
  candidates:          247161
  reachable plans:     3971
  reachable questions: 2425

GPU RESTART RECOVERY: COMPLETE
Development changed: NO
TEST plans generated: NO
TEST tuning: NO

NEXT ACTION:
RUN CELL 16A-M3 UNCHANGED.
